# eGOT (enhanced Graph-of-Thoughts) Uygulamasının Google Colab'da Çalıştırılması

| | |
|---|---|
| **Ad Soyad** | _…………………………_ |
| **Öğrenci No** | _…………………………_ |
| **Ödev** | YZ WarmUP-3: eGOT Uygulamasının Çalıştırılması |
| **Kaynak kod** | [NNeuralDynamics/eGOT](https://github.com/NNeuralDynamics/eGOT), commit `3d37906` (`master`'ın 3 Kasım 2026'daki HEAD'i; repodaki tek commit) [6] |
| **Dil modeli** | Qwen2.5-7B-Instruct; Colab'ın T4 GPU'sunda Ollama ile yerel ve ücretsiz çalışır (gerekçe: Bölüm 3) |
| **Örnek** | MultiHop-RAG [5] (repodaki `datasets/mhoprag_contexts.csv`) |

## Özet

eGOT, Sanda vd. (2026) tarafından önerilen GraphRAG tabanlı bir soru-cevap yöntemidir [1]. Bir belge koleksiyonundan LLM ile varlıklar ve ilişkiler çıkarılıp normalleştirilir; bunlardan bir graf ve bir vektör veritabanı kurulur. Bir soru geldiğinde LLM, bu veritabanlarını birkaç tur boyunca sorgulayarak cevabı oluşturur [1]. Turlar, düşünceleri (thought) bir grafın düğümleri gibi ele alan *Graph of Thoughts* (GoT) yaklaşımına [4] dayanır: Her thought bir alt soru ya da hipotezdir. Her biri için grafla zenginleştirilmiş bir arama yapılır ve bulunan kanıt bir sonraki turun thought'larını belirler [1, 2].

Bu notebook repoyu Colab'ın **T4 GPU** çalışma zamanında baştan sona çalıştırır. GPU, dil modelini yerel çalıştırmak için kullanılır:

1. Repo incelenir ve sistemin çalışma mantığı açıklanır (Bölüm 1).
2. Repo commit'e sabitlenerek klonlanır. Yalnızca kodun gerçekten import ettiği paketler kurulur. Colab'a Neo4j 5.26 LTS ve APOC kurulur. Dil modeli için Ollama kurulur ve Qwen2.5-7B-Instruct indirilir (Bölüm 2).
3. Dil modeli seçilir ve gerekçelendirilir (Bölüm 3).
4. Repo olduğu gibi çalışmıyor. Her hata önce **gerçekten tetiklenir**, sonra görünür bir yamayla düzeltilir (Bölüm 4).
5. Repodaki MultiHop-RAG örneği küçültülmüş bir alt kümeyle uçtan uca çalıştırılır. Sırasıyla veri hazırlanır, bilgi grafı (KG) kurulur, Neo4j'ye yüklenir, FastAPI sunucusu başlatılır, sorular GoT ile cevaplanır ve EM/F1 hesaplanır (Bölüm 5–9).
6. Karşılaşılan hatalar tabloda toplanır. Son olarak eGOT'un GraphRAG [3] yaklaşımından nasıl farklılaştığı değerlendirilir (Bölüm 10–11).

> **Dürüstlük notu:** Bu çalıştırmanın sonuçlarına ait hiçbir skor, süre ya da sayı metne elle yazılmamıştır (aşağıdaki kaba süre tahmini hariç). Tüm sonuçlar, Colab'da çalıştırıldığında hücre çıktılarında üretilir; yorumlar bu çıktılara atıf yapar.

## Nasıl çalıştırılır?

1. **Çalışma zamanı:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`. GPU, dil modelini Colab'da yerel çalıştırmak için gerekir (Bölüm 2.6). Ücretli bir API kullanılacaksa (Bölüm 0.1) CPU yeterlidir.
2. **Secrets:** Varsayılan ayarlarla hiçbir secret gerekmez; model yerel çalışır ve anahtar istemez. Secret yalnızca şu durumlarda eklenir (sol menüdeki 🔑 simgesi, "Notebook access" izni açık):
   - Ücretli bir OpenAI-uyumlu API kullanılacaksa: Bölüm 0.1'deki `LLM_BASE_URL`, `LLM_MODEL` ve `LLM_API_KEY_SECRET` değerleri değiştirilir ve anahtar bu adla (ör. `OPENAI_API_KEY`) eklenir.
   - `NEO4J_MODE = "aura"` seçilirse: `NEO4J_URI`, `NEO4J_USERNAME` ve `NEO4J_PASSWORD`.
3. `Çalışma zamanı → Tümünü çalıştır`. Varsayılan ayarlarla (8 soru, GoT derinliği 2) toplam süre kabaca **30–60 dakikadır**. Sürenin çoğu kurulumda, KG oluşturmada ve GoT sorgularında geçer. Gerçek süreler ilgili hücrelerin çıktısında yazar.
4. Hücreler yeniden çalıştırılabilir. Builder'ın JSON çıktıları ve bütün tahminler (demo dahil) dosyaya kaydedilir; dosya zaten varsa LLM tekrar çağrılmaz. Çalışma zamanı (VM) kapanmadığı sürece, yarıda kalan bir çalıştırmadan sonra `Tümünü çalıştır` yeniden kullanılabilir: kurulu paketler, Neo4j, Ollama ve indirilen model, KG ve kayıtlı cevaplar yeniden kullanılır; yalnızca eksik kalan işler yapılır.

## İçindekiler

0. Ayarlar ve yardımcı fonksiyonlar
1. eGOT'a genel bakış
2. Kurulum
3. Dil modeli seçimi ve gerekçesi
4. Hata düzeltmeleri
5. Veri hazırlama (MultiHop-RAG alt kümesi)
6. Bilgi grafının (KG) oluşturulması
7. Neo4j'ye yükleme, indeksler ve kontroller
8. Retrieval sunucusu ve tek soruluk GoT demosu
9. Toplu çalıştırma ve değerlendirme
10. Karşılaşılan hatalar ve çözümleri
11. Teknik değerlendirme: eGOT ile GraphRAG
12. Kaynaklar

## 0. Ayarlar ve yardımcı fonksiyonlar

### 0.1 Ayarlar

Notebook'un bütün ayarları aşağıdaki tek hücrede durur.

- **Dil modeli:** `LLM_BASE_URL`, `LLM_MODEL` ve `LLM_API_KEY_SECRET` (anahtarın Secrets'taki *adı*). Varsayılan model Qwen2.5-7B-Instruct'tır ve Colab'ın T4 GPU'sunda Ollama ile yerel çalışır (Bölüm 2.6). Ücretsizdir ve anahtar gerektirmez. eGOT'un iki bileşeni de bu modeli kullanır: KG kurulumu (graph-builder) ve retrieval sunucusu. OpenAI-uyumlu ücretli bir API'ye (ör. OpenAI `gpt-4o-mini`) geçmek için yalnızca bu üç değeri değiştirmek yeterlidir; o zaman GPU gerekmez.
- **Deney boyutu:** `N_QUESTIONS = 8`, `MAX_DEPTH = 2` ve `BUILDER_WORKERS = 2`, süreyi makul tutmak için küçük seçildi. Hepsi değiştirilebilir.
- **Neo4j:** Varsayılan `NEO4J_MODE = "local"` ile Neo4j Community 5.26 LTS bu Colab oturumuna kurulur. Kurulum başarısız olursa `"aura"` ile AuraDB Free kullanılabilir.

In [ ]:
# ================================ 0.1 AYARLAR ================================
# Dil modeli (gerekçe: Bölüm 3). Varsayılan: Qwen2.5-7B-Instruct, Colab'ın T4 GPU'sunda Ollama ile yerel çalışır
# (Bölüm 2.6). Ücretsizdir; API anahtarı ve kredi gerekmez. Çalışma zamanı türü: T4 GPU.
LLM_BASE_URL       = "http://127.0.0.1:11434/v1"   # Ollama'nın OpenAI-uyumlu uç noktası
LLM_MODEL          = "qwen2.5:7b"                  # Ollama kütüphanesindeki Qwen2.5-7B-Instruct (4-bit nicemlenmiş)
LLM_API_KEY_SECRET = None                          # yerel modelde anahtar yok
# OpenAI-uyumlu (ücretli) bir API'ye geçmek için yalnızca bu üç değeri değiştirmek yeterlidir; o zaman GPU gerekmez.
# Anahtarın DEĞERİ değil, Colab Secrets'taki ADI yazılır. Örnek:
#   OpenAI: "https://api.openai.com/v1", "gpt-4o-mini", "OPENAI_API_KEY"   (fiyatlar: 0.15 / 0.60 [10])

# Tahmini maliyet hesabı için fiyatlar (USD / 1M token). Yerel modelde API ücreti yoktur: None (maliyet hesaplanmaz).
PRICE_INPUT_PER_1M  = None
PRICE_OUTPUT_PER_1M = None

# Yerel model (Ollama) ayarları: yalnızca LLM_BASE_URL yerel Ollama'yı gösterdiğinde kullanılır (Bölüm 2.6)
OLLAMA_VERSION        = "0.40.14"  # GitHub'dan indirilecek sürüm (3 Kasım 2026'daki en yeni sürüm)
OLLAMA_CONTEXT_LENGTH = 16384      # bağlam penceresi (token). Ollama bu GPU'da varsayılan olarak 4096 kullanır

# Neo4j
NEO4J_MODE    = "local"    # "local": Colab'a Neo4j Community 5.26 LTS kurulur | "aura": AuraDB Free (bağlantı bilgileri Secrets'tan)
NEO4J_VERSION = "5.26.32"  # local modda indirilecek 5.26 LTS yaması (bulunamazsa en yeni 5.26.x sürümleri denenir)

# Deney boyutu ve eGOT parametreleri
N_QUESTIONS     = 8        # MultiHop-RAG'den seçilecek soru sayısı (question_type'lara dengeli dağıtılır)
SEED            = 42       # soru seçimi için sabit tohum
MAX_DEPTH       = 2        # GoT derinliği (repo kodunda varsayılan 3; makaledeki benchmark'larda 2 [1])
SYNTHESIS_STYLE = "mhop"   # "mhop": kısa cevap (got.py → MHOP_SYNTHESIS_PROMPT) | "long": repo'nun uzun sentez prompt'u
BUILDER_WORKERS = 2        # KG kurulumunda aynı anda işlenen dosya sayısı (repo: 6; hız sınırı riskini azaltır)
BUILDER_TEMPERATURE = 0.2  # repo config.toml: 0.8, graph-builder README örneği: 0.2
BUILDER_MAX_TOKENS  = 4096 # repo config.toml: 8192
RUN_BASELINE    = True     # aynı soruları GoT'suz modda (graph_vector_fulltext) da çalıştırıp karşılaştır
EVIDENCE_EXTRA_FIELDS = False     # True: kanıt dosyalarına Source / Published at satırları da eklenir (repo biçiminde yok)
SHOW_REQUIREMENTS_DRYRUN = True   # Bölüm 2.2: requirements.txt'nin kurulamadığını pip --dry-run ile göster (~3 dk)

# Sabitler
REPO_URL    = "https://github.com/NNeuralDynamics/eGOT.git"
REPO_COMMIT = "3d37906c1b22472ee970c85e8dbd69ebcd5f8ffa"   # master'ın 3 Kasım 2026'daki HEAD'i (repodaki tek commit)
WORK_DIR    = "/content/egot_work"

### 0.2 Gizli bilgiler

API anahtarı `google.colab.userdata` (Colab Secrets) ile okunur ve yalnızca `os.environ`'a konur. Hiçbir hücre anahtarın değerini yazdırmaz. Builder ve retrieval sunucusu ayrı süreçlerde çalışır; anahtarı bu ortam değişkeninden miras alırlar. Anahtar builder'ın yapılandırma dosyasına da yazılmaz (Bölüm 6.1). Yerel Neo4j'nin şifresi rastgele üretilir ve yalnızca dosyada saklanır.

In [ ]:
# 0.2 Gizli bilgiler: Colab Secrets → os.environ (değerler hiçbir yerde yazdırılmaz)
import os, sys

IN_COLAB = "google.colab" in sys.modules


def get_secret(name, required=True):
    """Colab Secrets'tan okur; Colab dışında aynı adlı ortam değişkenine bakar. Değeri asla yazdırmaz."""
    if not name:
        return None
    value = None
    if IN_COLAB:
        from google.colab import userdata
        try:
            value = userdata.get(name)
        except Exception as e:  # SecretNotFoundError, NotebookAccessError ...
            print(f"⚠️  '{name}' secret'ı okunamadı ({type(e).__name__}).")
    else:
        value = os.environ.get(name)
    if required and not value:
        raise RuntimeError(f"'{name}' bulunamadı. Sol menüdeki 🔑 Secrets bölümüne '{name}' adıyla ekleyin "
                           "ve 'Notebook access' iznini açın.")
    return value


_key = get_secret(LLM_API_KEY_SECRET, required=bool(LLM_API_KEY_SECRET))
os.environ["OPENAI_API_KEY"] = _key or "not-needed"  # retrieval (QA_integration.py) ve yamalı builder buradan okur
os.environ["OPENAI_BASE_URL"] = LLM_BASE_URL         # retrieval'daki ChatOpenAI(base_url=OPENAI_BASE_URL)
del _key
print(f"LLM: {LLM_MODEL} @ {LLM_BASE_URL}")
print("API anahtarı:", f"'{LLM_API_KEY_SECRET}' secret'ından okundu (değeri gösterilmez)." if LLM_API_KEY_SECRET
      else "kullanılmıyor.")

### 0.3 Yardımcı fonksiyonlar

- `run` / `run_py`: Bir komutu ya da Python kodunu **ayrı bir süreçte** çalıştırır ve çıktının sonunu gösterir. Builder ile retrieval kodu aynı üst seviye paket adını (`src`) kullandığı için ikisi de bu çekirdeğe import edilmez (Bölüm 4.12).
- `record_error`: Gerçekten tetiklenen hatanın mesajını kaydeder. Bu kayıtlar Bölüm 10.2'de tablo olarak gösterilir.
- `patch_file`: Repo dosyasında birebir metin değişikliği yapar. Beklenen metin dosyada tam bir kez geçmiyorsa `assert` ile durur, böylece yanlış yere yama yapılamaz. Yama zaten uygulanmışsa tekrar uygulanmaz.

In [ ]:
# 0.3 Yardımcı fonksiyonlar
import json, re, time, shutil, socket, secrets, hashlib, textwrap, subprocess, uuid, html
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

pd.set_option("display.max_colwidth", 140)

WORK = Path(WORK_DIR); WORK.mkdir(parents=True, exist_ok=True)
REPO        = WORK / "eGOT"                                 # repo klonu
BUILDER_DIR = REPO / "src" / "graph-builder"                # KG kurulum kodu
RETR_DIR    = REPO / "src" / "knowledge_graph_retrieval"    # FastAPI + GoT kodu
SERVER_DIR  = WORK / "server"                               # sunucunun çalıştırılacağı düzen (Bölüm 4.6)
LOG_DIR     = WORK / "logs"; LOG_DIR.mkdir(exist_ok=True)

# Alt süreçlerin ortak ayarları (anahtarlar os.environ'dan miras alınır, hiçbir yere yazılmaz)
os.environ.update({"TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1", "PYTHONWARNINGS": "ignore",
                   "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_HUB_VERBOSITY": "error", "TRANSFORMERS_VERBOSITY": "error",
                   "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"})   # herkese açık gömme modeli için HF token'ı gerekmez
LOCAL_LLM = bool(re.match(r"https?://(127\.0\.0\.1|localhost):11434(/|$)", LLM_BASE_URL))   # yerel Ollama mı? (Bölüm 2.6)
os.environ["EGOT_LLM_TIMEOUT"] = "600" if LOCAL_LLM else "120"   # tek LLM çağrısı için üst süre (sn); yerel model daha yavaş

NB_T0 = time.time()   # notebook'un başlangıç zamanı (Bölüm 11.0'da toplam süre)
ERRORS  = []   # bu çalıştırmada gerçekten yakalanan hatalar (Bölüm 10'da tablo olarak gösterilir)
_SECRET_RX = re.compile(r"\bsk-[A-Za-z0-9_*\-]{8,}|Bearer\s+[A-Za-z0-9._\-]+")


def redact(text):
    """Çıktıya basılacak metinden anahtar/şifre benzeri parçaları maskeler (sağlayıcı hata mesajları
    anahtarın bir kısmını içerebilir)."""
    text = _SECRET_RX.sub("***", str(text))
    for name in ("OPENAI_API_KEY", "NEO4J_PASSWORD"):
        value = os.environ.get(name)
        if value and len(value) >= 8:
            text = text.replace(value, "***")
    return text


PATCHES = []   # uygulanan yamalar
_EXC = re.compile(r"^(?:[A-Za-z_][\w.]*\.)?(?:[A-Z]\w*)?(?:Error|Exception)\b")


def exception_line(output, pattern=None):
    """Çıktıdaki son 'XxxError: mesaj' satırını (ya da verilen kalıba uyan son satırı) döndürür."""
    rx = re.compile(pattern) if pattern else _EXC
    lines = [l.strip() for l in output.splitlines() if rx.search(l.strip())]
    return lines[-1] if lines else (output.strip().splitlines() or ["(çıktı yok)"])[-1].strip()


def run(cmd, cwd=None, timeout=None, tail=20, env=None, show=True):
    """Komutu çalıştırır, çıktının son `tail` satırını gösterir; (çıkış kodu, tüm çıktı) döndürür."""
    p = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, timeout=timeout,
                       capture_output=True, text=True, env=env)
    out = (p.stdout or "") + (p.stderr or "")
    if show and out.strip():
        lines = out.rstrip().splitlines()
        if tail and len(lines) > tail:
            print(f"... ({len(lines) - tail} satır gizlendi)")
            lines = lines[-tail:]
        print(redact("\n".join(l[:400] for l in lines)))
    return p.returncode, out


def run_py(code, cwd, timeout=900, tail=20, env=None, show=True):
    """Python kodunu AYRI bir süreçte çalıştırır. Builder ve retrieval aynı üst seviye `src` paket adını
    kullandığı için ikisi de bu çekirdeğe import edilmez (Bölüm 4.10)."""
    return run([sys.executable, "-c", textwrap.dedent(code)], cwd=cwd, timeout=timeout, tail=tail, env=env, show=show)


def record_error(hid, where, output, pattern=None):
    """Tetiklenen hatayı ERRORS listesine ekler ve yakalanan mesajı gösterir."""
    line = redact(exception_line(output, pattern))[:300]
    ERRORS.append({"id": hid, "nerede": where, "yakalanan mesaj": line})
    print(f"\n➜ [{hid}] yakalanan hata: {line}")


def patch_file(rel_path, old, new, hid, note=""):
    """Dosyada birebir metin değişikliği yapar. Beklenen metin dosyada tam 1 kez geçmelidir (assert);
    yama zaten uygulanmışsa tekrar uygulanmaz, böylece hücreler yeniden çalıştırılabilir."""
    p = REPO / rel_path
    text = p.read_text(encoding="utf-8")
    if new in text:          # yeni metin (yama işaretiyle) zaten dosyada → tekrar uygulama
        print(f"[{hid}] zaten uygulanmış → {rel_path}")
        return
    n = text.count(old)
    assert n == 1, f"[{hid}] beklenen metin {rel_path} içinde {n} kez bulundu (1 olmalı)"
    p.write_text(text.replace(old, new), encoding="utf-8")
    if not any(x["id"] == hid and x["dosya"] == rel_path for x in PATCHES):
        PATCHES.append({"id": hid, "dosya": rel_path, "açıklama": note})
    print(f"[{hid}] yama uygulandı → {rel_path}")


def wrap(s, width=110, indent=""):
    return textwrap.fill(" ".join(str(s).split()), width=width, initial_indent=indent,
                         subsequent_indent=" " * len(indent))


print("Çalışma klasörü:", WORK)

## 1. eGOT'a Genel Bakış

### 1.1 Yöntem

Makaleye göre eGOT, LLM tabanlı bir hatla bir belge koleksiyonundaki varlıkları ve ilişkileri bulur, normalleştirir ve bunlardan graf ve vektör veritabanları kurar. Bir soru geldiğinde bu veritabanlarını birkaç tur boyunca LLM ile sorgulayarak cevabı oluşturur [1]. Makale, küçük hücreli akciğer kanseri (SCLC) üzerine 1046 PubMed Central yayınından kurulan bir grafta uzmanların hazırladığı soruları ve lupus–UV ilişkisine dair hipotez üretimini değerlendirir. Ayrıca MultiHop-RAG ve HotpotQA gibi çok adımlı soru-cevap benchmark'larında sonuç raporlar [1, 2].

### 1.2 Kodun yapısı ve kökeni

Repo, birbirinden bağımsız iki bileşenden oluşur [6]:

- **`src/graph-builder/`**: KG kurulumu. Bu kod R. McDermott'un açık kaynak *ai-knowledge-graph* projesinden uyarlanmıştır [8]. `templates/graph_template.html` bu projeye bağlantı verir, `pyproject.toml`'daki paket adı da `ai-knowledge-graph`'tır. eGOT bu koda chunk gömmelerini (`embedding.py`) ve chunk metni + gömme + üçlülerden oluşan "hybrid JSON" çıktısını eklemiştir.
- **`src/knowledge_graph_retrieval/`**: FastAPI sunucusu ve retrieval. `QA_integration.py` ile `constants.py`, Neo4j Labs'ın *LLM Graph Builder* projesinin backend kodundan uyarlanmıştır [7]. Örneğin `VECTOR_GRAPH_SEARCH_QUERY` Cypher sorgusu ve `CHAT_MODE_CONFIG_MAP` yapısı neredeyse aynıdır. eGOT'a özgü kısım, `got.py`'deki Graph-of-Thoughts döngüsü ve `graph_of_thought` modudur.

| Dosya | Görevi |
|---|---|
| `graph-builder/src/knowledge_graph_builder/main.py` | KG hattı: `.txt` → chunk'lar → gömme + LLM ile S-P-O üçlüleri → standardizasyon → ilişki çıkarımı → dosya başına hybrid JSON. Neo4j'ye yazmaz. |
| `.../llm.py` | OpenAI istemcisi (`call_openai`) ve LLM cevabından JSON ayıklama (`extract_json_from_text`). |
| `.../embedding.py` | `all-MiniLM-L6-v2` ile 384 boyutlu chunk gömmesi [15]. |
| `.../entity_standardization.py` | Varlık adı standardizasyonu (kural + LLM) ve ilişki çıkarımı (LLM, geçişli, sözcüksel). |
| `.../prompts.py` | Üçlü çıkarma, varlık çözümleme ve ilişki çıkarımı prompt'ları. |
| `.../visualization.py` | pyvis ile etkileşimli graf (HTML). |
| `graph-builder/src/scripts/neo4j/{hybrid,raw}_json_to_csv_generator.py` | JSON → `documents/chunks/entities/relationships.csv`. Repoda Neo4j'ye yükleme kodu (Cypher, sürücü, indeks) **yoktur**. |
| `graph-builder/src/scripts/multihoprag_generator.py` | MultiHop-RAG kanıtlarını Title/Author/Category/Fact/URL biçiminde `.txt` dosyalarına çevirir. |
| `knowledge_graph_retrieval/app.py` | FastAPI uygulaması: `POST /chat_bot`. |
| `.../QA_integration.py` | LLM (`ChatOpenAI`), gömme modeli, `Neo4jVector` hibrit retriever, sohbet geçmişi, GoT dışı modlar. |
| `.../constants.py` | Cypher retrieval sorguları ve mod ayarları. |
| `.../got.py` | Graph-of-Thoughts döngüsü ve sentez prompt'ları. |
| `.../metrics/evaluation.py` | ragas tabanlı isteğe bağlı değerlendirme. |
| `scripts/eval_multihopqa.py` | EM/F1 (SQuAD tarzı normalizasyon). |
| `scripts/eval_ultradomain.py` | LLM-hakem ile ikili karşılaştırma (Comprehensiveness / Diversity / Empowerment). |

### 1.3 Uçtan uca akış

```
 kanıt .txt dosyaları
        │  graph-builder/main.py
        │   • 100 kelimelik, 20 kelime örtüşmeli chunk'lar   • all-MiniLM-L6-v2 gömmesi (384 boyut)
        │   • LLM: S-P-O üçlüleri (JSON)                       • dosya bazında standardizasyon + ilişki çıkarımı
        ▼
 hybrid JSON (belge başına) ──► raw_json_to_csv_generator.py ──► CSV ──► Neo4j yükleyici (Bölüm 7, bizim yazdığımız)
                                                                            │
   (:Document {fileName})<-[:PART_OF]-(:Chunk {id,text,embedding})-[:HAS_ENTITY]->(:__Entity__ {id,embedding})
   (:__Entity__)-[:<YÜKLEM_TİPİ> {predicate}]->(:__Entity__)      indeksler: "vector" (Chunk.embedding), "keyword" (Chunk.text)
                                                                            │
 soru ──► POST /chat_bot (app.py) ──► QA_integration.QA_RAG ──► got.graph_of_thought_qa
                                                                            │
          ┌──────────────── Graph-of-Thoughts döngüsü (got.py) ────────────────┐
          │ 1) LLM 4–6 thought üretir, 2) LLM puanlar → en iyi 3'ü kuyruğa       │
          │ 3) her derinlikte her thought: hibrit retrieval → ara cevap (LLM)     │
          │    → tüm kanıtla yeni thought'lar (LLM) → puanlama (LLM) → en iyisi   │
          │ 4) bütün kanıt + ara cevaplar → final sentez (LLM)                   │
          └────────────────────────────────────────────────────────────────────┘
```

**KG kurulumu** (`main.py`, `entity_standardization.py`):
- Metin 100 kelimelik, 20 kelime örtüşmeli parçalara bölünür (`config.toml → [chunking]`).
- Her parça 384 boyutlu bir vektöre gömülür.
- LLM'den parçadaki bütün Özne–Yüklem–Nesne (S-P-O) üçlülerini JSON dizisi olarak vermesi istenir (`MAIN_USER_PROMPT`, yüklemler en fazla 3 kelime).
- **Standardizasyon:** Küçük harf ve stopword normalizasyonu, kelime/kök benzerliği ve LLM ile eş anlamlı varlık grupları birleştirilir.
- **İlişki çıkarımı:** Bağlantısız bileşenler LLM ile birbirine bağlanır. Ayrıca bileşen içi LLM çıkarımı, geçişli çıkarım (A→B→C ⇒ A→C) ve sözcüksel çıkarım yapılır. Bu yollarla eklenen üçlüler `inferred: true` olarak işaretlenir.
- **Önemli:** Standardizasyon ve çıkarım **her dosya için ayrı** yapılır. Farklı belgelerdeki aynı varlıklar ancak adları birebir aynıysa (prompt hepsini küçük harfe çevirir) Neo4j'de tek düğümde birleşir.

**Hibrit, grafla genişletilen retrieval** (`constants.VECTOR_GRAPH_SEARCH_QUERY` + LangChain `Neo4jVector`):
1. `vector` (gömme) ve `keyword` (tam metin) indekslerinden gelen chunk skorları, her biri kendi en yüksek skoruna bölünerek birleştirilir.
2. Chunk'lar belgelerine (`PART_OF`) göre gruplanır. Her chunk'ın varlıkları (`HAS_ENTITY`) alınır; en sık geçen 40 varlık tutulur.
3. **Uyarlamalı genişletme:** Soru vektörüyle varlık vektörünün benzerliği `s` olsun. `0.3 ≤ s ≤ 0.9` ise varlıktan 1 adım uzaktaki varlıklar ve ilişkiler eklenir (en fazla 20 yol). `s > 0.9` ise 2 adım uzaktakiler eklenir (en fazla 40 yol). `s < 0.3` ise yalnızca varlığın kendisi kalır (`VECTOR_GRAPH_SEARCH_EMBEDDING_MIN/MAX_MATCH`). Neo4j'nin `vector.similarity.cosine` fonksiyonu sonucu 0–1 aralığına ölçekler [13]. Bizim testimizde birbirine dik iki vektör 0.5 verdi; yani eşikler ham kosinüs değeri değil, bu ölçekteki değerlerdir.
4. LLM'e giden bağlam üç bölümden oluşur: `Text Content` (chunk metinleri), `Entities` ve `Relationships` (`:özne İLİŞKİ_TİPİ :nesne`). İlişkinin tipi (`type(r)`) doğrudan bağlama yazıldığı için yüklemin ilişki tipinde korunması önemlidir (Bölüm 7.2).

**Graph-of-Thoughts döngüsü** (`got.py → conduct_graph_of_thought`):
1. LLM sorudan 4–6 thought üretir. Ayrı bir çağrıda her birine 0–1 arası puan verir. En iyi 3 thought kuyruğa alınır.
2. Her derinlikte kuyruktaki her thought için sırasıyla: (a) hibrit retrieval, (b) bulunan belgelerle bir ara cevap (`CHAT_SYSTEM_TEMPLATE`), (c) o ana kadar toplanan bütün kanıtla yeni thought'lar, (d) bu thought'ların puanlanması. Her daldan en iyi 1 thought bir sonraki derinliğe geçer, yani kuyruk 3 dal olarak kalır.
3. Bütün thought'ların kanıtı ve ara cevapları birleştirilir; son bir LLM çağrısı final cevabı sentezler.

**Soru başına LLM çağrısı:** 2 (ilk üretim ve puanlama) + 3 dal × 3 çağrı × `max_depth` + 1 (sentez) = **2 + 9·d + 1**. `d = 2` için 21, repo varsayılanı `d = 3` için 30 çağrı eder. Bir thought'un retrieval'ı boş dönerse ara cevap çağrısı atlanır ve sayı biraz düşer. Kodun yapısı gereği son derinlikte üretilip puanlanan thought'lar (3 × 2 = 6 çağrı) hiç keşfedilmez.

**Derinlik seçimi:** Makale, benchmark sonuçlarının hepsinde maksimum derinliğin 2 iterasyon ve iterasyon başına 3 thought olarak ayarlandığını belirtir. Lab blogu da benchmark'larda tipik olarak iki iterasyon kullanıldığını yazar [1, 2]. Bu yüzden notebook'ta `MAX_DEPTH = 2` varsayılandır; repo kodundaki varsayılan 3'tür.

## 2. Kurulum

### 2.1 Repoyu klonlama

Repo, ödev için incelenen commit'e (`3d37906…`, `master`'ın 3 Kasım 2026'daki HEAD'i) sabitlenir. Böylece repo daha sonra güncellense bile notebook aynı kodu çalıştırır ve Bölüm 4'teki yamalar doğru yere uygulanır.

In [ ]:
# 2.1 Repoyu klonla ve commit'e sabitle
if not (REPO / ".git").exists():
    run(["git", "clone", "-q", REPO_URL, str(REPO)], tail=5)
run(["git", "-C", str(REPO), "checkout", "-q", REPO_COMMIT], tail=5)   # yerel yamalar korunur (yeniden çalıştırmada)
_, head = run(["git", "-C", str(REPO), "rev-parse", "HEAD"], show=False)
assert head.strip() == REPO_COMMIT, f"Beklenmeyen commit: {head.strip()}"
run(["git", "-C", str(REPO), "log", "-1", "--format=HEAD = %H (%ad) — %s", "--date=short"])

# Repo içeriğine kısa bakış: Python dosyaları ve satır sayıları
rows = []
for p in sorted(REPO.rglob("*.py")):
    rel = p.relative_to(REPO)
    rows.append({"dosya": str(rel), "satır": sum(1 for _ in p.open(encoding="utf-8", errors="ignore"))})
display(pd.DataFrame(rows))
print("datasets/ klasörü:", ", ".join(sorted(x.name for x in (REPO / "datasets").iterdir())))

### 2.2 graph-builder README'si ve `requirements.txt` neden doğrudan kullanılamıyor? (H1, H2)

- **H1:** `src/graph-builder/README.md` (satır 51 ve 150), "tam hattı tek komutla çalıştırmak" için `python src/main.py --query ...` komutunu verir. Bu komut PubMed'den makale indirip KG kuruyordu. Repoda böyle bir dosya yoktur; yalnızca `src/knowledge_graph_builder/main.py` vardır. Aşağıdaki hücre komutu çalıştırıp hatayı gösterir. Biz builder'ı doğrudan `knowledge_graph_builder/main.py` ile çalıştırıyoruz (Bölüm 6).
- **H2:** `src/graph-builder/requirements.txt` donmuş bir ortamın listesidir: 84 paketin neredeyse hepsi sabit sürümlüdür. Aralarında kodun hiç import etmediği büyük paketler de vardır (`indra`, `boto3`, `pysb`, `biopython`, `ndex2` …). Liste ayrıca Colab'da kurulu olan paketleri [16] eski sürümlere sabitler (örneğin `sympy==1.11.1`; Colab'daki torch daha yeni bir sympy ister). Hücre önce bu çakışmaları listeler, sonra `pip install --dry-run` ile kurulumu **hiçbir şey kurmadan** dener. Bu denemede kurulum `pysb==1.16.0`'da duruyor. Paket yalnızca kaynak kodu olarak yayımlanmıştır ve `setup.py`'si eski `ez_setup.py`'yi çağırır. pip'in yalıtılmış derleme ortamındaki güncel setuptools (84.x) artık `pkg_resources` içermediği için `ez_setup` `setuptools-18.3.2.zip`'i `pypi.python.org`'dan indirmeye çalışır. Bu adres yönlendirme döndürür; yönlendirmeyi takip etmeyen `curl` ZIP yerine yönlendirme cevabını kaydettiği için derleme `BadZipFile` ile durur (Colab ile aynı Python/paket sürümlerine sahip test ortamımızda gözlendi). Bu adım yaklaşık 3 dakika sürer; atlamak için `SHOW_REQUIREMENTS_DRYRUN = False` yapın.

**Çözüm:** Bölüm 2.4'te yalnızca kodun gerçekten import ettiği paketler kurulur. Colab'da hazır gelen `torch`, `transformers`, `sentence-transformers`, `numpy` ve `pandas` değiştirilmez.

In [ ]:
# 2.2 README'deki PubMed hattı ve graph-builder/requirements.txt
import importlib.metadata as md

# H1 — README'deki "python src/main.py ..." komutu: böyle bir dosya yok
code, out = run([sys.executable, "src/main.py", "--help"], cwd=BUILDER_DIR, tail=3)
if code != 0:
    record_error("H1", "graph-builder README: python src/main.py", out, pattern=r"can't open file|No such file")

# H2 — requirements.txt: ne kadar büyük ve Colab'daki sürümlerle nerede çakışıyor?
req = [l.strip() for l in (BUILDER_DIR / "requirements.txt").read_text().splitlines() if l.strip() and not l.startswith("#")]
pinned = [l for l in req if "==" in l]
print(f"\nrequirements.txt: {len(req)} paket, {len(pinned)} tanesi '==' ile sabitlenmiş.")
unused = ["indra", "boto3", "pysb", "biopython", "ndex2", "pybiopax", "mygene"]
print("Kodun hiçbir yerde import etmediği paketlerden bazıları:", ", ".join(u for u in unused if any(l.split("==")[0] == u for l in req)))
rows = []
for l in pinned:
    name, ver = l.split("==", 1)
    try:
        inst = md.version(name)
    except md.PackageNotFoundError:
        continue
    if inst != ver:
        rows.append({"paket": name, "requirements.txt": ver, "Colab'da kurulu": inst})
CORE = {"numpy", "pandas", "scipy", "openai", "sympy", "requests", "pydantic", "pydantic_core", "setuptools",
        "lxml", "SQLAlchemy", "httpx", "tqdm", "Jinja2", "urllib3", "packaging"}
print(f"Colab'da kurulu olup requirements.txt'nin farklı sürüme sabitlediği paket sayısı: {len(rows)}; bunlardan bazıları:")
display(pd.DataFrame([r for r in rows if r["paket"] in CORE]))
try:
    print("Not: torch", md.version("torch"), "→ gereksinimi:",
          [r for r in md.requires("torch") if r.lower().startswith("sympy")], "| requirements.txt: sympy==1.11.1")
except Exception:
    pass

if SHOW_REQUIREMENTS_DRYRUN:
    print("\npip install --dry-run -r requirements.txt (hiçbir şey kurulmaz, yalnızca çözümleme denenir; ~3 dk)...")
    t0 = time.time()
    code, out = run([sys.executable, "-m", "pip", "install", "--dry-run", "-r", "requirements.txt"],
                    cwd=BUILDER_DIR, timeout=1200, show=False)
    keep = [l for l in out.splitlines() if re.search(r"^ERROR|Collecting pysb|Downloading .*setuptools-18|BadZipFile|Would install", l)]
    print("\n".join(l[:300] for l in keep[-12:]))
    print(f"(çıkış kodu {code}, {time.time() - t0:.0f} sn)")
    if code != 0:
        record_error("H2", "pip install -r src/graph-builder/requirements.txt", out, pattern=r"^ERROR")

### 2.3 Retrieval bileşeninin `requirements.txt` dosyası yok (H3)

Kök `README.md` (satır 39) sunucu için `src/knowledge_graph_retrieval/` klasöründe `pip install -r requirements.txt` çalıştırmayı söyler, ama klasörde böyle bir dosya yoktur. Gerekli paketleri kodun import'larından çıkardık. Bunlar `langchain_*` paketleri, `fastapi`, `uvicorn` ve `python-dotenv`'dir. Ayrıca `python-multipart` (FastAPI'nin `Form(...)` alanları için) ve `tiktoken` (`TokenTextSplitter` için) gerekir. Bu son iki paket Colab'da zaten kurulu gelir [16].

In [ ]:
# 2.3 Retrieval README'si: "pip install -r requirements.txt" — dosya yok (H3)
code, out = run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=RETR_DIR, tail=3)
if code != 0:
    record_error("H3", "src/knowledge_graph_retrieval: pip install -r requirements.txt (README)", out, pattern=r"^ERROR")

### 2.4 Minimal paket kurulumu

LangChain paketleri, LangChain 1.x ile çalıştığını doğruladığımız sürümlere sabitlenmiştir. Kodun kullandığı eski modüller (`langchain.prompts`, `langchain.retrievers`) 1.x'te `langchain-core` ve `langchain-classic` paketlerine taşındı [14]; bunun için Bölüm 4.7'de yama yapılır. ragas **kurulmaz**: değerlendirme varsayılan olarak kapalıdır ve kurulursa Colab'daki paketleri değiştirir (Bölüm 4.8). Hücre, kurulumdan önce ve sonra Colab'ın önemli paketlerinin sürümlerini karşılaştırır.

> Not: `langchain-community 0.4.2`, `requests>=2.32.5` istediği için `requests` küçük bir sürüm atlayabilir. pip bu sırada Colab'ın kendi paketleriyle ilgili sürüm uyarısı yazdırabilir. Bu uyarılar notebook'un çalışmasını etkilemez.

In [ ]:
# 2.4 Yalnızca kodun gerçekten import ettiği paketleri kur (Colab'da hazır gelenler değiştirilmez)
import importlib.metadata as md


def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return "—"


WATCH = ["torch", "transformers", "sentence-transformers", "numpy", "pandas", "openai",
         "langchain-core", "fastapi", "uvicorn", "requests"]
before = {p: _ver(p) for p in WATCH}

PKGS = [
    # retrieval: QA_integration.py / got.py / app.py (LangChain 1.x ile çalışan sürümler, Bölüm 4.7)
    "langchain-classic==1.0.8", "langchain-community==0.4.2", "langchain-neo4j==0.11.0",
    "langchain-openai==1.7.3", "langchain-huggingface==1.2.3",
    "fastapi", "uvicorn", "python-multipart",  # FastAPI Form alanları python-multipart ister
    "python-dotenv", "tiktoken",               # load_dotenv(); TokenTextSplitter tiktoken ister
    # graph-builder: main.py, llm.py, config.py, visualization.py, utils/pdf_chunking.py
    "tomli", "pyvis==0.3.2", "PyPDF2==3.0.1", "networkx", "sentence-transformers", "openai",
    # scripts/eval_ultradomain.py (Bölüm 4.9'daki import testi için)
    "jsonlines",
]
code, out = run([sys.executable, "-m", "pip", "install", "-q", *PKGS], tail=12, timeout=1800)
assert code == 0, "pip kurulumu başarısız"

after = {p: _ver(p) for p in WATCH + ["langchain-classic", "langchain-community", "langchain-neo4j",
                                      "langchain-openai", "langchain-huggingface", "neo4j", "tomli", "pyvis", "PyPDF2"]}
display(pd.DataFrame([{"paket": p, "önce": before.get(p, "—"), "sonra": v,
                       "durum": ("yeni" if before.get(p, "—") == "—" else ("DEĞİŞTİ" if before[p] != v else "aynı"))}
                      for p, v in after.items()]))

### 2.5 Neo4j kurulumu

Retrieval sorgusu iki özellik ister: Neo4j 5'in `vector.similarity.cosine` fonksiyonu [13] ile *quantified path pattern* sözdizimi (`(()-[rels]-()){0,2}`) ve APOC Core fonksiyonları (`apoc.text.join`, `apoc.coll.*`) [12]. Bu yüzden Neo4j **Community 5.26 LTS** kullanıyoruz.

**`NEO4J_MODE = "local"` (varsayılan):**
1. OpenJDK 21 kurulur (`apt-get`). Neo4j 5.x Java 17 ile, 5.14'ten itibaren Java 21 ile de çalışır; 5.26 LTS için Java 21 önerilir [11].
2. Neo4j Community tarball'ı `dist.neo4j.org`'dan indirilir. `NEO4J_VERSION` bulunamazsa Maven Central'daki en yeni 5.26.x yamaları sırayla denenir.
3. APOC Core jar'ı dağıtımın `labs/` klasöründen `plugins/` klasörüne kopyalanır [12].
4. `neo4j.conf` ayarlanır: yalnızca yerel bağlantı, 1 GB heap, 512 MB page cache, anonim kullanım raporu kapalı.
5. İlk şifre `neo4j-admin dbms set-initial-password` ile, **ilk başlatmadan önce** belirlenir [11]. Şifre rastgele üretilir ve yazdırılmaz.
6. `neo4j start` çalıştırılır ve Bolt portunun (7687) açılması beklenir.

**`NEO4J_MODE = "aura"` (yedek):** Neo4j AuraDB Free örneğinin bağlantı bilgileri Secrets'tan (`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`) okunur. Bölüm 7'deki yükleyici yalnızca bu notebook'un kullandığı etiketleri (`Chunk`, `Document`, `__Entity__`, `Session`, `Message`) siler, ama yine de bu ödev için **ayrı bir Aura örneği** kullanmanız önerilir.

In [ ]:
# 2.5 Neo4j: local (Colab'a Neo4j Community 5.26 LTS + APOC) veya aura (AuraDB Free)
import requests
from neo4j import GraphDatabase

NEO4J_HOME = WORK / "neo4j"
NEO4J_DIST_URL = "https://dist.neo4j.org/neo4j-community-{v}-unix.tar.gz"
MAVEN_META = "https://repo1.maven.org/maven2/org/neo4j/neo4j/maven-metadata.xml"   # 5.26.x yama listesi için
PW_FILE = WORK / ".neo4j_local_password"   # yerel kurulumun rastgele şifresi (dosyada, yazdırılmaz)


def _java21_home():
    for cand in ("/usr/lib/jvm/java-21-openjdk-amd64", "/usr/lib/jvm/java-21-openjdk"):
        if Path(cand, "bin", "java").exists():
            return cand
    return None


def _candidate_versions():
    vers = [NEO4J_VERSION]
    try:
        xml = requests.get(MAVEN_META, timeout=30).text
        found = sorted(set(re.findall(r"<version>(5\.26\.\d+)</version>", xml)),
                       key=lambda v: int(v.split(".")[2]), reverse=True)
        vers += [v for v in found if v != NEO4J_VERSION][:5]
    except Exception as e:
        print("Maven sürüm listesi alınamadı:", type(e).__name__)
    return vers


def _download(url, dest):
    part = Path(str(dest) + ".part")                 # yarım kalan indirme tam dosya sanılmasın
    with requests.get(url, stream=True, timeout=120) as r:
        if r.status_code != 200:
            return False
        with open(part, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    part.rename(dest)
    return True


def _set_conf(conf, settings):
    """neo4j.conf'ta aynı anahtarın etkin satırlarını yoruma alır, yeni değerleri sona ekler."""
    lines = []
    for line in conf.read_text().splitlines():
        key = line.split("=", 1)[0].strip()
        lines.append("#" + line if (not line.lstrip().startswith("#") and key in settings) else line)
    lines += ["", "# --- eGOT Colab ayarları ---"] + [f"{k}={v}" for k, v in settings.items()]
    conf.write_text("\n".join(lines) + "\n")


def _bolt_open(port=7687):
    try:
        socket.create_connection(("127.0.0.1", port), timeout=2).close()
        return True
    except OSError:
        return False


def neo4j_local_setup():
    env = os.environ.copy()
    jh = _java21_home()
    if jh is None:                                   # 1) Java 21 (Neo4j 5.26 Java 17 veya 21 ister)
        print("OpenJDK 21 kuruluyor (apt-get)...")
        run("apt-get update -qq > /dev/null && apt-get install -y -qq openjdk-21-jre-headless > /dev/null", tail=5, timeout=1200)
        jh = _java21_home()
    assert jh, "Java 21 kurulamadı"
    env["JAVA_HOME"] = jh
    run([f"{jh}/bin/java", "-version"], tail=1)

    if not (NEO4J_HOME / "bin" / "neo4j").exists():  # 2) Neo4j Community tarball'ı
        for v in _candidate_versions():
            url, tgz = NEO4J_DIST_URL.format(v=v), WORK / f"neo4j-community-{v}-unix.tar.gz"
            print("İndiriliyor:", url)
            if tgz.exists() or _download(url, tgz):
                code, _ = run(["tar", "-xzf", str(tgz), "-C", str(WORK)], tail=3)
                if code == 0:
                    shutil.move(str(WORK / f"neo4j-community-{v}"), str(NEO4J_HOME))
                    break
                tgz.unlink()                         # bozuk arşiv: sil, sıradaki sürümü dene
            print("  bulunamadı, bir sonraki 5.26.x sürümü deneniyor")
        else:
            raise RuntimeError("Neo4j indirilemedi. NEO4J_MODE = 'aura' ile AuraDB Free kullanabilirsiniz.")
        apoc = sorted((NEO4J_HOME / "labs").glob("apoc-*core*.jar"))   # 3) APOC Core: labs/ → plugins/
        assert apoc, "labs/ klasöründe APOC Core jar'ı bulunamadı"
        for j in apoc:
            shutil.copy(j, NEO4J_HOME / "plugins" / j.name)
        print("APOC kopyalandı:", [j.name for j in apoc])
        _set_conf(NEO4J_HOME / "conf" / "neo4j.conf", {             # 4) yapılandırma
            "server.default_listen_address": "127.0.0.1",
            "server.memory.heap.initial_size": "1g",
            "server.memory.heap.max_size": "1g",
            "server.memory.pagecache.size": "512m",
            "dbms.security.procedures.unrestricted": "apoc.*",
            "dbms.usage_report.enabled": "false",
        })
        pw = secrets.token_urlsafe(18)                               # 5) ilk şifre (rastgele, yazdırılmaz)
        PW_FILE.write_text(pw); PW_FILE.chmod(0o600)
        code, out = run([str(NEO4J_HOME / "bin" / "neo4j-admin"), "dbms", "set-initial-password", pw], env=env, tail=2)
        assert code == 0, "neo4j-admin set-initial-password başarısız"
    if not _bolt_open():                                             # 6) başlat ve Bolt portunu bekle
        code, _ = run([str(NEO4J_HOME / "bin" / "neo4j"), "start"], env=env, tail=6, timeout=300)
        if code != 0:
            run(["tail", "-n", "40", str(NEO4J_HOME / "logs" / "neo4j.log")])
            raise RuntimeError("neo4j start başarısız (log yukarıda)")
    t0 = time.time()
    while not _bolt_open():
        if time.time() - t0 > 240:
            run(["tail", "-n", "40", str(NEO4J_HOME / "logs" / "neo4j.log")])
            raise RuntimeError("Neo4j 240 sn içinde Bolt portunu açmadı")
        time.sleep(2)
    print(f"Bolt portu (7687) açık ({time.time() - t0:.0f} sn beklendi).")
    os.environ.update({"NEO4J_URI": "bolt://localhost:7687", "NEO4J_USERNAME": "neo4j",
                       "NEO4J_PASSWORD": PW_FILE.read_text().strip(), "NEO4J_DATABASE": "neo4j"})


def neo4j_aura_setup():
    os.environ["NEO4J_URI"] = get_secret("NEO4J_URI")
    os.environ["NEO4J_USERNAME"] = get_secret("NEO4J_USERNAME", required=False) or "neo4j"
    os.environ["NEO4J_PASSWORD"] = get_secret("NEO4J_PASSWORD")
    os.environ["NEO4J_DATABASE"] = get_secret("NEO4J_DATABASE", required=False) or "neo4j"
    print("AuraDB bağlantı bilgileri Secrets'tan okundu (gösterilmez).")


assert NEO4J_MODE in ("local", "aura"), "NEO4J_MODE 'local' ya da 'aura' olmalı"
neo4j_local_setup() if NEO4J_MODE == "local" else neo4j_aura_setup()

DB = os.environ["NEO4J_DATABASE"]
DRIVER = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
                              notifications_min_severity="OFF")   # sunucu uyarılarını (deprecation vb.) çıktıya basma
_err = None
for _ in range(30):                     # Bolt açıldıktan sonra kimlik doğrulama birkaç saniye gecikebilir
    try:
        DRIVER.verify_connectivity()
        _err = None
        break
    except Exception as e:
        _err = e
        if "auth" in type(e).__name__.lower():   # yanlış kullanıcı/şifre: beklemenin anlamı yok
            break
        time.sleep(2)
if _err is not None:
    hint = ("Secrets'taki NEO4J_URI / NEO4J_USERNAME / NEO4J_PASSWORD değerlerini kontrol edin (Aura örneği "
            "duraklatılmış olabilir)." if NEO4J_MODE == "aura" else f"Neo4j loglarına bakın: {NEO4J_HOME / 'logs'}")
    raise RuntimeError(f"Neo4j'ye bağlanılamadı: {hint}\nAsıl hata: {type(_err).__name__}: {redact(str(_err))[:300]}") from None


def cypher(query, **params):
    records, _, _ = DRIVER.execute_query(query, params, database_=DB)
    return [r.data() for r in records]


display(pd.DataFrame(cypher("CALL dbms.components() YIELD name, versions, edition "
                            "RETURN name, versions[0] AS version, edition")))

### 2.6 Yerel dil modeli: Ollama ve Qwen2.5-7B-Instruct

Bu notebook dil modelini varsayılan olarak Colab'ın GPU'sunda **yerel** çalıştırır (gerekçe: Bölüm 3). Hücre sırasıyla şunları yapar:

1. GPU'yu kontrol eder. GPU yoksa durur ve çalışma zamanının nasıl değiştirileceğini yazar.
2. Ollama'yı [18] kurar. `OLLAMA_VERSION` sürümünün Linux paketi GitHub'dan indirilir ve `/usr/local` altına açılır. Ollama'nın resmî kurulum betiği de aynı paketi aynı yere açar; betik burada yalnızca yedek olarak kullanılır. Sürüm sabit olduğu için notebook her çalıştırmada aynı Ollama'yı kullanır.
3. Ollama sunucusunu arka planda başlatır (`127.0.0.1:11434`, log: `logs/ollama.log`). Sunucu OpenAI-uyumlu bir uç nokta (`/v1/chat/completions`) sunar [18]. eGOT'un OpenAI istemcileri bu adrese kodda değişiklik yapılmadan bağlanır.
4. `qwen2.5:7b` modelini indirir. Bu, Qwen2.5-7B-Instruct'ın [19, 20] Ollama kütüphanesindeki 4-bit nicemlenmiş sürümüdür. Ardından modeli GPU belleğine yükler. Modelin boyutu ve GPU'ya yüklenen oranı çıktıda yazar.

Sunucu ayarları:

- **Bağlam penceresi:** `OLLAMA_CONTEXT_LENGTH = 16384` token. Ollama'nın varsayılanı GPU belleğine göre seçilir ve 23 GiB'tan küçük GPU'larda (T4 dahil) 4096 token'dır. Bu sürümde pencereye sığmayan bir istek kesilmez, hata ile reddedilir. Bu yüzden pencere, eGOT'un en uzun prompt'larına yetecek kadar büyük seçildi. Bu çalıştırmadaki en uzun istek Bölüm 11.0'da raporlanır.
- **Paralellik:** Aynı anda 2 istek işlenir (`OLLAMA_NUM_PARALLEL=2`); KG kurulumundaki 2 işçiyle (`BUILDER_WORKERS`) uyumludur.
- **Bellek:** Model GPU belleğinde sürekli tutulur (`OLLAMA_KEEP_ALIVE=-1`).
- **Bulut:** Ollama'nın bulut özellikleri kapalıdır (`OLLAMA_NO_CLOUD=1`); model yalnızca bu Colab oturumunda çalışır.

Bölüm 0.1'de LLM için bir API seçildiyse bu hücre hiçbir şey kurmaz.

In [ ]:
# 2.6 Yerel dil modeli: Ollama'yı kur, sunucuyu başlat, modeli indir ve GPU'ya yükle (LLM bir API ise atlanır)
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_LOG = LOG_DIR / "ollama.log"
OLLAMA_NUM_PARALLEL = 2


def gpu_info():
    """GPU adı ve bellek kullanımı (GPU yoksa None)."""
    if shutil.which("nvidia-smi") is None:
        return None
    code, out = run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader", show=False)
    return out.strip() if code == 0 and out.strip() else None


def ollama_up():
    try:
        return requests.get(OLLAMA_URL + "/api/version", timeout=3).ok
    except requests.RequestException:
        return False


def ollama_log_tail(n=15):
    return "\n".join(OLLAMA_LOG.read_text(errors="replace").splitlines()[-n:]) if OLLAMA_LOG.exists() else "(log yok)"


HAS_GPU = gpu_info() is not None
print("GPU:", gpu_info() or "yok (CPU çalışma zamanı)")

if not LOCAL_LLM:
    print(f"Dil modeline API üzerinden erişiliyor ({LLM_BASE_URL}); Ollama kurulmadı.")
else:
    if not HAS_GPU and os.environ.get("EGOT_ALLOW_CPU_LLM") != "1":
        raise RuntimeError(
            "Yerel dil modeli için GPU gerekir, ama bu çalışma zamanında GPU yok.\n"
            "Çözüm: 'Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU' seçip kaydedin, sonra "
            "'Çalışma zamanı → Tümünü çalıştır'. (GPU yerine ücretli bir API kullanacaksanız Bölüm 0.1'deki üç değeri "
            "değiştirin.)")

    # (a) Kurulum: resmî kurulum betiğinin yaptığı gibi Linux paketini /usr/local altına açar; sürüm sabittir
    if shutil.which("ollama") is None:
        t0 = time.time()
        if shutil.which("zstd") is None:   # paket .tar.zst biçiminde
            run("apt-get update -qq > /dev/null && apt-get install -y -qq zstd > /dev/null", tail=3, timeout=900)
        url = f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
        code, _ = run(["bash", "-c", f"set -o pipefail; curl -fsSL '{url}' | zstd -dq | tar -xf - -C /usr/local"],
                      tail=5, timeout=1800)
        if code != 0 or shutil.which("ollama") is None:
            print("Sabit sürüm kurulamadı; Ollama'nın resmî kurulum betiği (en yeni sürüm) deneniyor.")
            run("curl -fsSL https://ollama.com/install.sh | sh", tail=8, timeout=1800)
        assert shutil.which("ollama"), "Ollama kurulamadı; yukarıdaki çıktıya bakın."
        print(f"Ollama kuruldu: {time.time() - t0:.0f} sn")

    # (b) Sunucuyu arka planda başlat
    if not ollama_up():
        env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(OLLAMA_CONTEXT_LENGTH),
                   OLLAMA_NUM_PARALLEL=str(OLLAMA_NUM_PARALLEL), OLLAMA_KEEP_ALIVE="-1", OLLAMA_NO_CLOUD="1")
        subprocess.Popen(["ollama", "serve"], env=env, stdout=open(OLLAMA_LOG, "ab"), stderr=subprocess.STDOUT,
                         start_new_session=True)
        for _ in range(60):
            if ollama_up():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama sunucusu başlamadı. Log'un sonu:\n" + ollama_log_tail())
    print("Ollama sürümü:", requests.get(OLLAMA_URL + "/api/version", timeout=5).json().get("version"))

    # (c) Modeli indir (yalnızca ilk seferde)
    def find_model():
        models = requests.get(OLLAMA_URL + "/api/tags", timeout=10).json().get("models", [])
        return next((m for m in models if m.get("name") in (LLM_MODEL, LLM_MODEL + ":latest")), None)

    info = find_model()
    if info is None:
        t0 = time.time()
        print(f"{LLM_MODEL} indiriliyor (yalnızca ilk seferde) ...")
        r = requests.post(OLLAMA_URL + "/api/pull", json={"model": LLM_MODEL, "stream": False}, timeout=3600)
        if not r.ok or r.json().get("status") != "success":
            raise RuntimeError(f"Model indirilemedi ({r.status_code}): {r.text[:300]}")
        info = find_model()
        print(f"İndirildi: {time.time() - t0:.0f} sn")
    d = info.get("details", {})
    print(f"Model: {info['name']} | {info['size'] / 1e9:.1f} GB | {d.get('parameter_size')} parametre | "
          f"nicemleme: {d.get('quantization_level')} | modelin desteklediği bağlam: {d.get('context_length', '?')} token")

    # (d) Modeli belleğe yükle ve orada tut: ilk sorudaki yükleme süresi ölçümlere karışmasın
    t0 = time.time()
    requests.post(OLLAMA_URL + "/api/generate", json={"model": LLM_MODEL, "keep_alive": -1}, timeout=900).raise_for_status()
    for p in requests.get(OLLAMA_URL + "/api/ps", timeout=10).json().get("models", []):
        share = min(1.0, p.get("size_vram", 0) / max(1, p.get("size", 1)))
        where = f"GPU'da %{100 * share:.0f}" if HAS_GPU else "CPU'da"
        print(f"Belleğe yüklendi ({time.time() - t0:.0f} sn): {p['name']} | {p['size'] / 1e9:.1f} GB, {where} | "
              f"bağlam penceresi: {p.get('context_length')} token, aynı anda {OLLAMA_NUM_PARALLEL} istek")
        if HAS_GPU and share < 0.99:
            print("⚠️  Modelin bir kısmı CPU belleğinde kaldı; cevaplar yavaşlar.")
    if HAS_GPU:
        print("GPU (ad, kullanılan bellek, toplam bellek):", gpu_info())

## 3. Dil Modeli Seçimi ve Gerekçesi

**Seçilen model: Qwen2.5-7B-Instruct** [19, 20]. Model, Colab'ın T4 GPU'sunda **Ollama** [18] ile yerel olarak çalışır (`qwen2.5:7b`, 4-bit nicemlenmiş). KG kurulumunda ve retrieval'da **aynı model** kullanılır.

**Hazırlanan modeller:** Dil modeli Bölüm 2.6'da Ollama kütüphanesinden bir kez indirilir; gerçek boyutu o hücrenin çıktısında yazar. eGOT'un sabit kodladığı gömme modeli `sentence-transformers/all-MiniLM-L6-v2` (384 boyut) [15] ise ilk kullanımda (Bölüm 4.2) Hugging Face'ten indirilir ve önbelleğe alınır. Builder, CSV üreticisi ve sunucu aynı önbelleği kullanır.

| Ödevdeki kriter | Değerlendirme |
|---|---|
| **eGOT ile uyum** | eGOT'un bütün LLM çağrıları OpenAI istemcisiyle yapılır. Builder `openai.OpenAI` kullanır (`llm.py`), retrieval `langchain_openai.ChatOpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)` kullanır (`QA_integration.py:45-50`). Ollama OpenAI-uyumlu bir uç nokta sunduğu için [18] model, kodda değişiklik yapılmadan yalnızca adres ve model adıyla bağlanır. Yazarlar da builder'da açık bir model kullanmış: `llm.py`'deki sabit adres, Jetstream'de çalışan Llama 4 Scout'u gösterir (H7). |
| **Colab'da çalıştırılabilirlik, GPU/RAM** | Colab'ın T4 GPU'sunun 16 GB belleği vardır. 4-bit nicemlenmiş 7B model ve 16.384 token'lık bağlam penceresi (2 paralel istek için) bu belleğe rahatça sığar; GPU'da kullanılan bellek Bölüm 2.6'da yazar. CPU'da 7B'lik bir model, soru başına ~21 LLM çağrısı (Bölüm 1.3) için fazla yavaştır. Bu yüzden **T4 GPU çalışma zamanı** gerekir. |
| **Cevap kalitesi ve JSON güvenilirliği** | eGOT'un her aşaması yapılandırılmış çıktı bekler: üçlüler JSON dizisi, thought'lar ve puanlar JSON nesnesi olmalıdır (`JsonOutputParser`). Qwen2.5'in model kartı, yapılandırılmış çıktı (özellikle JSON) üretimindeki iyileşmeyi öne çıkarır [20]. Bölüm 3.1'deki küçük test modelin bu biçime uyduğunu gösterir. Bu çalıştırmada üçlü çıkarılamayan belge sayısı Bölüm 6.3'te, boş dönen retrieval adımları Bölüm 9'da raporlanır. 7B'lik nicemlenmiş bir model büyük ticari modellerden zayıf kalabilir; bu, Bölüm 11.5'te bir sınırlama olarak ele alınır. |
| **Maliyet / API kullanımı** | Ücretsizdir: API anahtarı, kredi ya da ödeme gerekmez. Harcanan kaynak süre ve Colab'ın GPU kotasıdır. Token sayıları yine ölçülür (Bölüm 9.3 ve 11.0). |
| **Bağlam uzunluğu** | Model kartına göre model 32.768 token'a kadar bağlamı doğrudan destekler [20]. Burada 16.384 token kullanılır. Ollama'nın bu GPU'daki varsayılanı (4.096 token) eGOT'un uzun prompt'ları için riskliydi (Bölüm 2.6). |
| **Lisans ve gizlilik** | Ağırlıklar açıktır ve Apache 2.0 lisanslıdır [20]. Veriler ve prompt'lar Colab oturumunun dışına çıkmaz. |

**Ücretli alternatif: OpenAI `gpt-4o-mini`.** Aynı hat, Bölüm 0.1'deki üç değer (`"https://api.openai.com/v1"`, `"gpt-4o-mini"`, `"OPENAI_API_KEY"`) değiştirilerek, GPU olmadan CPU çalışma zamanında da çalışır. Repodaki final sentez adımı da `gpt-4o-mini` adını sabit yazar (`got.py:266`). Bu seçeneği iki nedenle varsayılan yapmadık. Birincisi, OpenAI API'si önceden kredi satın almayı gerektirir [21]; fiyatı 1M girdi token'ı için 0,15 \$, 1M çıktı token'ı için 0,60 \$'dır [10]. İkincisi, `gpt-4o-mini-2024-07-18` sürümü 14 Ocak 2027'de kapanacaktır [9]; açık bir model ise notebook'un ileride de aynı şekilde çalışmasını sağlar.

**Seçmediğimiz modeller:** 14B'lik bir model T4'e sığar, ama soru başına ~21 çağrıda belirgin biçimde yavaşlar. 32B ve üstü modeller 4-bit hâlde bile T4'ün belleğine sığmaz. 0,5–3B'lik modeller daha hızlıdır, ancak eGOT'un JSON çıktısı ve çok adımlı akıl yürütme beklentileri için zayıf kalma riskleri yüksektir. Akıl yürütme ("thinking") modelleri ise cevaptan önce uzun bir düşünme metni üretir. Bu hem süreyi uzatır hem de eGOT'un JSON ayrıştırıcılarını bozabilir.

### 3.1 Bağlantı ve JSON testi

In [ ]:
# 3.1 Seçilen modele kısa bir bağlantı ve JSON testi (eGOT'un her aşaması JSON çıktı bekler)
import openai
from openai import OpenAI

_client = OpenAI(base_url=LLM_BASE_URL, api_key=os.environ["OPENAI_API_KEY"], max_retries=3)
API_HINTS = {   # sık görülen hesap/anahtar sorunları için anlaşılır açıklama (asıl hata mesajı da yazdırılır)
    openai.AuthenticationError: f"API anahtarı geçersiz. Secrets'taki '{LLM_API_KEY_SECRET}' değerini kontrol edin.",
    openai.PermissionDeniedError: "Anahtarın bu modele ya da projeye erişim izni yok.",
    openai.RateLimitError: "Hız sınırı ya da kota aşıldı. 'insufficient_quota' görüyorsanız OpenAI hesabında kredi/ödeme "
                           "tanımlı değildir (platform.openai.com → Billing).",
    openai.NotFoundError: f"Model bulunamadı: LLM_MODEL = {LLM_MODEL!r} ve LLM_BASE_URL'i kontrol edin.",
    openai.APIConnectionError: f"Sunucuya bağlanılamadı: LLM_BASE_URL = {LLM_BASE_URL!r}.",
}
if LOCAL_LLM:   # yerel Ollama için daha isabetli açıklamalar
    API_HINTS[openai.APIConnectionError] = "Ollama sunucusuna bağlanılamadı; Bölüm 2.6'nın çıktısına bakın."
    API_HINTS[openai.NotFoundError] = f"Model Ollama'da yok: LLM_MODEL = {LLM_MODEL!r} (Bölüm 2.6 indirir)."
t0 = time.time()
try:
    resp = _client.chat.completions.create(
        model=LLM_MODEL, temperature=0, max_tokens=60,
        messages=[{"role": "user", "content": 'Return only this JSON array, filled in: '
                   '[{"subject": "...", "predicate": "...", "object": "..."}] for the sentence: "Marie Curie discovered polonium."'}])
except tuple(API_HINTS) as e:
    hint = next(h for cls, h in API_HINTS.items() if isinstance(e, cls))
    raise RuntimeError(f"LLM bağlantı testi başarısız: {hint}\nAsıl hata: {type(e).__name__}: {redact(str(e))[:300]}") from None
answer = resp.choices[0].message.content
print(f"Model: {resp.model} | süre: {time.time() - t0:.1f} sn | token: {resp.usage.prompt_tokens} girdi + {resp.usage.completion_tokens} çıktı")
print("Cevap:", answer)
try:
    print("JSON olarak ayrıştırıldı:", json.loads(re.sub(r"^```(?:json)?|```$", "", answer.strip()).strip()))
except json.JSONDecodeError as e:
    print("⚠️  Cevap doğrudan JSON değil:", e)

## 4. Hata Düzeltmeleri

Repo olduğu gibi çalışmıyor. Bu bölümde her hata için aynı sıra izlenir:

1. Hata, sebebi ve çözümü açıklanır.
2. Orijinal kod çalıştırılarak hata **gerçekten tetiklenir** ve mesajı çıktıda gösterilir.
3. Dosyada birebir metin değişikliği yapılır (`patch_file`, `assert` ile).
4. Düzeltmenin işe yaradığı doğrulanır.

Bir hatayı düzeltmek çoğu zaman bir sonrakini görünür kılar. Bu yüzden hatalar karşılaşılma sırasıyla numaralandırıldı (H4, H5, …). Değiştirilen her satır `[Colab yaması …]` ya da `[ölçüm eki]` yorumuyla işaretlendi; tam liste Bölüm 4.13'teki `git diff`'tedir. Builder ve retrieval kodu her seferinde ayrı bir Python sürecinde import edilir.

| Bölüm | Bileşen | Hata |
|---|---|---|
| 4.1 | graph-builder | H4: `main.py` repoda olmayan 8 prompt sabitini import ediyor |
| 4.2 | graph-builder | H5: `embedding.py` `cuda:2` cihazını sabit kodluyor |
| 4.3 | graph-builder | H6: `print(..., exc_info=True)` → TypeError; H7: asıl hata ortaya çıkıyor |
| 4.4 | graph-builder | (H7'nin çözümü) `call_openai` sabit Jetstream adresi/anahtarı |
| 4.5 | graph-builder | H8: sabit 6 worker; boş klasörde `max_workers=0` |
| 4.6 | retrieval | H9: `from src...` import'ları; H10 ortaya çıkıyor |
| 4.7 | retrieval | H10: LangChain 1.x'te `langchain.prompts/retrievers` yok; H11 ortaya çıkıyor |
| 4.8 | retrieval | H11: ragas modül seviyesinde import ediliyor; H12 ortaya çıkıyor |
| 4.9 | retrieval | H12: `FORMAT_ANSWER_PROMPT` tanımlı değil |
| 4.10 | retrieval | H13: sentezde tanımsız `client` / `deepseek_client`, sabit model adı |
| 4.11 | scripts | H14: `eval_ultradomain.py` import hatası; H15: `eval_multihopqa.py` boş yollar |
| 4.12 | — | Çökme olmayan davranış notları |

### 4.1 H4: graph-builder import edilemiyor (`ImportError`)

- **Belirti:** `import src.knowledge_graph_builder.main` → `ImportError: cannot import name 'MAIN_LUPUS_USER_PROMPT' ...`. Paketin `__init__.py` dosyası `main`'i de import ettiği için builder'ın **hiçbir** modülü kullanılamaz.
- **Sebep:** `main.py:35-46`, `prompts.py`'de tanımlı olmayan 8 sabiti import ediyor: `MAIN_MHOP_PROMPT`, `MAIN_SCLC_*` (2), `MAIN_UV_*` (2), `MAIN_PLASTICS_*` (2) ve `MAIN_LUPUS_USER_PROMPT`. Üçlü çıkarımı ise `main.py:66`'da `MAIN_MHOP_PROMPT`'u kullanıyor.
- **Çözüm:** Yalnızca var olan `MAIN_SYSTEM_PROMPT` ve `MAIN_USER_PROMPT` import edilir. `MAIN_MHOP_PROMPT` yerine genel amaçlı `MAIN_USER_PROMPT` kullanılır.
- **Sınırlama:** Yazarların MultiHop-RAG için kullandığı özel prompt (`MAIN_MHOP_PROMPT`) repoda yok, bu yüzden KG'yi genel prompt'la kuruyoruz. Bu, sonuçların makaleyle kıyaslanamamasının bir sebebidir (Bölüm 11). Ayrıca `MAIN_USER_PROMPT` kişiler için "konumları, meslekleri ve bilinen işleriyle ilişki **kurma**" der (`prompts.py:23`). eGOT'un dayandığı projedeki prompt bunun tersini, yani bu ilişkilerin kurulmasını ister [8].

In [ ]:
# 4.1 graph-builder: main.py var olmayan 8 prompt sabitini import ediyor
B_MAIN = "src/graph-builder/src/knowledge_graph_builder/main.py"
B_LLM  = "src/graph-builder/src/knowledge_graph_builder/llm.py"
B_EMB  = "src/graph-builder/src/knowledge_graph_builder/embedding.py"

# (a) Hatayı tetikle
code, out = run_py("import src.knowledge_graph_builder.main", cwd=BUILDER_DIR, tail=3)
if code != 0:
    record_error("H4", "graph-builder: import src.knowledge_graph_builder.main", out)

# (b) Hangi sabitler eksik?
prompts_src = (BUILDER_DIR / "src/knowledge_graph_builder/prompts.py").read_text()
wanted = ["MAIN_SYSTEM_PROMPT", "MAIN_USER_PROMPT", "MAIN_LUPUS_USER_PROMPT", "MAIN_UV_USER_PROMPT", "MAIN_UV_SYSTEM_PROMPT",
          "MAIN_PLASTICS_SYSTEM_PROMPT", "MAIN_PLASTICS_USER_PROMPT", "MAIN_SCLC_USER_PROMPT", "MAIN_SCLC_SYSTEM_PROMPT",
          "MAIN_MHOP_PROMPT"]
print("prompts.py'de TANIMLI OLMAYAN sabitler:", [n for n in wanted if not re.search(rf"^{n}\s*=", prompts_src, re.M)])

# (c) Yama: yalnızca var olan iki prompt'u import et; MAIN_MHOP_PROMPT yerine MAIN_USER_PROMPT
patch_file(B_MAIN, '''from src.knowledge_graph_builder.prompts import (
    MAIN_SYSTEM_PROMPT,
    MAIN_USER_PROMPT,
    MAIN_LUPUS_USER_PROMPT,
    MAIN_UV_USER_PROMPT,
    MAIN_UV_SYSTEM_PROMPT,
    MAIN_PLASTICS_SYSTEM_PROMPT,
    MAIN_PLASTICS_USER_PROMPT,
    MAIN_SCLC_USER_PROMPT,
    MAIN_SCLC_SYSTEM_PROMPT,
    MAIN_MHOP_PROMPT
)''', '''from src.knowledge_graph_builder.prompts import (  # [Colab yaması B1] yalnızca var olan prompt'lar
    MAIN_SYSTEM_PROMPT,
    MAIN_USER_PROMPT,
)''', "B1", "Var olmayan 8 prompt sabitinin import'u kaldırıldı")
patch_file(B_MAIN, "    user_prompt = MAIN_MHOP_PROMPT\n",
           "    user_prompt = MAIN_USER_PROMPT  # [Colab yaması B1] MAIN_MHOP_PROMPT repoda tanımlı değil\n",
           "B1", "Üçlü çıkarımında MAIN_MHOP_PROMPT yerine MAIN_USER_PROMPT")

# (d) Doğrula
code, out = run_py("import src.knowledge_graph_builder.main; print('graph-builder import edildi ✓')", cwd=BUILDER_DIR, tail=2)
assert code == 0

### 4.2 H5: `embedding.py` `cuda:2` cihazını sabit kodluyor

- **Belirti:** GPU varken model `cuda:2` cihazına yüklenmeye çalışılır. Tek GPU'lu bir makinede (ör. Colab T4) bu cihaz yoktur ve yükleme hata verir. `load_embedding_model` hatayı yalnızca yazdırıp `None` döndürür; `get_embedding` boş liste döndürür; `main.py:170-172` ise gömmesi olmayan chunk'ı yalnızca bir uyarı satırı yazarak **atlar** (LLM'e de gönderilmez). Sonuçta hiçbir belge KG'ye girmez ve program çökmeden biter.
- **Sebep:** `embedding.py:21`: `device = 'cuda:2' if torch.cuda.is_available() else 'cpu'`. Yazarların çok GPU'lu sunucusuna özgü bir ayar.
- **Gösterim:** Bu notebook'un varsayılan çalışma zamanında (tek T4 GPU) hata **gerçekten** oluşur, çünkü `cuda:2` diye bir cihaz yoktur. CPU çalışma zamanında ise `torch.cuda.is_available()` `False` döndüğü için bu dal hiç çalışmaz. O durumda hata, alt süreçte `torch.cuda.is_available` geçici olarak `True` döndürülerek taklit edilir (**taklit**). İki durumda hata mesajı farklıdır, ama sonuç (model `None`, chunk atlanır) aynıdır.
- **Çözüm:** `'cuda'` (varsa), yoksa `'cpu'`.

In [ ]:
# 4.2 graph-builder: embedding.py 'cuda:2' cihazını sabit kodluyor
DEMO_EMB = '''
import torch
import src.knowledge_graph_builder.embedding as emb
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("Görünen GPU sayısı:", n_gpu)
if n_gpu == 0 and SIMULATE:
    print("CPU çalışma zamanı: tek GPU'lu bir makine taklit ediliyor (torch.cuda.is_available -> True)")
    torch.cuda.is_available = lambda: True
model = emb.load_embedding_model("all-MiniLM-L6-v2")
vec = emb.get_embedding("Marie Curie discovered polonium.", model)
print("Dönen model:", type(model).__name__, "| gömme boyutu:", len(vec))
print("main.py bu durumda chunk'ı atlar:", "EVET (if not embedding: continue)" if not vec else "hayır")
'''
# (a) Hatayı göster (GPU'da gerçekten oluşur, CPU'da taklit edilir) — yalnızca orijinal satır hâlâ yerindeyse
if "device = 'cuda:2' if" in (REPO / B_EMB).read_text():
    code, out = run_py("SIMULATE = True\n" + DEMO_EMB, cwd=BUILDER_DIR, tail=8)
    if "Error loading model" in out:
        record_error("H5", "graph-builder: embedding.load_embedding_model (cuda:2)" + ("" if HAS_GPU else " — taklit"),
                     out, pattern=r"^Error loading model")
else:
    print("Yama daha önce uygulanmış; hata gösterimi atlandı (ilk çalıştırmada tetiklenir).")

# (b) Yama
patch_file(B_EMB, "    device = 'cuda:2' if torch.cuda.is_available() else 'cpu'",
           "    device = 'cuda' if torch.cuda.is_available() else 'cpu'  # [Colab yaması B3] 'cuda:2' yerine",
           "B3", "Sabit 'cuda:2' yerine 'cuda' (varsa) ya da 'cpu'")

# (c) Doğrula (gerçek ortamda, taklit yok)
code, out = run_py("SIMULATE = False\n" + DEMO_EMB, cwd=BUILDER_DIR, tail=4)
assert code == 0 and "gömme boyutu: 384" in out

### 4.3 H6: hata yakalama bloğu kendisi `TypeError` fırlatıyor; H7: asıl hata

- **Belirti (H6):** Bir dosyanın işlenmesi hata verince builder `TypeError: print() got an unexpected keyword argument 'exc_info'` ile çöker. (Eski Python sürümlerinde mesaj `'exc_info' is an invalid keyword argument for print()` şeklindedir.)
- **Sebep:** `main.py:352`: `print(f"...{e}", exc_info=True)`. `exc_info` parametresi `logging` fonksiyonlarına aittir, `print`'te yoktur. Hata `except` bloğunun içinde oluştuğu için asıl hatanın yerine bu `TypeError` görünür ve program sonlanır.
- **Gösterim:** Builder küçük bir demo dosyasıyla çalıştırılır. Jetstream sunucusuna gerçekten istek göndermemek için OpenAI istemcisi, kendisine verilen adresi raporlayıp hata fırlatan bir **kayıt istemcisiyle** değiştirildi. Böylece ağ isteği yapılmaz, ama orijinal kodun hangi adrese bağlanacağı görülür.
- **Çözüm:** `print(...)` + `traceback.print_exc()`.
- **H7 (düzeltmeden sonra görünen asıl hata):** Config'te `base_url` OpenAI'ı gösterdiği hâlde istemciye `https://llm.jetstream-cloud.org/llama-4-scout/v1` adresi ve `api_key='empty'` veriliyor. Bu, Bölüm 4.4'te düzeltilir.

In [ ]:
# 4.3 graph-builder: hata yakalama bloğundaki print(..., exc_info=True)
# Küçük bir demo girdisi ve config hazırla (Bölüm 6'daki gerçek config ile aynı biçim)
DEMO_IN, DEMO_OUT = WORK / "demo_in", WORK / "demo_out"
shutil.rmtree(DEMO_OUT, ignore_errors=True); DEMO_IN.mkdir(exist_ok=True)
(DEMO_IN / "demo.txt").write_text("Title: Demo article\nAuthor: Test Author\nCategory: science\n\n"
                                  "Fact: Marie Curie discovered polonium and radium in Paris.\n\nURL: https://example.org/demo\n")
DEMO_CFG = WORK / "demo_config.toml"
DEMO_CFG.write_text(f'''[llm]
model = "{LLM_MODEL}"
api_key = "env:OPENAI_API_KEY"   # gerçek anahtar dosyaya yazılmaz
base_url = "{LLM_BASE_URL.rstrip('/')}/chat/completions"
max_tokens = 1024
temperature = 0.2

[chunking]
chunk_size = 100
overlap = 20
''')

# Ağ isteği göndermeyen "kayıt istemcisi": OpenAI istemcisine hangi adres/anahtar verildiğini raporlar ve hata fırlatır.
# (Jetstream sunucusuna gerçekten istek göndermemek için; yollar ortam değişkeniyle verilir.)
DEMO_ENV = os.environ.copy()
DEMO_ENV.update({"DEMO_IN": str(DEMO_IN), "DEMO_OUT": str(DEMO_OUT), "DEMO_CFG": str(DEMO_CFG)})
DEMO_BUILDER = '''
import os
import src.knowledge_graph_builder.llm as L
import src.knowledge_graph_builder.main as M

class KayitIstemcisi:
    def __init__(self, base_url=None, api_key=None, **kw):
        key = api_key if api_key in (None, "empty") else "<OPENAI_API_KEY ortam değişkeni (gizli)>"
        raise ConnectionError(f"[demo] istek gönderilmedi; istemciye verilen base_url={base_url!r}, api_key={key!r}")

L.OpenAI = KayitIstemcisi
M.main_kg_builder(os.environ["DEMO_IN"], os.environ["DEMO_OUT"], config_path=os.environ["DEMO_CFG"],
                  experiment_name="demo", no_standardize=True, no_inference=True)
print("main_kg_builder tamamlandı (çökme yok)")
'''


def show_lines(out, pattern, last=0):
    """Çıktıdan yalnızca kalıba uyan satırları (ve isteğe bağlı son `last` satırı) gösterir."""
    lines = out.rstrip().splitlines()
    picked = [l for l in lines if re.search(pattern, l)]
    for l in picked + (["   ⋮"] + lines[-last:] if last else []):
        print(l[:300])


# (a) Hatayı tetikle: LLM çağrısı başarısız olunca except bloğu kendisi TypeError fırlatıyor
if "exc_info=True" in (REPO / B_MAIN).read_text():
    code, out = run_py(DEMO_BUILDER, cwd=BUILDER_DIR, env=DEMO_ENV, show=False)
    show_lines(out, r"^Error calling OpenAI API|^Traceback|During handling", last=3)
    if code != 0:
        record_error("H6", "graph-builder: main.py except bloğu (print exc_info)", out)
else:
    print("Yama daha önce uygulanmış; hata gösterimi atlandı (ilk çalıştırmada tetiklenir).")

# (b) Yama
patch_file(B_MAIN, '''                print(f"Error occurred during file processing: {e}", exc_info=True)''',
           '''                # [Colab yaması B4] print() exc_info almaz; asıl hatayı ve izini yazdır
                import traceback
                print(f"Error occurred during file processing: {e!r}")
                traceback.print_exc()''', "B4", "print(..., exc_info=True) → hata + traceback")

# (c) Aynı demo: artık program çökmüyor ve asıl hata görünüyor
code, out = run_py(DEMO_BUILDER, cwd=BUILDER_DIR, env=DEMO_ENV, show=False)
show_lines(out, r"^Error calling OpenAI API|^Error occurred during file processing|main_kg_builder tamamlandı")
assert code == 0
if "jetstream" in out:
    record_error("H7", "graph-builder: llm.call_openai (B4 düzeltilince görünen asıl hata; kayıt istemcisiyle)", out,
                 pattern=r"^Error calling OpenAI API")

### 4.4 `call_openai` config'i yok sayıyor (H7'nin çözümü) + yeniden deneme

- **Sebep:** `llm.py:114-116`: `OpenAI(base_url="https://llm.jetstream-cloud.org/llama-4-scout/v1", api_key="empty")`. Fonksiyonun aldığı `base_url` parametresi hiç kullanılmıyor. Builder'daki bütün LLM çağrıları bu fonksiyondan geçer: üçlü çıkarımı (`main.py`), LLM ile varlık çözümleme ve ilişki çıkarımı (`entity_standardization.py`). Jetstream, yazarların kullandığı bir sunucudur ve bizim erişimimiz yoktur.
- **Çözüm:**
  - `base_url` config'ten okunur. Repo'nun config biçimi uç noktanın tam adresini verdiği için sondaki `/chat/completions` kırpılır.
  - Anahtar `OPENAI_API_KEY` ortam değişkeninden okunur; config dosyasına yazılmaz.
  - OpenAI SDK'sının hız sınırı ve geçici hatalarda üstel geri çekilmeyle (backoff) yeniden deneme sayısı 2'den 6'ya çıkarıldı.
  - **Ölçüm eki:** Her çağrının token kullanımı bir dosyaya yazılır (`EGOT_USAGE_LOG`), böylece KG kurulumunun maliyeti ölçülebilir (Bölüm 9.3, 11).
- **Doğrulama:** (a) Kayıt istemcisiyle, istemciye artık bizim adresimizin ve ortamdaki anahtarın verildiği görülür (anahtar gösterilmez). (b) Demo metninden gerçek modelle üçlü çıkarılır (1 LLM çağrısı).

In [ ]:
# 4.4 graph-builder: call_openai sabit Jetstream adresi ve "empty" anahtarı kullanıyor
patch_file(B_LLM, '''import requests
import json
import re
from openai import OpenAI
''', '''import requests
import json
import re
import os
import threading
import time
from openai import OpenAI


# [Colab yaması B2] yardımcılar: base_url normalizasyonu ve token kullanım kaydı (ölçüm)
def _normalize_base_url(url):
    url = (url or os.environ.get("OPENAI_BASE_URL") or "https://api.openai.com/v1").rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


_USAGE_LOCK = threading.Lock()


def _log_usage(response):
    path = os.environ.get("EGOT_USAGE_LOG")
    usage = getattr(response, "usage", None)
    if not path or usage is None:
        return
    row = {"t": time.time(), "prompt_tokens": getattr(usage, "prompt_tokens", 0) or 0,
           "completion_tokens": getattr(usage, "completion_tokens", 0) or 0}
    with _USAGE_LOCK, open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\\n")
''', "B2", "Yardımcılar: base_url normalizasyonu, token kaydı")
patch_file(B_LLM, '''    client = OpenAI(
        base_url="https://llm.jetstream-cloud.org/llama-4-scout/v1", api_key="empty"
    )
''', '''    # [Colab yaması B2] sabit Jetstream adresi/anahtarı yerine config'teki base_url ve ortamdaki
    # OPENAI_API_KEY; SDK'nın üstel geri çekilmeli (backoff) yeniden deneme sayısı artırıldı
    client = OpenAI(
        base_url=_normalize_base_url(base_url),
        api_key=os.environ.get("OPENAI_API_KEY") or "empty",
        max_retries=int(os.environ.get("EGOT_LLM_MAX_RETRIES", "6")),
        timeout=float(os.environ.get("EGOT_LLM_TIMEOUT", "120")),
    )
''', "B2", "OpenAI istemcisi config base_url + ortam anahtarı + retry/backoff")
patch_file(B_LLM, '''        return response.choices[0].message.content
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")''', '''        _log_usage(response)  # [Colab yaması B2] token sayımı (ölçüm)
        return response.choices[0].message.content or ""
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")''', "B2", "Token kaydı; boş içerik None yerine ''")

# (a) Doğrula — ağ isteği olmadan: istemciye artık config'teki adres ve ortamdaki anahtar veriliyor
code, out = run_py(DEMO_BUILDER, cwd=BUILDER_DIR, env=DEMO_ENV, show=False)
print(exception_line(out, r"^Error calling OpenAI API"))
assert LLM_BASE_URL.rstrip("/") in out and "gizli" in out, "İstemciye beklenen base_url/anahtar verilmedi"

# (b) Doğrula — gerçek LLM ile: demo metninden üçlü çıkarımı (1 LLM çağrısı; geçersiz cevapta en fazla 3 deneme)
shutil.rmtree(DEMO_OUT, ignore_errors=True)
for attempt in range(1, 4):
    code, out = run_py('''
import json, glob, os
import src.knowledge_graph_builder.main as M
M.main_kg_builder(os.environ["DEMO_IN"], os.environ["DEMO_OUT"], config_path=os.environ["DEMO_CFG"],
                  experiment_name="demo", no_standardize=True, no_inference=True)
for f in glob.glob(os.path.join(os.environ["DEMO_OUT"], "demo_json_outputs", "*.json")):
    for ch in json.load(open(f)):
        print("chunk gömme boyutu:", len(ch.get("embedding", [])))
        for t in ch.get("triples", []):
            print("  üçlü:", t["subject"], "|", t["predicate"], "|", t["object"])
''', cwd=BUILDER_DIR, env=DEMO_ENV, show=False)
    if code == 0 and "üçlü:" in out:
        break
    print(f"Deneme {attempt}: modelden geçerli üçlü JSON'u gelmedi →", exception_line(out, r"ERROR|Error|failed")[:200])
show_lines(out, r"^chunk gömme boyutu|^  üçlü:")
assert code == 0 and "üçlü:" in out, "Builder gerçek LLM ile üçlü üretemedi (model adını/anahtarı kontrol edin)"

### 4.5 H8: sabit 6 paralel worker

- **Risk:** `main.py:342` her zaman en fazla 6 dosyayı aynı anda işler. Her dosya birden çok LLM çağrısı yaptığı için bu, API hız sınırlarına (rate limit) takılma riski taşır ve parametre olarak değiştirilemez.
- **Tetiklenen hata (H8):** Aynı satır, girdi klasöründe `.txt` yoksa `ThreadPoolExecutor(max_workers=0)` çağırır ve builder `ValueError: max_workers must be greater than 0` ile çöker.
- **Çözüm:** Worker sayısı `EGOT_BUILDER_WORKERS` ortam değişkeninden okunur. Değişken verilmezse repo değeri 6 kullanılır, sayı en az 1'dir. Bu notebook'ta `BUILDER_WORKERS = 2`.

In [ ]:
# 4.5 graph-builder: sabit 6 paralel worker (hız sınırı riski) ve boş klasörde 0 worker hatası
EMPTY_IN = WORK / "demo_empty"; EMPTY_IN.mkdir(exist_ok=True)
EMPTY_ENV = dict(DEMO_ENV, DEMO_IN=str(EMPTY_IN), DEMO_OUT=str(WORK / "demo_empty_out"))
DEMO_EMPTY = '''
import os
import src.knowledge_graph_builder.main as M
M.main_kg_builder(os.environ["DEMO_IN"], os.environ["DEMO_OUT"], config_path=os.environ["DEMO_CFG"], experiment_name="demo")
print("boş klasör sorunsuz işlendi")
'''
# (a) Hatayı tetikle: klasörde .txt yoksa max_workers = min(6, 0) = 0 olur
code, out = run_py(DEMO_EMPTY, cwd=BUILDER_DIR, env=EMPTY_ENV, tail=3)
if code != 0:
    record_error("H8", "graph-builder: main.py max_workers = min(6, len(input_files))", out)

# (b) Yama: worker sayısı EGOT_BUILDER_WORKERS ortam değişkeninden (varsayılan repo değeri 6), en az 1
patch_file(B_MAIN, "    max_workers = min(6, len(input_files))",
           '''    # [Colab yaması B5] paralel dosya sayısı parametre (EGOT_BUILDER_WORKERS); boş klasörde 0 worker hatası da önlenir
    max_workers = max(1, min(int(os.environ.get("EGOT_BUILDER_WORKERS", "6")), len(input_files)))''',
           "B5", "Worker sayısı parametre (EGOT_BUILDER_WORKERS), en az 1")

# (c) Doğrula
code, out = run_py(DEMO_EMPTY, cwd=BUILDER_DIR, env=EMPTY_ENV, tail=1)
assert code == 0

### 4.6 H9: retrieval README'deki gibi çalıştırılamıyor (`ModuleNotFoundError: No module named 'src'`)

- **Belirti:** README, sunucuyu `src/knowledge_graph_retrieval/` klasöründe `uvicorn app:app` ile başlatmayı söylüyor. Ancak `app.py:6`, `QA_integration.py:31,33` ve `got.py:11-16` modülleri `from src.QA_integration ...`, `from src.constants ...` ve `from src.metrics...` diye import ediyor. Bu klasörde `src` adında bir paket yoktur.
- **Sebep:** Kod, yazarların ortamında `<sunucu>/src/` altında duruyordu ve `app.py` bunun bir üst klasöründeydi. Repoya ise farklı bir düzenle konmuş.
- **Çözüm (kodda değişiklik yok):** Bir sunucu klasörü oluşturulur: `<server>/src → knowledge_graph_retrieval/` ve `<server>/app.py → knowledge_graph_retrieval/app.py` (sembolik bağlantılar). Uvicorn bu klasörde çalıştırılır. Sembolik bağlantı kullandığımız için sonraki yamalar doğrudan repo dosyalarına uygulanır ve sunucu yamalı kodu çalıştırır.
- Düzen kurulunca `src` bulunur, ama bu kez bir sonraki hata görünür (H10).

In [ ]:
# 4.6 retrieval: README'deki gibi çalıştırınca `src.` import'ları bulunamıyor
R_QA, R_GOT, R_EVAL = ("src/knowledge_graph_retrieval/QA_integration.py", "src/knowledge_graph_retrieval/got.py",
                       "src/knowledge_graph_retrieval/metrics/evaluation.py")

# (a) README: "cd src/knowledge_graph_retrieval && uvicorn app:app" → uvicorn önce `import app` yapar
code, out = run_py("import app", cwd=RETR_DIR, tail=2)
if code != 0:
    record_error("H9", "retrieval: import app (README'deki çalışma klasörü)", out)

# (b) Çözüm (kodda değişiklik yok): sunucu klasörü — <server>/src → knowledge_graph_retrieval, <server>/app.py
SERVER_DIR.mkdir(exist_ok=True)
for name, target in (("src", RETR_DIR), ("app.py", RETR_DIR / "app.py")):
    link = SERVER_DIR / name
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(target)
run(["ls", "-l", str(SERVER_DIR)], tail=4)

# (c) Tekrar dene: `src` artık bulunuyor, sıradaki hata ortaya çıkıyor (Bölüm 4.7)
code, out = run_py("import app", cwd=SERVER_DIR, tail=2)
if code != 0:
    record_error("H10", "retrieval: import app (sunucu düzeniyle)", out)

### 4.7 H10: LangChain 1.x'te `langchain.prompts` ve `langchain.retrievers` yok

- **Belirti:** `ModuleNotFoundError: No module named 'langchain.prompts'`.
- **Sebep:** Kod LangChain 0.x için yazılmış: `QA_integration.py:14,17,19`, `got.py:8` ve `metrics/evaluation.py:3`. LangChain 1.0 ile bu eski modüller ana `langchain` paketinden çıkarıldı. Prompt sınıfları `langchain_core.prompts`'ta, eski retriever'lar ve sıkıştırıcılar ise ayrı `langchain-classic` paketindedir [14]. Colab'da `langchain` 1.x kurulu gelir [16].
- **Çözüm:**
  - `langchain.prompts` → `langchain_core.prompts`
  - `langchain.retrievers` → `langchain_classic.retrievers`
  - `langchain.retrievers.document_compressors` → `langchain_classic.retrievers.document_compressors`
  
  `langchain-neo4j 0.11.0`, `langchain-openai 1.7.3` ve `langchain-huggingface 1.2.3` bu ortamda sorunsuz import ediliyor. (`langchain.docstore` yalnızca bu hattın kullanmadığı `scripts/general/token_analyzer.py`'de geçtiği için ona dokunulmadı.)

In [ ]:
# 4.7 retrieval: LangChain 1.x'te langchain.prompts / langchain.retrievers modülleri yok
import importlib.util
for mod in ("langchain.prompts", "langchain.retrievers", "langchain_core.prompts", "langchain_classic.retrievers"):
    try:
        ok = importlib.util.find_spec(mod) is not None
    except ModuleNotFoundError:
        ok = False
    print(f"{mod:32s} {'var' if ok else 'YOK'}")
print("langchain sürümü:", md.version("langchain"), "| langchain-classic:", md.version("langchain-classic"))

patch_file(R_QA, "from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder",
           "from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # [Colab yaması R3]",
           "R3", "langchain.prompts → langchain_core.prompts")
patch_file(R_QA, "from langchain.retrievers import ContextualCompressionRetriever",
           "from langchain_classic.retrievers import ContextualCompressionRetriever  # [Colab yaması R3]",
           "R3", "langchain.retrievers → langchain_classic.retrievers")
patch_file(R_QA, "from langchain.retrievers.document_compressors import EmbeddingsFilter, DocumentCompressorPipeline",
           "from langchain_classic.retrievers.document_compressors import EmbeddingsFilter, DocumentCompressorPipeline  # [Colab yaması R3]",
           "R3", "langchain.retrievers.document_compressors → langchain_classic")
patch_file(R_GOT, "from langchain.prompts import ChatPromptTemplate",
           "from langchain_core.prompts import ChatPromptTemplate  # [Colab yaması R3]", "R3", "langchain.prompts → langchain_core.prompts")
patch_file(R_EVAL, "from langchain.prompts import ChatPromptTemplate",
           "from langchain_core.prompts import ChatPromptTemplate  # [Colab yaması R3]", "R3", "langchain.prompts → langchain_core.prompts")

# Tekrar dene: sıradaki hata (Bölüm 4.8)
code, out = run_py("import app", cwd=SERVER_DIR, tail=2)
if code != 0:
    record_error("H11", "retrieval: metrics/evaluation.py ragas import'u", out)

### 4.8 H11: ragas modül seviyesinde import ediliyor

- **Belirti:** `ModuleNotFoundError: No module named 'ragas'`. `QA_integration.py:33` `RAGEvaluator`'ı import ettiği için sunucu hiç başlamaz, oysa değerlendirme varsayılan olarak kapalıdır (`evaluation=False`).
- **Sebep:** `metrics/evaluation.py:6-10` ragas'ı dosyanın en üstünde import ediyor.
- **Neden ragas'ı kurmuyoruz?** ragas kurulsa bile sorun çözülmüyor. Ayrı bir test ortamında `ragas 0.4.3` + `langchain-community 0.4.2` ile `from ragas import evaluate`, `ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'` verdi. Ayrıca ragas kurulumu Colab'daki `datasets`/`pyarrow` sürümlerini değiştirir.
- **Çözüm:** ragas import'u `try/except` içine alınır. Değerlendirme yalnızca `evaluation=True` istenirse anlaşılır bir hatayla durur; bu notebook'ta değerlendirme için repo'nun EM/F1 betiği kullanılır (Bölüm 9).
- Düzeltmeden sonra sunucu import edilir. GoT modunda yüklenen `got.py` ise bir sonraki hatayı verir (H12).

In [ ]:
# 4.8 retrieval: metrics/evaluation.py ragas'ı modül seviyesinde import ediyor
patch_file(R_EVAL, '''from ragas import evaluate
from ragas.metrics import ResponseRelevancy, LLMContextPrecisionWithoutReference, Faithfulness, AnswerAccuracy, ContextRelevance, ResponseGroundedness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset
''', '''try:  # [Colab yaması R4] ragas opsiyonel: yalnızca evaluation=True istendiğinde gerekir
    from ragas import evaluate
    from ragas.metrics import ResponseRelevancy, LLMContextPrecisionWithoutReference, Faithfulness, AnswerAccuracy, ContextRelevance, ResponseGroundedness
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas import EvaluationDataset
    RAGAS_IMPORT_ERROR = None
except Exception as _ragas_err:  # ImportError ya da ragas'ın kendi bağımlılık hataları
    RAGAS_IMPORT_ERROR = _ragas_err
''', "R4", "ragas import'u opsiyonel (try/except)")
patch_file(R_EVAL, '''    def evaluate_using_ragas(self, metric_details: Dict[str, Any], ground_truth_answer: str = None) -> Dict[str, Any]:
''', '''    def evaluate_using_ragas(self, metric_details: Dict[str, Any], ground_truth_answer: str = None) -> Dict[str, Any]:
        if RAGAS_IMPORT_ERROR is not None:  # [Colab yaması R4]
            raise RuntimeError(f"evaluation=True için ragas gerekli ama yüklenemedi: {RAGAS_IMPORT_ERROR!r}")
''', "R4", "evaluation=True iken anlaşılır hata")

# Doğrula: app artık import ediliyor (QA_integration import sırasında gömme modelini de yükler)
code, out = run_py("import app; print('app import edildi ✓')", cwd=SERVER_DIR, tail=1)
assert code == 0

# Sıradaki hata yalnızca GoT modunda görülür: QA_RAG, got.py'yi mode == graph_of_thought iken import eder
code, out = run_py("import src.got", cwd=SERVER_DIR, tail=2)
if code != 0:
    record_error("H12", "retrieval: got.py → from src.constants import FORMAT_ANSWER_PROMPT", out)

### 4.9 H12: `FORMAT_ANSWER_PROMPT` tanımlı değil

- **Belirti:** `ImportError: cannot import name 'FORMAT_ANSWER_PROMPT' from 'src.constants'`. `QA_RAG`, `got.py`'yi yalnızca `mode == "graph_of_thought"` iken import eder (`QA_integration.py:617`). Bu yüzden sunucu açılır, ama ilk GoT isteği hata verir.
- **Sebep:** `got.py:16` bu sabiti import ediyor, ama `constants.py`'de böyle bir tanım yok. Sabiti kullanan tek yer `format_answer()` metodu ve bu metot hiçbir yerden çağrılmıyor.
- **Çözüm:** Import isteğe bağlı yapıldı; sabit yoksa `None` olur. Eksik prompt'un içeriğini bilmediğimiz için yeni bir prompt uydurulmadı.

In [ ]:
# 4.9 retrieval: got.py, constants.py'de olmayan FORMAT_ANSWER_PROMPT'u import ediyor
consts_src = (RETR_DIR / "constants.py").read_text()
print("constants.py'de FORMAT_ANSWER_PROMPT tanımı var mı?", bool(re.search(r"^FORMAT_ANSWER_PROMPT\s*=", consts_src, re.M)))
print("got.py'de kullanıldığı yerler:", [i + 1 for i, l in enumerate((RETR_DIR / "got.py").read_text().splitlines())
                                         if "FORMAT_ANSWER_PROMPT" in l])

patch_file(R_GOT, "from src.constants import FORMAT_ANSWER_PROMPT\n",
           '''try:  # [Colab yaması R5] FORMAT_ANSWER_PROMPT constants.py'de yok; onu kullanan format_answer() hiç çağrılmıyor
    from src.constants import FORMAT_ANSWER_PROMPT
except ImportError:
    FORMAT_ANSWER_PROMPT = None
''', "R5", "Eksik FORMAT_ANSWER_PROMPT import'u opsiyonel")

code, out = run_py("import src.got; print('got.py import edildi ✓')", cwd=SERVER_DIR, tail=1)
assert code == 0

### 4.10 H13: final sentez tanımsız istemcileri kullanıyor

- **Belirti:** GoT döngüsü bütün thought'ları işledikten sonra, son adımda `NameError: name 'client' is not defined` oluşur. `graph_of_thought_qa` (`got.py:471-480`) hatayı yakalar ve kullanıcıya yalnızca `"An error occurred during the Graph of Thought retrieval process."` döndürür. Yani soru başına ~20 LLM çağrısı harcanır, ama cevap gelmez.
- **Sebep:** `synthesize_answer_deepseek` (`got.py:251`) ve `synthesize_answer_gpt` (`got.py:265-266`), dosyada hiç oluşturulmamış `deepseek_client` ve `client` nesnelerini kullanıyor; model adları da (`deepseek/deepseek-chat`, `gpt-4o-mini`) sabit yazılmış. `conduct_graph_of_thought` her zaman `synthesize_answer_gpt`'yi çağırıyor (`got.py:398`).
- **Gösterim:** Fonksiyon doğrudan çağrıldığında hata LLM çağrısından önce oluşur; bu yüzden gösterim ücretsizdir.
- **Çözüm:**
  - Sentez, sunucuya gönderilen modelle (`self.llm`, yani `ChatOpenAI`) yapılır; sistem mesajı ve prompt şablonları korunur.
  - `EGOT_SYNTHESIS_STYLE=mhop` iken `got.py`'de zaten tanımlı olan ama kullanılmayan kısa cevap prompt'u `MHOP_SYNTHESIS_PROMPT` kullanılır; MultiHop-RAG'in kısa altın cevaplarıyla EM/F1 hesaplamak için uygundur. `long` seçeneği repo'nun uzun sentez prompt'unu (`SYNTHESIS_PROMPT_ultradomain`) kullanır.
  - GoT derinliği `EGOT_MAX_DEPTH` ile ayarlanır. Ortam değişkeni verilmezse repo davranışı korunur (`max_depth=3`, uzun prompt).
- **Ölçüm eki (hata düzeltmesi değil):** `get_llm`'e bir LangChain callback eklendi; sunucudaki bütün LLM çağrılarını ve token'ları sayar. GoT cevabına `info.llm_usage` olarak eklenir. `ChatOpenAI` için yeniden deneme sayısı 6'ya çıkarıldı. `retrieve_documents`'ın yuttuğu hatanın izi (traceback) artık sunucu loguna yazılıyor.

In [ ]:
# 4.10 retrieval: final sentez tanımsız `client` / `deepseek_client` kullanıyor; model sabit "gpt-4o-mini"
# (a) Hatayı tetikle: sentez fonksiyonunu doğrudan çağır (LLM çağrısı yapılmaz, hata ondan önce oluşur)
if "deepseek_client.chat" in (REPO / R_GOT).read_text():
    code, out = run_py('''
from src.got import GraphOfThoughtsRetriever
GraphOfThoughtsRetriever.synthesize_answer_gpt(None, "Soru?", "Kanıt")
''', cwd=SERVER_DIR, tail=2)
    if code != 0:
        record_error("H13", "retrieval: got.py synthesize_answer_gpt (final sentez)", out)
else:
    print("Yama daha önce uygulanmış; hata gösterimi atlandı (ilk çalıştırmada tetiklenir).")
src_lines = (RETR_DIR / "got.py").read_text().splitlines()
print("\ngraph_of_thought_qa bu hatayı yakalayıp yalnızca şu mesajı döndürüyor:")
print("\n".join(l for l in src_lines if "An error occurred during the Graph of Thought" in l))

# (b) Ölçüm eki (hata düzeltmesi değil): sunucudaki tüm ChatOpenAI çağrılarını ve token'ları sayan callback,
#     ayrıca LLM çağrılarına üstel backoff'lu yeniden deneme
patch_file(R_QA, '''def get_llm(model):
    try:''', '''class _LLMUsageCounter(BaseCallbackHandler):
    """[ölçüm eki] Sunucudaki tüm ChatOpenAI çağrılarını ve token'ları sayar."""

    def __init__(self):
        self._lock = threading.Lock()
        self.calls = self.prompt_tokens = self.completion_tokens = 0

    def on_llm_end(self, response, **kwargs):
        pt = ct = 0
        try:
            usage = (response.llm_output or {}).get("token_usage") or {}
            pt, ct = usage.get("prompt_tokens") or 0, usage.get("completion_tokens") or 0
            if not (pt or ct):
                meta = getattr(response.generations[0][0].message, "usage_metadata", None) or {}
                pt, ct = meta.get("input_tokens", 0), meta.get("output_tokens", 0)
        except Exception:
            pass
        with self._lock:
            self.calls += 1
            self.prompt_tokens += pt
            self.completion_tokens += ct

    def snapshot(self):
        with self._lock:
            return {"calls": self.calls, "prompt_tokens": self.prompt_tokens,
                    "completion_tokens": self.completion_tokens}

    def diff(self, before):
        now = self.snapshot()
        return {k: now[k] - before.get(k, 0) for k in now}


LLM_USAGE = _LLMUsageCounter()


def get_llm(model):
    try:''', "M1", "Ölçüm: LLM çağrı/token sayacı (callback)")
patch_file(R_QA, '''        llm = ChatOpenAI(
                    base_url=OPENAI_BASE_URL,
                    api_key=OPENAI_API_KEY,
                    model=model,
                    temperature=0,
                    )''', '''        llm = ChatOpenAI(
                    base_url=OPENAI_BASE_URL,
                    api_key=OPENAI_API_KEY,
                    model=model,
                    temperature=0,
                    max_retries=int(os.getenv("EGOT_LLM_MAX_RETRIES", "6")),  # [Colab eki] üstel backoff'lu yeniden deneme
                    timeout=float(os.getenv("EGOT_LLM_TIMEOUT", "120")),
                    callbacks=[LLM_USAGE],  # [ölçüm eki]
                    )''', "M1", "ChatOpenAI: retry/backoff + sayaç")
patch_file(R_QA, '''        error_message = f"Error retrieving documents: {str(e)}"
        logging.error(error_message)''', '''        error_message = f"Error retrieving documents: {str(e)}"
        logging.error(error_message, exc_info=True)  # [Colab eki] yutulan hatanın izi sunucu log'una yazılır''',
           "N1", "Yutulan retrieval hatasının izi log'a")

# (c) Düzeltme: sentez sunucuya gönderilen modelle (self.llm) yapılır; derinlik ve sentez biçimi parametre
patch_file(R_GOT, '''from src.metrics.evaluation import RAGEvaluator
''', '''from src.metrics.evaluation import RAGEvaluator
import os
from src.QA_integration import LLM_USAGE  # [ölçüm eki] LLM çağrı/token sayacı

# [Colab yaması R6] GoT derinliği ve sentez biçimi ortam değişkeniyle seçilir
# (değişken verilmezse repo davranışı korunur: max_depth=3 ve uzun sentez prompt'u)
EGOT_MAX_DEPTH = int(os.getenv("EGOT_MAX_DEPTH", "3"))
EGOT_SYNTHESIS_STYLE = os.getenv("EGOT_SYNTHESIS_STYLE", "long").lower()
''', "R6", "MAX_DEPTH / SYNTHESIS_STYLE parametreleri, sayaç import'u")
patch_file(R_GOT, '''    def synthesize_answer_deepseek(self, question: str, evidence_summary: str) -> str:
        prompt = SYNTHESIS_PROMPT_ultradomain.format(
            original_question=question, evidence_summary=evidence_summary)

        response = deepseek_client.chat.completions.create(
            model="deepseek/deepseek-chat",
            messages=[
                {"role": "system", "content": "You are a Question Answering assistant that uses the context provided and answers the question asked.  "},
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content.strip()

    def synthesize_answer_gpt(self, question: str, evidence_summary: str) -> str:
        prompt = SYNTHESIS_PROMPT_ultradomain.format(
            original_question=question, evidence_summary=evidence_summary)

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a Question Answering assistant that uses the context provided and answers the question asked.  "},
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content.strip()
''', '''    # [Colab yaması R6] Orijinal iki fonksiyon tanımsız `deepseek_client` / `client` nesnelerini ve sabit
    # "deepseek/deepseek-chat" / "gpt-4o-mini" modellerini kullanıyordu (NameError). Sentez artık sunucuya
    # gönderilen modelle (self.llm) yapılır; sistem mesajı ve prompt şablonları aynen korunur.
    def _synthesize_with_llm(self, template: str, question: str, evidence_summary: str) -> str:
        from langchain_core.messages import SystemMessage, HumanMessage
        prompt = template.format(original_question=question, evidence_summary=evidence_summary)
        response = self.llm.invoke([
            SystemMessage(content="You are a Question Answering assistant that uses the context provided and answers the question asked.  "),
            HumanMessage(content=prompt),
        ])
        return response.content.strip()

    def synthesize_answer_deepseek(self, question: str, evidence_summary: str) -> str:
        return self._synthesize_with_llm(SYNTHESIS_PROMPT_ultradomain, question, evidence_summary)

    def synthesize_answer_gpt(self, question: str, evidence_summary: str) -> str:
        # EGOT_SYNTHESIS_STYLE=mhop → MultiHop-RAG için kısa cevap (MHOP_SYNTHESIS_PROMPT, got.py'de tanımlı)
        # EGOT_SYNTHESIS_STYLE=long → repo'nun orijinal uzun sentez prompt'u (SYNTHESIS_PROMPT_ultradomain)
        template = MHOP_SYNTHESIS_PROMPT if EGOT_SYNTHESIS_STYLE == "mhop" else SYNTHESIS_PROMPT_ultradomain
        return self._synthesize_with_llm(template, question, evidence_summary)
''', "R6", "Sentez self.llm ile; MHOP/uzun prompt seçimi")
patch_file(R_GOT, '''    start_time = time.time()

    try:
        llm, model_name = get_llm(model=model)
        retriever = GraphOfThoughtsRetriever(''', '''    start_time = time.time()
    usage_before = LLM_USAGE.snapshot()  # [ölçüm eki]

    try:
        llm, model_name = get_llm(model=model)
        retriever = GraphOfThoughtsRetriever(''', "M1", "Ölçüm: istek başına sayaç başlangıcı")
patch_file(R_GOT, '''        final_answer, all_retrieved_info = retriever.conduct_graph_of_thought(
            question)''', '''        final_answer, all_retrieved_info = retriever.conduct_graph_of_thought(
            question, max_depth=EGOT_MAX_DEPTH)  # [Colab yaması R6] derinlik parametresi''',
           "R6", "conduct_graph_of_thought(max_depth=EGOT_MAX_DEPTH)")
patch_file(R_GOT, '''                "response_time": total_time,
                "thought_process": all_retrieved_info,''', '''                "response_time": total_time,
                "llm_usage": LLM_USAGE.diff(usage_before),  # [ölçüm eki]
                "max_depth": EGOT_MAX_DEPTH,
                "synthesis_style": EGOT_SYNTHESIS_STYLE,
                "thought_process": all_retrieved_info,''', "M1", "Ölçüm: cevaba llm_usage eklendi")

# (d) Doğrula: sentezi gerçek modelle bir kez çalıştır (1 LLM çağrısı)
env = os.environ.copy(); env.update({"EGOT_SYNTHESIS_STYLE": SYNTHESIS_STYLE, "EGOT_MAX_DEPTH": str(MAX_DEPTH)})
code, out = run_py(f'''
from src.QA_integration import get_llm, LLM_USAGE
from src.got import GraphOfThoughtsRetriever, EGOT_MAX_DEPTH, EGOT_SYNTHESIS_STYLE
g = GraphOfThoughtsRetriever.__new__(GraphOfThoughtsRetriever)   # Neo4j'ye bağlanmadan yalnızca sentez
g.llm, _ = get_llm({LLM_MODEL!r})
print("EGOT_MAX_DEPTH =", EGOT_MAX_DEPTH, "| EGOT_SYNTHESIS_STYLE =", EGOT_SYNTHESIS_STYLE)
print("Sentez cevabı:", repr(g.synthesize_answer_gpt("Which element did Marie Curie discover in 1898 besides radium?",
      "Thought: discoveries\\nAnswer: In 1898 Marie and Pierre Curie discovered polonium and radium.")))
print("Sayaç:", LLM_USAGE.snapshot())
''', cwd=SERVER_DIR, env=env, tail=3)
assert code == 0

### 4.11 H14 ve H15: değerlendirme betikleri

- **H14:** `scripts/eval_ultradomain.py:10` `from constants import ...` diyor, ama dosyanın adı `constant.py`. Sonuç: `ModuleNotFoundError: No module named 'constants'`. Ayrıca adres `OPENAI_URL` ortam değişkeninden okunuyor (satır 16); diğer bileşenler ise `OPENAI_BASE_URL` kullanıyor. **Çözüm:** `constant` olarak düzeltildi; `OPENAI_URL` yoksa `OPENAI_BASE_URL` kullanılıyor. Doğrulama `--help` ile yapılır. Betiğin kendisi bu çalışmada kullanılmıyor, çünkü UltraDomain verisini çalıştırmıyoruz (Bölüm 5).
- **H15:** `scripts/eval_multihopqa.py:57-62` iki CSV yolunu boş string olarak bırakmış: `pd.read_csv("")` → `FileNotFoundError`. **Çözüm:** Betik olarak çalıştırmak yerine EM/F1 fonksiyonlarını (`normalize_answer`, `compute_em`, `compute_f1`) import edip Bölüm 9'da kullanıyoruz. Bu modül `src` paketini kullanmadığı için çekirdeğe import edilebilir.

In [ ]:
# 4.11 scripts/: değerlendirme betikleri
S_ULTRA = "scripts/eval_ultradomain.py"

# (a) eval_ultradomain.py: `from constants import ...` ama dosyanın adı constant.py
code, out = run([sys.executable, "scripts/eval_ultradomain.py", "--help"], cwd=REPO, tail=2)
if code != 0:
    record_error("H14", "scripts/eval_ultradomain.py (import)", out)
print("scripts/ içeriği:", sorted(p.name for p in (REPO / "scripts").glob("*.py")))
patch_file(S_ULTRA, "from constants import SYSTEM_PROMPT, USER_PROMPT_TEMPLATE, DATASET, MAX_QUERIES",
           "from constant import SYSTEM_PROMPT, USER_PROMPT_TEMPLATE, DATASET, MAX_QUERIES  # [Colab yaması S1] dosya adı constant.py",
           "S1", "constants → constant")
patch_file(S_ULTRA, 'BASE_URL = os.getenv("OPENAI_URL")',
           'BASE_URL = os.getenv("OPENAI_URL") or os.getenv("OPENAI_BASE_URL")  # [Colab yaması S1] diğer bileşenlerle aynı değişken',
           "S1", "OPENAI_URL yoksa OPENAI_BASE_URL")
code, out = run([sys.executable, "scripts/eval_ultradomain.py", "--help"], cwd=REPO, tail=4)
assert code == 0

# (b) eval_multihopqa.py: CSV yolları boş string — betik olarak çalıştırılamıyor; fonksiyonlarını import edeceğiz
code, out = run([sys.executable, "scripts/eval_multihopqa.py"], cwd=REPO, tail=2)
if code != 0:
    record_error("H15", "scripts/eval_multihopqa.py (pd.read_csv(''))", out)
if str(REPO / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO / "scripts"))   # bağımsız modül: `src` paketini kullanmaz, çekirdeğe import edilebilir
from eval_multihopqa import normalize_answer, compute_em, compute_f1
print("compute_em('The Eiffel Tower.', 'eiffel tower') =", compute_em("The Eiffel Tower.", "eiffel tower"),
      "| compute_f1('Paris, France', 'Paris') =", round(compute_f1("Paris, France", "Paris"), 3))

### 4.12 Çökme olmayan davranış notları

Aşağıdakiler program hatası vermez, ama sonuçları etkiler. Kodu değiştirmedik; sonuçları yorumlarken bunlar dikkate alındı.

1. **Retrieval hataları yutuluyor.** `retrieve_documents` (`QA_integration.py:272-291`) her hatayı yakalayıp `docs=None` döndürür. GoT ise bu durumda thought'a `"No relevant information found for this thought."` yazıp devam eder (`got.py:237-245`). Neo4j şeması, indeksi ya da APOC eksikse sistem hata vermez, yalnızca kanıtsız cevap üretir. Bu yüzden: (a) hatanın izi artık sunucu loguna yazılıyor (Bölüm 4.10), (b) retrieval ayrı bir sağlık testiyle sınanıyor (Bölüm 8.2), (c) boş dönen retrieval adımları soru başına sayılıyor (Bölüm 9).
2. **Doküman sınırı.** `format_documents` (`QA_integration.py:184-192`), `CHAT_TOKEN_CUT_OFF` sözlüğünde model adını arar. Anahtarlar (`openai_gpt_4o_mini` gibi) gerçek API model adlarıyla eşleşmediği için varsayılan 4 kullanılır: her thought'ta en fazla 4 doküman LLM'e gider.
3. **`top_k` ayarı uygulanmıyor.** GoT modu `top_k: 5` ister (`constants.py:801`), ama `create_retriever` bunu LangChain'e `search_kwargs={'top_k': ...}` olarak verir (`QA_integration.py:382`). `VectorStore.similarity_search_with_relevance_scores` ise `k` parametresini kullanır (varsayılan 4) ve `top_k`'yı dikkate almaz. Yani fiilen sorgu başına en fazla 4 chunk gelir. Bu kalıp LLM Graph Builder'dan miras kalmıştır [7].
4. **Sohbet geçmişi.** GoT dışı modlarda cevap sohbet geçmişine yazılır ve arka planda özetlenir (`QA_integration.py:431-433, 487-521`). Aynı `session_id` ile gelen bir sonraki sorunun retrieval'ı bu geçmişten üretilen bir arama sorgusuyla yapılır (`create_document_retriever_chain`, satır 320-326). Sorular birbirini etkilemesin diye **her soruya ayrı `session_id`** veriyoruz. `write_access=false` ile geçmiş Neo4j yerine sunucu belleğinde tutulur, böylece grafa `Session`/`Message` düğümü eklenmez.
5. **Aynı paket adı.** graph-builder (`src/graph-builder/src/`) ve retrieval (sunucu düzeninde `src/`) aynı üst seviye `src` paket adını kullanır. İkisi aynı Python sürecinde import edilirse hangisinin yükleneceği `sys.path` sırasına bağlıdır. Bu yüzden builder, CSV ve görselleştirme adımları alt süreçte, retrieval ise uvicorn sunucusunda çalıştırılır.
6. **Sunucu logundaki Cypher uyarıları.** Neo4j sorgu sırasında üç tür uyarı verir: `CALL { WITH ... }` sözdiziminin eskidiği (deprecation), `__Community__` etiketinin olmadığı (LLM Graph Builder'dan kalan topluluk özelliği; eGOT topluluk düğümü üretmez) ve `Document.url` özelliğinin olmadığı. Hiçbiri sonucu değiştirmez; notebook çıktısında gizlenir.
7. **`remove_entities_and_relationships`** (`got.py:299-333`), adına rağmen yalnızca `Entities:`/`Relationships:` başlıklarını siler. Silmek için aradığı `Entity:` önekli satırlar sorgu çıktısında olmadığından varlık ve ilişki satırları thought üretimi ve sentez bağlamında kalır.
8. **Yan etki:** `conduct_graph_of_thought` her thought'u sunucunun çalışma klasöründeki `output.txt` dosyasına ekler (`got.py:358-359`).

In [ ]:
# 4.12 Davranış notları için küçük kontroller (kod değişikliği yok)
code, out = run_py(f'''
from src.constants import CHAT_TOKEN_CUT_OFF, CHAT_MODE_CONFIG_MAP
model = {LLM_MODEL!r}
cutoff = 4                                  # QA_integration.format_documents'taki varsayılan
for names, value in CHAT_TOKEN_CUT_OFF.items():
    if model in names:
        cutoff = value
        break
print("CHAT_TOKEN_CUT_OFF anahtarları:", [k if isinstance(k, str) else list(k) for k in CHAT_TOKEN_CUT_OFF])
print(f"model={{model!r}} için thought başına kullanılan en fazla doküman: {{cutoff}}")
print("graph_of_thought modu ayarı top_k =", CHAT_MODE_CONFIG_MAP["graph_of_thought"]["top_k"])
''', cwd=SERVER_DIR, tail=4)

# LangChain'in retriever'ı search_kwargs['top_k']'yi değil k parametresini kullanır (varsayılan 4):
import inspect
from langchain_core.vectorstores import VectorStore
print("VectorStore.similarity_search_with_relevance_scores imzası:",
      inspect.signature(VectorStore.similarity_search_with_relevance_scores))

### 4.13 Yamaların özeti

Uygulanan yamalar ve repo'ya göre tam fark (`git diff`). Bölüm 7.1'deki CSV üreticisi yaması (G1) daha sonra uygulandığı için bu listede yer almaz; Bölüm 10.2'de tüm yamalar birlikte gösterilir.

In [ ]:
# 4.13 Yamaların özeti ve tam diff
display(pd.DataFrame(PATCHES))
run(["git", "-C", str(REPO), "diff", "--stat"], tail=20)
_, diff = run(["git", "-C", str(REPO), "diff"], show=False)
(WORK / "egot_colab_yamalari.diff").write_text(diff)
display(HTML("<details><summary><b>Tam diff'i göster/gizle</b> (" + str(len(diff.splitlines())) + " satır)</summary>"
             "<pre style='font-size:11px'>" + html.escape(diff) + "</pre></details>"))

## 5. Veri Hazırlama: MultiHop-RAG Alt Kümesi

**MultiHop-RAG** [5], Eylül–Aralık 2023 arasında yayımlanmış İngilizce haber makalelerinden oluşan bir bilgi tabanı ve 2.556 çok adımlı sorudan oluşur. Her sorunun kanıtı 2–4 belgeye dağılmıştır. Soru türleri: *inference*, *comparison*, *temporal* ve *null*.

Repo bu veri setinden 1000 satırı `datasets/mhoprag_contexts.csv` olarak içerir. Dosyanın satır sayısı, sütunları ve *null* türü içermemesi, `multihoprag_generator.py`'nin yaptığıyla uyumludur: Bu betik, Hugging Face'teki veri setinden [17] kanıt listesi boş olmayan sorular arasından `seed=42` ile 1000 satır seçer. Her satırda soru (`query`), altın cevap (`answer`), soru türü ve kanıt listesi (`evidence_list`: title, author, category, fact, published_at, source, url) bulunur.

Veri seti repo ile birlikte geldiği için (`datasets/`) ayrıca indirilmez.

**Neden bu örnek?** Repodaki diğer örnekler bu ödev için uygun değil. `hotpotqa_corpus.csv` aslında JSON içeriyor ve sorularla kanıtlar arasında eşleme yok. UltraDomain (agriculture/legal/mix) bağlamları ise çok uzun; KG kurmak saatler ve yüksek API maliyeti gerektirir. MultiHop-RAG'de her sorunun kendi kanıtları satırda verildiği için küçük ama tutarlı bir alt küme kurulabiliyor.

### 5.1 Veriyi okuma

İki veri tuzağı var:
- **H16 — `evidence_list` biçimi:** Sütun, bir numpy dizisinin metin gösterimi (repr) olarak kaydedilmiş ve sözlükler arasında virgül yok. Bu yüzden ne `json.loads` ne de `ast.literal_eval` doğrudan çalışıyor (hücre iki hatayı da gösterir). `}` ile `{` arasına virgül ekleyip (`re.sub(r"\}\s*\{", "}, {", s)`) `ast.literal_eval` uygulamak 1000 satırın hepsini ayrıştırıyor.
- **`mhoprag_questions.csv` ile fark:** İki dosya aynı sırada, ama birebir aynı değil. 5 satırda cevap `True` / `TRUE` olarak farklı yazılmış, bir soruda da satır sonları farklı (`\n` / `\r\n`). Soru, cevap ve kanıt tutarlı olsun diye **`mhoprag_contexts.csv`'nin kendi `query`/`answer` sütunlarını** kullanıyoruz.

In [ ]:
# 5.1 MultiHop-RAG verisini yükle
import ast

CTX = pd.read_csv(REPO / "datasets" / "mhoprag_contexts.csv")
QS = pd.read_csv(REPO / "datasets" / "mhoprag_questions.csv")
print("mhoprag_contexts.csv :", CTX.shape, list(CTX.columns))
print("mhoprag_questions.csv:", QS.shape, list(QS.columns))
display(CTX["question_type"].value_counts().rename("soru sayısı").to_frame())

# evidence_list sütunu: dict'ler arasında virgül olmayan bir numpy dizisi repr'i
sample = CTX.loc[0, "evidence_list"]
print(sample[:260].replace("\n", "⏎\n"), "...")
for name, fn in (("json.loads", json.loads), ("ast.literal_eval", ast.literal_eval)):
    try:
        fn(sample)
        print(f"{name}: başarılı")
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e)[:120]}"
        print(f"{name}: {msg}")
        ERRORS.append({"id": "H16", "nerede": f"datasets/mhoprag_contexts.csv evidence_list → {name}", "yakalanan mesaj": msg})


def parse_evidence(s):
    """'}' ile '{' arasına virgül ekleyip Python literal'i olarak ayrıştır."""
    return ast.literal_eval(re.sub(r"\}\s*\{", "}, {", s))


CTX["evidence"] = CTX["evidence_list"].map(parse_evidence)
ok = CTX["evidence"].map(lambda v: isinstance(v, list) and all(isinstance(d, dict) for d in v)).sum()
print(f"\nDüzeltmeyle ayrıştırılan satır: {ok}/{len(CTX)}")
print("Soru başına kanıt sayısı:", CTX["evidence"].map(len).value_counts().sort_index().to_dict())
print("Bir kanıtın alanları:", list(CTX.loc[0, "evidence"][0].keys()))

# contexts.csv ile questions.csv aynı sırada mı? Neden contexts.csv'nin kendi sütunlarını kullanıyoruz?
dq = CTX.index[CTX["query"] != QS["question"]].tolist()
da = CTX.index[CTX["answer"].astype(str) != QS["answer"].astype(str)].tolist()
print(f"\nquery ≠ question olan satırlar: {dq} | answer farklı olan satırlar: {da}")
display(pd.DataFrame({"contexts.answer": CTX.loc[da, "answer"], "questions.answer": QS.loc[da, "answer"]}))
for i in dq:
    print(f"Satır {i}: satır sonu farkı → contexts: {CTX.loc[i, 'query'].count(chr(13))} adet \\r, "
          f"questions: {QS.loc[i, 'question'].count(chr(13))} adet \\r")

### 5.2 Soru seçimi

`N_QUESTIONS` soru, `question_type`'lara mümkün olduğunca eşit dağıtılarak ve sabit tohumla (`SEED`) seçilir. 8 soru için dağılım 3 *comparison*, 3 *inference* ve 2 *temporal* şeklindedir. `qid`, sorunun `mhoprag_contexts.csv`'deki satır numarasıdır.

In [ ]:
# 5.2 N_QUESTIONS soruyu question_type'lara dengeli ve sabit tohumla seç
types = CTX["question_type"].value_counts().index.tolist()          # en sık türden en aza
per_type = {t: N_QUESTIONS // len(types) for t in types}
for t in types[: N_QUESTIONS % len(types)]:
    per_type[t] += 1
SEL = pd.concat([CTX[CTX["question_type"] == t].sample(n=per_type[t], random_state=SEED)
                 for t in types if per_type[t] > 0])
SEL = SEL.rename_axis("row").reset_index()                          # row: mhoprag_contexts.csv'deki satır no
SEL["qid"] = SEL["row"].map(lambda r: f"mhop_{r:04d}")
print("Tür başına soru:", per_type)
display(SEL.assign(kanıt=SEL["evidence"].map(len))[["qid", "question_type", "query", "answer", "kanıt"]]
        .style.set_properties(subset=["query"], **{"white-space": "pre-wrap", "text-align": "left"}))

### 5.3 Kanıt belgeleri

KG **yalnızca seçilen soruların kanıtlarıyla** kurulur. Her kanıt, repodaki `multihoprag_generator.py` ile aynı biçimde (Title / Author / Category / Fact / URL) bir `.txt` dosyasına yazılır. İki farkımız var:

- Repo dosyaya yalnızca makale başlığını ad olarak verir. Aynı makaleden iki farklı kanıt gelirse ikincisi birincinin üzerine yazılır. Bunu önlemek için dosya adına içeriğin kısa bir özetini (hash) ekliyoruz. Hücre, repo'nun adlandırmasıyla kaç kanıtın kaybolacağını da yazdırır.
- Repo biçimi kanıtın kaynağını (`source`) ve tarihini (`published_at`) dosyaya **yazmaz**. Oysa *temporal* ve *comparison* soruları çoğunlukla "X'in şu tarihli haberi…" gibi bu bilgilere dayanır. Repo'ya sadık kalmak için varsayılan olarak bu alanları eklemiyoruz. `EVIDENCE_EXTRA_FIELDS = True` ile eklenebilirler; bu durumda KG ayrı bir klasörde yeniden kurulur. Bu, sonuçları yorumlarken dikkate alınması gereken bir sınırlamadır (Bölüm 11).

In [ ]:
# 5.3 Kanıtları repodaki biçimde .txt dosyalarına yaz (scripts/multihoprag_generator.py ile aynı alanlar)
def sanitize_filename(title):          # multihoprag_generator.py'deki fonksiyonun aynısı
    if not title:
        return "Untitled"
    sanitized = re.sub(r'[<>:"/\\|?*\n]', '_', title)
    return sanitized[:150]


def evidence_text(ev):
    text = (f"Title: {ev.get('title', 'No Title Provided')}\n"
            f"Author: {ev.get('author', 'N/A')}\n"
            f"Category: {ev.get('category', 'N/A')}\n\n")
    if EVIDENCE_EXTRA_FIELDS:          # repo biçiminde olmayan, isteğe bağlı alanlar
        text += f"Source: {ev.get('source', 'N/A')}\nPublished at: {ev.get('published_at', 'N/A')}\n\n"
    return text + f"Fact: {ev.get('fact', 'No Fact Provided')}\n\nURL: {ev.get('url', 'No URL Provided')}\n"


SUBSET_TAG = f"mhop_n{N_QUESTIONS}_s{SEED}" + ("_extra" if EVIDENCE_EXTRA_FIELDS else "")
DATA_DIR = WORK / "data" / SUBSET_TAG
TXT_DIR = DATA_DIR / "txt"; TXT_DIR.mkdir(parents=True, exist_ok=True)

files, q_docs, by_title = {}, {}, {}
for _, q in SEL.iterrows():
    for ev in q["evidence"]:
        content = evidence_text(ev)
        title_name = sanitize_filename(ev.get("title"))
        # Repo dosyayı yalnızca başlıkla adlandırır: aynı makaleden iki farklı kanıt birbirinin üzerine yazılır.
        # Çakışmayı önlemek için ada içeriğin kısa özetini (hash) ekliyoruz; aynı kanıt tek dosya olur.
        name = f"{title_name[:100]}__{hashlib.md5(content.encode()).hexdigest()[:8]}"
        files[name] = content
        by_title.setdefault(title_name, set()).add(name)
        q_docs.setdefault(q["qid"], []).append(name)
for name, content in files.items():
    (TXT_DIR / f"{name}.txt").write_text(content, encoding="utf-8")
SEL["docs"] = SEL["qid"].map(lambda k: sorted(set(q_docs[k])))

n_ev = int(SEL["evidence"].map(len).sum())
lost = sum(len(v) - 1 for v in by_title.values())
print(f"{len(SEL)} soru → {n_ev} kanıt → {len(files)} benzersiz .txt dosyası ({TXT_DIR})")
print(f"Repo'nun yalnızca başlıkla adlandırmasıyla üzerine yazılacak kanıt sayısı: {lost}")
first = sorted(TXT_DIR.glob("*.txt"))[0]
print(f"\nÖrnek dosya: {first.name}\n" + "-" * 80 + "\n" + first.read_text())

## 6. Bilgi Grafının (KG) Oluşturulması

### 6.1 Yapılandırma

Builder, repodaki `config.toml` biçiminde bir dosya okur. Chunk boyutu (100 kelime, 20 örtüşme), standardizasyon ve çıkarım ayarları repo değerleriyle aynıdır. Farklar:
- `model` ve `base_url` seçilen LLM'i gösterir.
- `temperature` 0.2'dir. Repo config'i 0.8, graph-builder README örneği 0.2 kullanır; daha kararlı JSON için düşük değeri seçtik.
- `max_tokens` 4096'dır (repo: 8192); 100 kelimelik bir chunk'ın üçlüleri için fazlasıyla yeterlidir.

API anahtarı dosyaya **yazılmaz**: yamalı `llm.py` onu ortam değişkeninden okur.

In [ ]:
# 6.1 Builder yapılandırması (repo'daki config.toml biçiminde; API anahtarı dosyaya YAZILMAZ)
BUILDER_CFG = WORK / "builder_config.toml"
BUILDER_CFG.write_text(f'''[llm]
model = "{LLM_MODEL}"
api_key = "env:OPENAI_API_KEY"   # yamalı llm.py anahtarı ortam değişkeninden okur
base_url = "{LLM_BASE_URL.rstrip('/')}/chat/completions"   # yamalı llm.py sondaki /chat/completions'ı kırpar
max_tokens = {BUILDER_MAX_TOKENS}
temperature = {BUILDER_TEMPERATURE}

[chunking]          # repo değerleri
chunk_size = 100
overlap = 20

[standardization]   # repo değerleri
enabled = true
use_llm_for_entities = true

[inference]         # repo değerleri
enabled = true
use_llm_for_inference = true
apply_transitive = true

[visualization]
edge_smooth = "dynamic"
''')
print(BUILDER_CFG.read_text())

### 6.2 Builder'ı çalıştırma

`src/knowledge_graph_builder/main.py`, kanıt klasöründeki her `.txt` için bir hybrid JSON üretir. Çağrılar paralel işlenir (`BUILDER_WORKERS`). JSON'u zaten olan dosyaları builder kendisi atlar, bu yüzden hücre yeniden çalıştırıldığında LLM tekrar çağrılmaz. Builder üçlü çıkarılamayan bir belge için JSON yazmadığından, LLM'in tek seferlik bozuk bir cevabı belgeyi KG'nin dışında bırakabilir. Örneğin model JSON dizisinde tek bir virgülü atlarsa repo'nun `extract_json_from_text` onarımı başarısız olur ve o chunk'ın bütün üçlüleri atılır; belge tek chunk'lıysa KG'ye hiç girmez. (Bu, testlerimizde açık bir modelle bir kez yaşandı.) Bu yüzden JSON'u eksik kalan belgeler için builder bir kez daha çalıştırılır. Builder çok ayrıntılı yazdığı için (ör. her dosyanın bütün üçlüleri) çıktı filtrelenerek gösterilir; tam log dosyaya yazılır.

In [ ]:
# 6.2 graph-builder'ı çalıştır (ayrı süreç; çıktı filtrelenerek gösterilir, tam log dosyaya yazılır)
KG_OUT = DATA_DIR / "kg"
JSON_DIR = KG_OUT / "mhop_json_outputs"             # main.py: <output>/<experiment_name>_json_outputs
BUILD_LOG = LOG_DIR / f"builder_{SUBSET_TAG}.log"
USAGE_LOG = LOG_DIR / f"builder_usage_{SUBSET_TAG}.jsonl"
KG_META = DATA_DIR / "kg_meta.json"

txts = sorted(TXT_DIR.glob("*.txt"))
done = {p.stem for p in JSON_DIR.glob("*.json")} if JSON_DIR.exists() else set()
todo = [p for p in txts if p.stem not in done]
print(f"{len(txts)} belge: {len(done)} tanesinin JSON'u zaten var (LLM tekrar çağrılmaz), {len(todo)} tanesi işlenecek.")

KEEP = re.compile(r"Processing file \d+/\d+|Extracted a total|Standardized \d+|Added -?\d+ inferred|Saved processed|"
                  r"Skipping|generation failed|Error|ERROR|Traceback|Warning: Failed", re.I)
meta = json.loads(KG_META.read_text()) if KG_META.exists() else {"runs": []}
# Builder, üçlü çıkamayan belge için JSON yazmaz. LLM'in tek seferlik bozuk bir cevabı yüzünden belge kaybolmasın diye
# eksik kalan belgeler bir kez daha denenir (JSON'u olan belgeler atlanır, yani LLM yalnızca eksikler için çağrılır).
for attempt in (1, 2):
    todo = [p for p in txts if not (JSON_DIR / f"{p.stem}.json").exists()]
    if not todo:
        break
    if attempt == 2:
        print(f"\n{len(todo)} belge için JSON üretilmedi; bir kez daha deneniyor: {[p.stem[:40] for p in todo]}")
    env = os.environ.copy()
    env.update({"EGOT_BUILDER_WORKERS": str(BUILDER_WORKERS), "EGOT_USAGE_LOG": str(USAGE_LOG)})
    t0 = time.time()
    with open(BUILD_LOG, "a", encoding="utf-8") as log:
        proc = subprocess.Popen([sys.executable, "src/knowledge_graph_builder/main.py", "-i", str(TXT_DIR),
                                 "-o", str(KG_OUT), "-c", str(BUILDER_CFG), "-e", "mhop"],
                                cwd=BUILDER_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if KEEP.search(line) and len(line) < 300:
                print(redact(line.rstrip()))
        proc.wait()
    elapsed = time.time() - t0
    meta["runs"].append({"files": len(todo), "seconds": round(elapsed, 1), "exit": proc.returncode})
    KG_META.write_text(json.dumps(meta))
    print(f"\nÇıkış kodu: {proc.returncode} | süre: {elapsed:.0f} sn | tam log: {BUILD_LOG}")
BUILD_SECONDS = sum(r["seconds"] for r in meta["runs"])
print(f"KG kurulumunun toplam süresi (tüm çalıştırmalar): {BUILD_SECONDS:.0f} sn")

### 6.3 KG istatistikleri ve örnek üçlüler

`inferred=True` olan üçlüler bir chunk'tan doğrudan çıkarılmamış, standardizasyon sonrası çıkarım adımlarında (LLM, geçişli ya da sözcüksel) eklenmiştir. Repo, üçlü çıkarılamayan bir belge için JSON yazmaz (`main.py:233-235`). Böyle bir belgenin chunk'ı da KG'ye girmez, yani retrieval onu hiç bulamaz. Bu yüzden "JSON'u olmayan belge" sayısı ayrıca raporlanır.

In [ ]:
# 6.3 KG istatistikleri ve örnek üçlüler
rows, n_chunks, chunk_words = [], 0, []
for jf in sorted(JSON_DIR.glob("*.json")):
    for ch in json.loads(jf.read_text(encoding="utf-8")):
        group = ch.get("id") == "inferred-triples-group"
        if not group:
            n_chunks += 1
            chunk_words.append(len(ch.get("text", "").split()))
        for t in ch.get("triples", []):
            rows.append({"doc": jf.stem, "subject": t.get("subject"), "predicate": t.get("predicate"),
                         "object": t.get("object"), "inferred": bool(group or t.get("inferred"))})
TRIPLES = pd.DataFrame(rows, columns=["doc", "subject", "predicate", "object", "inferred"])
missing = [p.stem for p in TXT_DIR.glob("*.txt") if not (JSON_DIR / f"{p.stem}.json").exists()]

usage = [json.loads(l) for l in USAGE_LOG.read_text().splitlines()] if USAGE_LOG.exists() else []
BUILDER_USAGE = {"calls": len(usage), "prompt_tokens": sum(u["prompt_tokens"] for u in usage),
                 "completion_tokens": sum(u["completion_tokens"] for u in usage)}
ents = set(TRIPLES["subject"].astype(str)) | set(TRIPLES["object"].astype(str))
stats = {
    "belge (.txt)": len(list(TXT_DIR.glob("*.txt"))),
    "JSON üretilen belge": len(list(JSON_DIR.glob("*.json"))),
    "JSON'u olmayan (üçlü çıkmadığı için atlanan) belge": len(missing),
    "chunk": n_chunks,
    "chunk başına ortalama kelime": round(sum(chunk_words) / max(1, len(chunk_words)), 1),
    "üçlü (toplam)": len(TRIPLES),
    "  chunk'tan çıkarılan": int((~TRIPLES["inferred"]).sum()),
    "  çıkarım (inferred)": int(TRIPLES["inferred"].sum()),
    "benzersiz varlık": len(ents),
    "benzersiz yüklem": TRIPLES["predicate"].nunique(),
    "builder LLM çağrısı": BUILDER_USAGE["calls"],
    "builder token (girdi / çıktı)": f"{BUILDER_USAGE['prompt_tokens']} / {BUILDER_USAGE['completion_tokens']}",
    "builder süresi (sn)": round(BUILD_SECONDS),
}
display(pd.DataFrame(stats.items(), columns=["ölçü", "değer"]))
if missing:
    print("Üçlü çıkmadığı için KG'ye girmeyen belgeler:", missing)
print("\nÖrnek üçlüler (chunk'tan çıkarılan):")
display(TRIPLES[~TRIPLES["inferred"]].sample(min(12, int((~TRIPLES["inferred"]).sum())), random_state=0)
        .assign(doc=lambda d: d["doc"].str[:45]))
if TRIPLES["inferred"].any():
    print("Örnek çıkarım üçlüleri (inferred=True):")
    display(TRIPLES[TRIPLES["inferred"]].head(8).assign(doc=lambda d: d["doc"].str[:45]))
print("En sık 12 yüklem:")
display(TRIPLES["predicate"].value_counts().head(12).rename("adet").to_frame())

### 6.4 Görselleştirme

Repodaki `visualization.visualize_knowledge_graph` (pyvis) ile ilk sorunun kanıt belgelerinden kurulan alt graf çizilir. Kesikli kenarlar çıkarımla eklenen (`inferred`) ilişkilerdir. Düğüm boyutu derece/aradalık merkeziliğine göre belirlenir. Grafik etkileşimlidir: yakınlaştırılabilir, düğümler sürüklenebilir.

In [ ]:
# 6.4 Repodaki visualization.py ile etkileşimli graf (pyvis) — ilk sorunun kanıt belgelerinden kurulan alt graf
viz_docs = set(SEL.iloc[0]["docs"])
sub = TRIPLES[TRIPLES["doc"].isin(viz_docs)]
if len(sub) < 5:                       # alt graf çok küçükse tüm KG'den ilk 150 üçlü
    sub = TRIPLES.head(150)
VIZ_JSON, VIZ_HTML = WORK / "viz_triples.json", WORK / "kg_viz.html"
VIZ_JSON.write_text(json.dumps(sub[["subject", "predicate", "object", "inferred"]].to_dict("records")))
code, out = run_py(f'''
import json
from src.knowledge_graph_builder.visualization import visualize_knowledge_graph
triples = json.load(open({str(VIZ_JSON)!r}))
stats = visualize_knowledge_graph(triples, {str(VIZ_HTML)!r}, edge_smooth="dynamic")
print("Grafik istatistikleri:", stats)
''', cwd=BUILDER_DIR, tail=2)
assert code == 0
print("Soru:", SEL.iloc[0]["query"])
import warnings
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
display(HTML(f'<iframe srcdoc="{html.escape(VIZ_HTML.read_text(encoding="utf-8"))}" width="100%" height="620" '
             'style="border:1px solid #ccc"></iframe>'))

## 7. Neo4j'ye Yükleme, İndeksler ve Kontroller

### 7.1 JSON → CSV ve H17

Repo, JSON'ları CSV'ye çeviren iki betik içerir (`scripts/neo4j/hybrid_…` ve `raw_json_to_csv_generator.py`). İki betiğin de girdi/çıktı yolları yazarların sunucusuna göre sabit yazılmıştır (`/home/exouser/...`, satır 9-10).

- **H17 (sessiz hata):** Betik olduğu gibi çalıştırılınca hata vermez. Bu yolu oluşturur, içinde 0 JSON bulur (`Found 0 JSON files.`), boş CSV'ler yazar ve "✅ CSV export complete!" der. Hücre bunu gösterir, sonra betiğin `/home/exouser` altında oluşturduğu boş klasörleri temizler.
- **Çözüm:** Yollar ortam değişkenlerinden (`EGOT_JSON_DIR`, `EGOT_CSV_DIR`) okunur.

`raw` üreticiyi seçtik, çünkü yüklemi (predicate) olduğu gibi korur. Betik varlık gömmelerini de (aynı `all-MiniLM-L6-v2`, 384 boyut) hesaplar.

In [ ]:
# 7.1 JSON → CSV: repodaki raw_json_to_csv_generator.py (yüklemi olduğu gibi korur)
G_RAW = "src/graph-builder/src/scripts/neo4j/raw_json_to_csv_generator.py"
G_HYB = "src/graph-builder/src/scripts/neo4j/hybrid_json_to_csv_generator.py"
print("\n".join(l for l in (REPO / G_RAW).read_text().splitlines() if l.startswith(("JSON_FOLDER_PATH", "OUTPUT_DIR"))))

# (a) Olduğu gibi çalıştır: sabit /home/exouser/... yolu → hata vermeden 0 dosya işler ve boş CSV yazar
if 'JSON_FOLDER_PATH = "/home/exouser' in (REPO / G_RAW).read_text():
    home_existed = Path("/home/exouser").exists()
    code, out = run([sys.executable, str(REPO / G_RAW)], cwd=WORK, tail=4, timeout=600)
    if "Found 0 JSON files" in out:
        ERRORS.append({"id": "H17", "nerede": G_RAW, "yakalanan mesaj": "Found 0 JSON files (sabit /home/exouser yolu; boş CSV'ler yazıldı)"})
        print("\n➜ [H17] sessiz hata: betik başarılı görünüyor ama hiçbir JSON okumadı.")
    if not home_existed:
        shutil.rmtree("/home/exouser", ignore_errors=True)   # betiğin oluşturduğu boş klasörleri temizle
else:
    print("Yama daha önce uygulanmış; hata gösterimi atlandı (ilk çalıştırmada tetiklenir).")

# (b) Yama: yollar ortam değişkeninden
patch_file(G_RAW, 'JSON_FOLDER_PATH = "/home/exouser/llm-knowledge-graph-builder/data/ultradomain/output/agriculture/agriculture_json_outputs"',
           'JSON_FOLDER_PATH = os.environ["EGOT_JSON_DIR"]  # [Colab yaması G1] sabit /home/exouser yolu yerine', "G1", "JSON klasörü parametre")
patch_file(G_RAW, 'OUTPUT_DIR = "/home/exouser/llm-knowledge-graph-builder/data/ultradomain/output/agriculture/agriculture_csv_outputs"',
           'OUTPUT_DIR = os.environ["EGOT_CSV_DIR"]  # [Colab yaması G1] sabit /home/exouser yolu yerine', "G1", "CSV klasörü parametre")

# (c) Çalıştır
CSV_DIR = DATA_DIR / "csv"
env = os.environ.copy(); env.update({"EGOT_JSON_DIR": str(JSON_DIR), "EGOT_CSV_DIR": str(CSV_DIR)})
code, out = run([sys.executable, str(REPO / G_RAW)], cwd=WORK, env=env, tail=3, timeout=1200)
assert code == 0
CSV = {n: pd.read_csv(CSV_DIR / f"{n}.csv", dtype=str, keep_default_na=False)
       for n in ("documents", "chunks", "entities", "relationships")}
for n, df in CSV.items():
    print(f"{n:14s} {len(df):5d} satır | sütunlar: {list(df.columns)}")
display(CSV["relationships"].head(5))

### 7.2 Yüklem neden ilişki TİPİ olarak saklanıyor?

Retrieval sorgusu bağlamdaki ilişkileri `:özne TİP :nesne` biçiminde yazar (`type(r)`; `constants.py:523-536`). İlişkinin anlamı LLM'e yalnızca **tipi** üzerinden ulaşır; property'ler bağlama hiç girmez.

`hybrid_json_to_csv_generator.normalize_predicate` (satır 41-56) yüklemleri yalnızca beş tipe indirger: `CREATED`, `HAS`, `IS`, `ARE` ve `RELATED`. Bunu da alt dize araması ile yapar; örneğin "chased" içinde "has" geçtiği için `HAS` olur. Böylece "acquired", "sued", "reported" gibi yüklemlerin hepsi `RELATED` olur ve ilişkinin anlamı kaybolur. Aşağıdaki hücre, repo'nun bu fonksiyonunu (betiği çalıştırmadan, yalnızca tanımını alarak) bizim KG'mizin yüklemlerine uygular.

**Bizim yaklaşımımız:** Yüklem, Neo4j'nin kabul ettiği bir tip adına çevrilir: büyük harf, harf/rakam dışı karakterler `_` (ör. `"is CEO of"` → `IS_CEO_OF`). Orijinal yüklem ilişkinin `predicate` property'sinde saklanır. Retrieval sorgusunun hariç tuttuğu ya da şemanın kullandığı tiplerle çakışan adlar (`PART_OF`, `HAS_ENTITY` …) `REL_` önekiyle ayrılır. Aksi hâlde varlıklar arasındaki bir "part of" ilişkisi, sorgunun `!HAS_ENTITY&!PART_OF` filtresi yüzünden genişletmede görünmezdi.

In [ ]:
# 7.2 Neden hybrid değil raw üretici? hybrid_json_to_csv_generator.normalize_predicate'i kendi yüklemlerimizde dene
import ast as _ast

RESERVED_TYPES = {"PART_OF", "HAS_ENTITY", "IN_COMMUNITY", "NEXT", "LAST_MESSAGE"}   # retrieval/şema için ayrılmış tipler


def rel_type(predicate):
    """Yüklemi Neo4j ilişki tipine çevirir: büyük harf, harf/rakam dışı → '_' (orijinali property olarak saklanır)."""
    t = re.sub(r"[^A-Za-z0-9]+", "_", str(predicate)).strip("_").upper() or "RELATED_TO"
    if t[0].isdigit():
        t = "R_" + t
    if t in RESERVED_TYPES:
        t = "REL_" + t
    return t[:64]


hyb_src = (REPO / G_HYB).read_text()
tree = _ast.parse(hyb_src)
ns = {}
for node in tree.body:   # betiğin tamamını çalıştırmadan (model yükler, dosya yazar) yalnızca iki tanımı al
    if (isinstance(node, _ast.FunctionDef) and node.name == "normalize_predicate") or \
       (isinstance(node, _ast.Assign) and getattr(node.targets[0], "id", "") == "COMMON_PREDICATES"):
        exec(_ast.get_source_segment(hyb_src, node), ns)
preds = TRIPLES["predicate"].astype(str).value_counts().head(15).index
display(pd.DataFrame({"yüklem (LLM)": preds,
                      "hybrid üretici tipi": [ns["normalize_predicate"](p)[0] for p in preds],
                      "bizim tipimiz": [rel_type(p) for p in preds]}))
all_preds = TRIPLES["predicate"].astype(str).unique()
print(f"Benzersiz yüklem: {len(all_preds)} → hybrid üretici: {len({ns['normalize_predicate'](p)[0] for p in all_preds})} tip, "
      f"bizim yükleyici: {len({rel_type(p) for p in all_preds})} tip")

### 7.3 Neo4j yükleyici

Repoda olmadığı için yükleyiciyi biz yazdık. Hedef şema, retrieval sorgusunun (`VECTOR_GRAPH_SEARCH_QUERY`) beklediği şemadır:

```
(:Chunk {id, text, embedding})-[:PART_OF]->(:Document {fileName})
(:Chunk)-[:HAS_ENTITY]->(:__Entity__ {id, embedding})
(:__Entity__)-[:<YÜKLEM_TİPİ> {predicate, inferred, fileName, chunkIds}]->(:__Entity__)
indeksler: "vector" (Chunk.embedding, 384 boyut, cosine) ve "keyword" (Chunk.text, full-text)
```

- Yazma işlemleri 500'lük gruplar hâlinde `UNWIND` + `MERGE` ile yapılır. Tekrar çalıştırılırsa aynı düğüm ikinci kez oluşmaz.
- `HAS_ENTITY` kenarları, bir chunk'tan çıkarılan her üçlünün öznesine ve nesnesine bağlanır. Çıkarımla eklenen üçlülerin chunk'ı olmadığı için (`chunkId = "inferred"`) yalnızca varlıklar arası ilişki olarak eklenir.
- Boş ya da 384 boyutlu olmayan gömmeler `null` yazılır, çünkü `vector.similarity.cosine` farklı boyutlu vektörlerde hata verir (yerel testte doğrulandı) ve bu hata bütün retrieval sorgusunu düşürür.
- Yüklemeden önce yalnızca bu notebook'un etiketleri (`Chunk`, `Document`, `__Entity__`, `Session`, `Message`) temizlenir. Benzersizlik kısıtları `MERGE`'ü hızlandırır.

In [ ]:
# 7.3 Neo4j yükleyici (repoda yok; retrieval sorgusunun beklediği şemaya göre yazıldı) — batched UNWIND
EMB_DIM = 384


def _emb(s):
    v = json.loads(s) if s else []
    return v if len(v) == EMB_DIM else None   # boş/yanlış boyutlu vektör → null (vector.similarity.cosine hata verir)


def write_batches(query, rows, size=500):
    for i in range(0, len(rows), size):
        DRIVER.execute_query(query, {"rows": rows[i:i + size]}, database_=DB)
    return len(rows)


def reset_graph():
    """Yalnızca bu notebook'un kullandığı etiketlerdeki düğümleri siler (Aura'da başka verilere dokunmaz)."""
    for label in ("Chunk", "Document", "__Entity__", "Session", "Message"):
        while cypher(f"MATCH (n:`{label}`) WITH n LIMIT 5000 DETACH DELETE n RETURN count(*) AS c")[0]["c"]:
            pass


t0 = time.time()
reset_graph()
for q in ("CREATE CONSTRAINT egot_document IF NOT EXISTS FOR (d:Document) REQUIRE d.fileName IS UNIQUE",
          "CREATE CONSTRAINT egot_chunk IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE",
          "CREATE CONSTRAINT egot_entity IF NOT EXISTS FOR (e:__Entity__) REQUIRE e.id IS UNIQUE"):
    cypher(q)

docs = [{"fileName": r["fileName"]} for r in CSV["documents"].to_dict("records")]
chunks = [{"id": r["id"], "text": r["text"], "fileName": r["fileName"], "embedding": _emb(r["embedding"])}
          for r in CSV["chunks"].to_dict("records")]
ents = [{"id": r["id"], "embedding": _emb(r["embedding"])} for r in CSV["entities"].to_dict("records") if r["id"].strip()]
rels = [r for r in CSV["relationships"].to_dict("records") if r["subject"].strip() and r["object"].strip()]

n = {}
n["Document"] = write_batches("UNWIND $rows AS row MERGE (:Document {fileName: row.fileName})", docs)
n["Chunk"] = write_batches("""
    UNWIND $rows AS row
    MERGE (c:Chunk {id: row.id})
    SET c.text = row.text, c.fileName = row.fileName, c.embedding = row.embedding
    WITH c, row MATCH (d:Document {fileName: row.fileName})
    MERGE (c)-[:PART_OF]->(d)""", chunks)
n["__Entity__"] = write_batches("UNWIND $rows AS row MERGE (e:__Entity__ {id: row.id}) SET e.embedding = row.embedding", ents)
chunk_rels = [{"chunkId": r["chunkId"], "s": r["subject"], "o": r["object"]} for r in rels if r["chunkId"] != "inferred"]
n["HAS_ENTITY (üçlü)"] = write_batches("""
    UNWIND $rows AS row
    MATCH (c:Chunk {id: row.chunkId})
    MATCH (s:__Entity__ {id: row.s})
    MATCH (o:__Entity__ {id: row.o})
    MERGE (c)-[:HAS_ENTITY]->(s)
    MERGE (c)-[:HAS_ENTITY]->(o)""", chunk_rels)

# Varlıklar arası ilişkiler: yüklem → ilişki TİPİ (bağlama type(r) yazılır), orijinal yüklem property olarak
by_type = {}
for r in rels:
    by_type.setdefault(rel_type(r["predicate"]), []).append(
        {"s": r["subject"], "o": r["object"], "p": r["predicate"], "file": r["fileName"],
         "chunk": r["chunkId"], "inferred": r["chunkId"] == "inferred"})
for t, rows in by_type.items():   # t yalnızca [A-Z0-9_] içerir; sorguya güvenle yazılabilir
    write_batches(f"""
        UNWIND $rows AS row
        MATCH (s:__Entity__ {{id: row.s}})
        MATCH (o:__Entity__ {{id: row.o}})
        MERGE (s)-[r:`{t}`]->(o)
        ON CREATE SET r.predicate = row.p, r.inferred = row.inferred, r.fileName = row.file, r.chunkIds = [row.chunk]
        ON MATCH SET r.chunkIds = CASE WHEN row.chunk IN r.chunkIds THEN r.chunkIds ELSE r.chunkIds + row.chunk END""", rows)
n["varlık-varlık ilişkisi (satır)"] = len(rels)
n["ilişki tipi"] = len(by_type)

# İndeksler: retrieval'ın beklediği adlarla
cypher("""CREATE VECTOR INDEX `vector` IF NOT EXISTS FOR (c:Chunk) ON (c.embedding)
          OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}""")
cypher("CREATE FULLTEXT INDEX `keyword` IF NOT EXISTS FOR (c:Chunk) ON EACH [c.text]")
cypher("CALL db.awaitIndexes(300)")
print(f"Yükleme {time.time() - t0:.1f} sn sürdü.")
display(pd.DataFrame(n.items(), columns=["yazılan", "adet"]))

### 7.4 Kontroller

1. APOC smoke test: `RETURN apoc.text.join(['a','b'],'-')`.
2. Etiket ve ilişki tipi sayıları.
3. `vector` ve `keyword` indekslerinin durumu (`ONLINE`) ve yapılandırması (384 boyut, cosine).
4. Retrieval sorgusunun doğrudan testi: `constants.VECTOR_GRAPH_SEARCH_QUERY`, ilk sorunun gömmesiyle (`$query_vector`) vektör indeksi üzerinden çalıştırılır. Dönen bağlamda chunk metni, varlıklar ve ilişkiler görünmelidir. Sabitler `src` paketi import edilmeden, dosyadan doğrudan okunur.

In [ ]:
# 7.4 Kontroller: APOC, düğüm/ilişki sayıları, indeksler ve retrieval sorgusunun doğrudan testi
print("APOC smoke test:", cypher("RETURN apoc.text.join(['a','b'],'-') AS joined, apoc.coll.toSet([1,1,2]) AS toSet, "
                                 "apoc.version() AS apoc")[0])
display(pd.DataFrame(cypher("MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS adet ORDER BY adet DESC")))
rel_counts = pd.DataFrame(cypher("MATCH ()-[r]->() RETURN type(r) AS tip, count(*) AS adet ORDER BY adet DESC"))
print(f"İlişki tipi sayısı: {len(rel_counts)} | toplam ilişki: {int(rel_counts['adet'].sum())}. En sık 12 tip:")
display(rel_counts.head(12))
display(pd.DataFrame(cypher("SHOW INDEXES YIELD name, type, state, labelsOrTypes, properties, options "
                            "WHERE type IN ['VECTOR','FULLTEXT'] RETURN name, type, state, labelsOrTypes, properties, "
                            "options.indexConfig AS config")))
print("Gömmesi olmayan chunk / varlık:",
      cypher("MATCH (c:Chunk) WHERE c.embedding IS NULL RETURN count(c) AS c")[0]["c"], "/",
      cypher("MATCH (e:__Entity__) WHERE e.embedding IS NULL RETURN count(e) AS c")[0]["c"])

# retrieval sorgusunu (constants.VECTOR_GRAPH_SEARCH_QUERY) doğrudan çalıştır — `src` paketini import etmeden
import importlib.util
spec = importlib.util.spec_from_file_location("egot_constants", RETR_DIR / "constants.py")
egot_constants = importlib.util.module_from_spec(spec); spec.loader.exec_module(egot_constants)
from sentence_transformers import SentenceTransformer
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
q = SEL.iloc[0]["query"]
qv = EMBEDDER.encode(q).tolist()
res = cypher("CALL db.index.vector.queryNodes('vector', 4, $query_vector) YIELD node, score "
             + egot_constants.VECTOR_GRAPH_SEARCH_QUERY, query_vector=qv)
print(f"\nSoru: {q}\nDönen belge: {len(res)}")
for r in res[:2]:
    m = r["metadata"]
    print(f"\n— kaynak: {m['source']} | skor: {r['score']:.3f} | chunk: {len(m['chunkdetails'])} | "
          f"varlık: {len(m['entities']['entityids'])} | ilişki: {len(m['entities']['relationshipids'])}")
    print(textwrap.indent(r["text"][:900], "   "))

## 8. Retrieval Sunucusu ve Tek Soruluk GoT Demosu

### 8.1 Sunucuyu başlatma

FastAPI uygulaması, Bölüm 4.6'daki sunucu klasöründe `uvicorn app:app` ile arka planda başlatılır. Loglar `logs/server.log` dosyasına yazılır. Sunucu Neo4j ve LLM bilgilerini ortam değişkenlerinden okur: `NEO4J_URI/USERNAME/PASSWORD/DATABASE`, `OPENAI_API_KEY`, `OPENAI_BASE_URL`. GoT derinliği (`EGOT_MAX_DEPTH`) ve sentez biçimi (`EGOT_SYNTHESIS_STYLE`) de başlatma sırasında verilir. Ayarlar değişmiş olabileceği için hücre sunucuyu her çalıştırmada yeniden başlatır.

İstekler `POST /chat_bot` uç noktasına form alanlarıyla gönderilir: `question`, `model`, `mode`, `session_id`, `write_access`, `evaluation`. Her soruya ayrı bir `session_id` verilir (Bölüm 4.12, not 4).

In [ ]:
# 8.1 FastAPI sunucusunu (uvicorn) arka planda başlat — loglar dosyaya
SERVER_URL = "http://127.0.0.1:8000"
SERVER_LOG = LOG_DIR / "server.log"


def server_up():
    try:
        return requests.get(SERVER_URL + "/", timeout=3).ok
    except requests.RequestException:
        return False


def stop_server():
    run(["pkill", "-f", "uvicorn app:app --host 127.0.0.1 --port 8000"], show=False)
    for _ in range(20):
        if not server_up():
            return
        time.sleep(1)


def start_server():
    stop_server()                                   # ayarlar (MAX_DEPTH vb.) değişmiş olabilir: her zaman yeniden başlat
    env = os.environ.copy()
    env.update({"EGOT_MAX_DEPTH": str(MAX_DEPTH), "EGOT_SYNTHESIS_STYLE": SYNTHESIS_STYLE})
    log = open(SERVER_LOG, "a", encoding="utf-8")
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
                            cwd=SERVER_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while not server_up():
        if proc.poll() is not None or time.time() - t0 > 300:
            run(["tail", "-n", "30", str(SERVER_LOG)])
            raise RuntimeError("Sunucu başlatılamadı (log yukarıda)")
        time.sleep(2)
    print(f"Sunucu {time.time() - t0:.0f} sn'de hazır →", requests.get(SERVER_URL + "/").json())


def ask(question, mode="graph_of_thought", timeout=1800):
    """POST /chat_bot — her soru için ayrı session_id (sohbet geçmişi sorular arasında taşınmaz)."""
    data = {"question": question, "model": LLM_MODEL, "mode": mode, "session_id": f"{mode}-{uuid.uuid4().hex[:10]}",
            "write_access": "false",   # geçmiş Neo4j yerine sunucu belleğinde tutulur (grafa Session/Message eklenmez)
            "evaluation": "false"}
    t0 = time.time()
    r = requests.post(SERVER_URL + "/chat_bot", data=data, timeout=timeout)
    r.raise_for_status()
    return r.json()["data"], time.time() - t0


def log_size():
    return SERVER_LOG.stat().st_size if SERVER_LOG.exists() else 0


def log_since(offset):
    with open(SERVER_LOG, encoding="utf-8", errors="ignore") as f:
        f.seek(offset)
        return f.read()


# Tahmin kayıtları: her cevap dosyaya yazılır; kaydı olan soru tekrar sorulmaz (demo sorusu da Bölüm 9'da yeniden kullanılır)
PRED_DIR = DATA_DIR / "predictions"; PRED_DIR.mkdir(exist_ok=True)
MODEL_TAG = re.sub(r"[^A-Za-z0-9.]+", "-", LLM_MODEL)
GOT_FILE = PRED_DIR / f"got_{MODEL_TAG}_d{MAX_DEPTH}_{SYNTHESIS_STYLE}.jsonl"


def load_preds(path):
    return {r["qid"]: r for r in map(json.loads, path.read_text().splitlines())} if path.exists() else {}


def ask_and_record(q, mode, path, attempts=2):
    """Soruyu sorar (geçici bir hatada bir kez daha dener), sonucu `path` dosyasına ekler ve kaydı döndürür.
    Başarısız olursa None döndürür ve hiçbir şey yazmaz; böylece hücre yeniden çalıştırılınca soru yine sorulur."""
    for attempt in range(1, attempts + 1):
        off = log_size()
        try:
            res, secs = ask(q["query"], mode=mode)
            info = res.get("info", {})
            err = info.get("error") or (None if res.get("message") not in (None, "", "Something went wrong")
                                        else "sunucu boş/hatalı cevap döndürdü")
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        if not err:
            break
        print(f"   {q['qid']}: deneme {attempt}/{attempts} başarısız → {redact(err)}")
    else:
        return None
    depth_counts = [int(k) for _, k in re.findall(r"Depth (\d+): Expanding (\d+) thoughts", log_since(off))]
    tp = info.get("thought_process", []) or []
    rec = {"qid": q["qid"], "question_type": q["question_type"], "question": q["query"], "answer": str(q["answer"]),
           "mode": mode, "prediction": res.get("message", ""), "seconds": round(secs, 1),
           "llm_usage": info.get("llm_usage"), "max_depth": info.get("max_depth"),
           "synthesis_style": info.get("synthesis_style"), "sources": info.get("sources", []),
           "n_thoughts": len(tp), "depth_counts": depth_counts,
           "empty_retrievals": sum(str(t.get("answer", "")).startswith("No relevant information found") for t in tp),
           # düşünce süreci (retrieval bağlamı olmadan) sonradan incelemek için saklanır
           "thought_process": [{"thought": t.get("thought"), "answer": t.get("answer"), "sources": t.get("sources", [])}
                               for t in tp]}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return rec


start_server()

### 8.2 Retrieval sağlık testi

GoT, retrieval hatalarını yutup kanıtsız devam eder (Bölüm 4.12, not 1). Bu yüzden demodan önce repo'nun **kendi** retriever zinciri ayrı bir süreçte, ilk üç soruyla çalıştırılır: `get_neo4j_retriever` → `Neo4jVector` hibrit arama + `VECTOR_GRAPH_SEARCH_QUERY` → `ContextualCompressionRetriever` (TokenTextSplitter + EmbeddingsFilter). Her soru için dönen doküman sayısı, kaynakları ve sorguya benzerlikleri yazdırılır. Doküman dönmezse hücre durur.

In [ ]:
# 8.2 Retrieval sağlık testi: repo'nun kendi retriever zinciriyle (QA_integration) doküman dönüyor mu?
# GoT, retrieval hatalarını yutup "No relevant information found" ile devam ettiği için bunu ayrıca kontrol ediyoruz.
code, out = run_py(f'''
import logging
logging.basicConfig(level=logging.WARNING)
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)   # Cypher uyarıları (bkz. Bölüm 4.12) gizlenir
from langchain_neo4j import Neo4jGraph
from langchain_core.messages import HumanMessage
from src.QA_integration import (get_llm, get_neo4j_retriever, create_document_retriever_chain,
                                retrieve_documents, get_chat_mode_settings)
graph = Neo4jGraph(refresh_schema=False, sanitize=True)          # NEO4J_URI/USERNAME/PASSWORD/DATABASE ortamdan
settings = get_chat_mode_settings("graph_of_thought")
llm, _ = get_llm({LLM_MODEL!r})
chain = create_document_retriever_chain(llm, get_neo4j_retriever(graph=graph, document_names=None, chat_mode_settings=settings))
for q in {list(SEL["query"].head(3))!r}:
    docs, _ = retrieve_documents(chain, [HumanMessage(content=q)])
    print(f"\\nSoru: {{q[:110]}}")
    print(f"  dönen doküman: {{len(docs or [])}}")
    for d in (docs or [])[:3]:
        chunks = len(d.metadata.get("chunkdetails", []))
        print(f"  - {{str(d.metadata.get('source'))[:70]}} | chunk: {{chunks}} | benzerlik: "
              f"{{d.state.get('query_similarity_score', 0):.3f}}")
    assert docs, "Retrieval boş döndü — şema/indeksleri kontrol edin"
''', cwd=SERVER_DIR, show=False)
# initialize_neo4j_vector retrieval sorgusunun tamamını yazdırıyor; yalnızca sonuç satırlarını göster
lines = out.splitlines()
start = next((i for i, l in enumerate(lines) if l.startswith("Soru:")), 0)
print("\n".join(l[:250] for l in lines[start:] if not l.startswith(("WARNING", "INFO"))))
assert code == 0, "Retrieval sağlık testi başarısız"

### 8.3 GoT demosu

İlk seçilen soru `mode=graph_of_thought` ile sorulur. Çıktıda sırasıyla şunlar görünür:
- final cevap, altın cevap, süre ve ölçülen LLM çağrısı/token sayısı (`info.llm_usage`);
- **düşünce süreci** (`info.thought_process`): her adımda araştırılan thought, o thought için bulunan belgelerle üretilen ara cevap ve kaynak belgeler;
- isteğin işlendiği sürede sunucu loguna düşen hata satırları.

Derinlik etiketleri sunucu logundaki `Depth d: Expanding n thoughts` satırlarından okunur. Uzun ara cevaplar kısaltılarak gösterilir.

Demonun cevabı, toplu çalıştırmanın kullandığı tahmin dosyasına (Bölüm 9.1) yazılır. Bu yüzden aynı soru Bölüm 9'da **tekrar sorulmaz**. Hücre yeniden çalıştırılırsa cevap dosyadan okunur ve LLM çağrılmaz. Geçici bir sunucu ya da API hatasında istek bir kez daha denenir.

In [ ]:
# 8.3 Tek soruluk GoT demosu: /chat_bot'a mode=graph_of_thought ile istek
demo = SEL.iloc[0]
print(wrap(f"Soru ({demo['question_type']}): {demo['query']}"))
print("Altın cevap:", demo["answer"])
DEMO = load_preds(GOT_FILE).get(demo["qid"])
if DEMO:
    print("\n(Bu soru bu ayarlarla daha önce soruldu; cevap dosyadan okundu, LLM çağrılmadı.)")
else:
    off = log_size()
    DEMO = ask_and_record(demo, "graph_of_thought", GOT_FILE)   # Bölüm 9.1 bu kaydı yeniden kullanır
    if not DEMO:
        run(["tail", "-n", "30", str(SERVER_LOG)])
        raise RuntimeError("GoT isteği iki denemede de başarısız oldu; sunucu logunun son satırları yukarıda.")
    errs = [l for l in log_since(off).splitlines() if re.search(r"ERROR|Error ", l)]
    print(f"Bu istek sırasında sunucu logundaki hata satırı sayısı: {len(errs)}")
    for l in errs[:5]:
        print("  ", redact(l[:250]))

print(f"\neGOT cevabı: {DEMO['prediction']!r}")
print(f"Süre: {DEMO['seconds']:.1f} sn | LLM kullanımı: {DEMO.get('llm_usage')} | derinlik: {DEMO.get('max_depth')} | "
      f"sentez: {DEMO.get('synthesis_style')}")

# Düşünce süreci: thought → ara cevap → kaynaklar. Derinlikler sunucu logundaki "Depth d: Expanding n thoughts"
# satırlarından okunur (her derinlikte kaç thought işlendiği).
labels = [d for d, k in enumerate(DEMO.get("depth_counts", []), 1) for _ in range(k)]
for i, step in enumerate(DEMO.get("thought_process", [])):
    print(f"\n── Derinlik {labels[i] if i < len(labels) else '?'} · adım {i + 1} " + "─" * 60)
    print(wrap(step["thought"], indent="Thought  : ")[:600])
    print(wrap(step["answer"], indent="Ara cevap: ")[:900])
    print("Kaynaklar:", ", ".join(s[:60] for s in step["sources"]) or "—")

## 9. Toplu Çalıştırma ve Değerlendirme

### 9.1 GoT ile bütün sorular

Seçilen her soru ayrı bir `session_id` ile `graph_of_thought` modunda sorulur. Her soru için şunlar kaydedilir: süre, ölçülen LLM çağrısı ve token sayısı, kaynaklar, boş dönen retrieval adımlarının sayısı ve düşünce süreci (thought → ara cevap → kaynaklar). Sonuçlar `predictions/*.jsonl` dosyasına yazılır.
- Kaydı olan sorular (Bölüm 8.3'teki demo sorusu dahil) tekrar sorulmaz.
- Geçici bir hata alan soru bir kez daha denenir. İki denemede de başarısız olan soru kaydedilmez, böylece hücre yeniden çalıştırılınca yalnızca o soru sorulur.

**Ölçüler:**
- **EM / F1:** Repodaki `scripts/eval_multihopqa.py` fonksiyonları. Cevap ve tahmin küçük harfe çevrilir; noktalama ve `a/an/the` atılır. EM birebir eşleşmedir, F1 kelime düzeyinde örtüşmedir.
- **İçerme** (bizim eklememiz): Normalize edilmiş altın cevap, normalize edilmiş tahminin içinde geçiyor mu? Uzun cevaplar için bilgi verir.
- **Kanıt bulma** (bizim eklememiz): Sorunun kanıt belgelerinden kaçı GoT'un kullandığı kaynaklar arasında? Retrieval'ın doğru belgelere ulaşıp ulaşmadığını gösterir.

In [ ]:
# 9.1 Toplu çalıştırma: her soru ayrı session_id ile; sonuçlar dosyaya yazılır (varsa tekrar sorulmaz)
def run_batch(mode, path):
    preds = load_preds(path)
    for i, q in enumerate(SEL.to_dict("records"), 1):
        if q["qid"] in preds:
            print(f"[{i}/{len(SEL)}] {q['qid']}: dosyadan okundu (LLM çağrılmadı)")
            continue
        rec = ask_and_record(q, mode, path)
        if rec is None:
            print(f"[{i}/{len(SEL)}] {q['qid']}: iki denemede de cevap alınamadı (kaydedilmedi; hücre tekrar çalıştırılabilir)")
            continue
        preds[q["qid"]] = rec
        print(f"[{i}/{len(SEL)}] {q['qid']}: {rec['seconds']:5.1f} sn | tahmin: {rec['prediction'][:70]!r} | "
              f"altın: {q['answer']!r}")
    return preds


def score(preds):
    df = pd.DataFrame([preds[k] for k in SEL["qid"] if k in preds])
    if df.empty:
        return df
    df["EM"] = [compute_em(p, a) for p, a in zip(df["prediction"], df["answer"])]
    df["F1"] = [compute_f1(p, a) for p, a in zip(df["prediction"], df["answer"])]
    # Ek ölçüler (repo'da yok, bizim eklememiz): altın cevap tahminin içinde geçiyor mu? kanıt belgeleri bulundu mu?
    df["içerme"] = [float(normalize_answer(a) in normalize_answer(p)) for p, a in zip(df["prediction"], df["answer"])]
    gold = dict(zip(SEL["qid"], SEL["docs"]))
    df["kanıt_bulma"] = [len({Path(s).stem for s in srcs} & set(gold[k])) / len(gold[k])
                         for k, srcs in zip(df["qid"], df["sources"])]
    return df


GOT_PREDS = run_batch("graph_of_thought", GOT_FILE)
GOT = score(GOT_PREDS)
assert not GOT.empty, "Hiç GoT tahmini yok — sunucu logunu (LOG_DIR/server.log) kontrol edin"
GOT["llm_çağrısı"] = GOT["llm_usage"].map(lambda u: (u or {}).get("calls"))
GOT["token_girdi"] = GOT["llm_usage"].map(lambda u: (u or {}).get("prompt_tokens"))
GOT["token_çıktı"] = GOT["llm_usage"].map(lambda u: (u or {}).get("completion_tokens"))
GOT.drop(columns=["llm_usage", "thought_process"], errors="ignore").to_csv(
    PRED_DIR / f"sonuclar_got_{MODEL_TAG}_d{MAX_DEPTH}.csv", index=False)
display(GOT[["qid", "question_type", "answer", "prediction", "EM", "F1", "içerme", "kanıt_bulma", "seconds",
             "llm_çağrısı", "token_girdi", "token_çıktı", "empty_retrievals"]])
print("Kaydedildi:", PRED_DIR / f"sonuclar_got_{MODEL_TAG}_d{MAX_DEPTH}.csv",
      "| düşünce süreçleriyle birlikte tüm cevaplar:", GOT_FILE.name)

### 9.2 Karşılaştırma: GoT'suz mod (`graph_vector_fulltext`)

Aynı sorular, eGOT'un GoT kullanmayan varsayılan moduyla da sorulur. Bu modda aynı hibrit, grafla genişletilen retrieval bir kez yapılır ve LLM bir kez cevap verir; cevap sohbet geçmişine yazılır ve arka planda özetlenir. Bu modun cevap prompt'u (`CHAT_SYSTEM_TEMPLATE`) 4–5 cümlelik açıklamalı cevap ister. GoT'un `mhop` sentezi ise tek kelime ya da kısa bir ifade döndürür. Bu yüzden iki modun **EM değerleri doğrudan karşılaştırılamaz**; içerme ve kanıt bulma ölçüleri daha adil bir karşılaştırma sağlar.

In [ ]:
# 9.2 (İsteğe bağlı) Aynı sorular GoT'suz modda: graph_vector_fulltext (tek retrieval + tek cevap)
BASE = pd.DataFrame()
if RUN_BASELINE:
    BASE_FILE = PRED_DIR / f"baseline_{MODEL_TAG}.jsonl"
    BASE = score(run_batch("graph_vector_fulltext", BASE_FILE))
    if not BASE.empty:
        BASE.drop(columns=["llm_usage", "thought_process"], errors="ignore").to_csv(
            PRED_DIR / f"sonuclar_baseline_{MODEL_TAG}.csv", index=False)
        display(BASE[["qid", "question_type", "answer", "prediction", "EM", "F1", "içerme", "kanıt_bulma", "seconds"]])
else:
    print("RUN_BASELINE = False: karşılaştırma atlandı.")

### 9.3 Özet tablo, soru türlerine göre sonuçlar, maliyet ve grafik

Tahmini maliyet, ölçülen token sayıları ile Bölüm 0.1'deki fiyatlardan hesaplanır. Varsayılan yerel modelde API ücreti olmadığı için bu sütun boş kalır; token sayıları yine ölçülür. İndeksleme satırı builder'ın bütün LLM çağrılarını içerir. Sorgu satırı bütün GoT sorularını (Bölüm 8.3'teki demo sorusu dahil) kapsar; Bölüm 3–4'teki birkaç test çağrısı dahil değildir.

Ardından iki modun cevapları soru soru yan yana gösterilir. En sonda ortalama ölçüler ve soru başına süre bir grafikte özetlenir. Süre farklı bir ölçekte olduğu için ayrı bir grafikte çizilir (iki y-eksenli tek grafik yanıltıcı olurdu). Grafikteki değerlerin tamamı yukarıdaki tablodadır; grafik `predictions/` klasörüne PNG olarak da kaydedilir.

In [ ]:
# 9.3 Özet: EM/F1 (scripts/eval_multihopqa.py), süre ve LLM kullanımı
theory = 2 + 9 * MAX_DEPTH + 1


def cost(pt, ct):
    if PRICE_INPUT_PER_1M is None or PRICE_OUTPUT_PER_1M is None:
        return None
    return round(pt / 1e6 * PRICE_INPUT_PER_1M + ct / 1e6 * PRICE_OUTPUT_PER_1M, 4)


rows = [{"mod": f"graph_of_thought (d={MAX_DEPTH})", "soru": len(GOT), "EM": GOT["EM"].mean(), "F1": GOT["F1"].mean(),
         "içerme": GOT["içerme"].mean(), "kanıt_bulma": GOT["kanıt_bulma"].mean(),
         "ort. süre (sn)": GOT["seconds"].mean(), "ort. LLM çağrısı (ölçülen)": GOT["llm_çağrısı"].mean(),
         "LLM çağrısı (teorik)": theory, "ort. token": (GOT["token_girdi"] + GOT["token_çıktı"]).mean()}]
if not BASE.empty:
    rows.append({"mod": "graph_vector_fulltext (GoT'suz)", "soru": len(BASE), "EM": BASE["EM"].mean(), "F1": BASE["F1"].mean(),
                 "içerme": BASE["içerme"].mean(), "kanıt_bulma": BASE["kanıt_bulma"].mean(),
                 "ort. süre (sn)": BASE["seconds"].mean(), "ort. LLM çağrısı (ölçülen)": None,
                 "LLM çağrısı (teorik)": "1 + 1 (arka planda özet)", "ort. token": None})
SUMMARY = pd.DataFrame(rows)
display(SUMMARY.round(3))

print("Soru türüne göre (GoT):")
display(GOT.groupby("question_type")[["EM", "F1", "içerme", "kanıt_bulma", "seconds", "llm_çağrısı"]].mean().round(3))

got_pt, got_ct = int(GOT["token_girdi"].sum()), int(GOT["token_çıktı"].sum())
COSTS = pd.DataFrame([
    {"aşama": "İndeksleme (KG kurulumu)", "LLM çağrısı": BUILDER_USAGE["calls"], "token girdi": BUILDER_USAGE["prompt_tokens"],
     "token çıktı": BUILDER_USAGE["completion_tokens"], "süre (sn)": round(BUILD_SECONDS),
     "tahmini maliyet ($)": cost(BUILDER_USAGE["prompt_tokens"], BUILDER_USAGE["completion_tokens"])},
    {"aşama": f"Sorgu: GoT, {len(GOT)} soru", "LLM çağrısı": int(GOT["llm_çağrısı"].sum()), "token girdi": got_pt,
     "token çıktı": got_ct, "süre (sn)": round(GOT["seconds"].sum()), "tahmini maliyet ($)": cost(got_pt, got_ct)},
])
print("İndeksleme ve sorgu maliyeti (bu çalıştırma):")
display(COSTS)
if not BASE.empty:
    print("Not: GoT'suz mod uzun, cümle hâlinde cevap verir (CHAT_SYSTEM_TEMPLATE: '4-5 sentences'); EM bu yüzden ~0 olabilir.\n"
          "Karşılaştırmada 'içerme' ve F1 daha bilgilendiricidir. Bu iki modun cevap biçimleri farklıdır.")

    # Soru soru yan yana: altın cevap, GoT cevabı, GoT'suz cevabın başı
    side = GOT[["qid", "question_type", "answer", "prediction", "EM", "içerme", "seconds"]].merge(
        BASE[["qid", "prediction", "içerme", "seconds"]], on="qid", suffixes=(" GoT", " GoT'suz"))
    side["prediction GoT'suz"] = side["prediction GoT'suz"].map(lambda s: s if len(s) <= 140 else s[:140] + "…")
    side = side.rename(columns={"answer": "altın", "prediction GoT": "GoT cevabı", "prediction GoT'suz": "GoT'suz cevap",
                                "EM": "EM GoT"})
    print("\nSoru soru karşılaştırma:")
    display(side)

# Grafik: solda iki modun ölçüleri, sağda soru başına süre. İki ölçek farklı olduğu için iki ayrı grafik
# (çift y-ekseni kullanılmaz). Renkler kimliği taşır ve iki grafikte aynıdır; değerlerin tablosu yukarıdadır.
import matplotlib.pyplot as plt
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = {"GoT": "#2a78d6", "GoT'suz": "#eb6834"}   # kategorik palet, slot 1-2 (renk körlüğü kontrollerinden geçer)
modes = [("GoT", GOT)] + ([("GoT'suz", BASE)] if not BASE.empty else [])
metrics = [("EM", "EM"), ("F1", "F1"), ("içerme", "İçerme"), ("kanıt_bulma", "Kanıt bulma")]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [3, 1.2]}, facecolor=SURFACE)
bar_w, gap = 0.13, 0.02
for j, (name, df) in enumerate(modes):
    offset = (j - (len(modes) - 1) / 2) * (bar_w + gap)
    vals = [df[m].mean() for m, _ in metrics]
    ax1.bar([i + offset for i in range(len(metrics))], vals, width=bar_w, color=SERIES[name], label=name, zorder=2)
    for i, v in enumerate(vals):                   # 0 değerli çubuk görünmez; "veri yok" sanılmasın diye 0 yazılır
        if v < 0.005:
            ax1.text(i + offset, 0.01, "0", ha="center", va="bottom", fontsize=8, color=MUTED)
ax1.set_xticks(range(len(metrics)), [label for _, label in metrics])
ax1.set_ylim(0, 1.05)
ax1.set_title(f"Ortalama ölçüler ({len(GOT)} soru)", loc="left", fontsize=11, color=INK)
ax1.legend(frameon=False, ncol=len(modes), loc="lower right", bbox_to_anchor=(1.0, 1.0), labelcolor=INK2, fontsize=9,
           handlelength=1.0, borderaxespad=0.2)   # başlık satırında, çizim alanının dışında: çubuklarla çakışmaz
secs = [df["seconds"].mean() for _, df in modes]
ax2.bar(range(len(modes)), secs, width=0.35, color=[SERIES[n] for n, _ in modes], zorder=2)
for i, v in enumerate(secs):                       # yalnızca iki çubuk: değerler uçlarına yazılır
    ax2.text(i, v, f"{v:.1f} sn", ha="center", va="bottom", fontsize=9, color=INK2)
ax2.set_xticks(range(len(modes)), [n for n, _ in modes])
ax2.set_xlim(-0.6, len(modes) - 0.4)
ax2.set_ylim(0, max(secs) * 1.2)
ax2.set_title("Soru başına süre", loc="left", fontsize=11, color=INK)
for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    for side_name in ("top", "right", "left"):
        ax.spines[side_name].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelcolor=INK2, length=0, labelsize=9)
fig.tight_layout()
fig.savefig(PRED_DIR / f"ozet_grafik_{MODEL_TAG}_d{MAX_DEPTH}.png", dpi=150, facecolor=SURFACE)
plt.show()

## 10. Karşılaşılan Hatalar ve Çözümleri

### 10.1 Hata tablosu

Tablodaki her hata bu notebook'ta gerçekten tetiklenir; ilgili hücrenin çıktısında mesajı görünür ve Bölüm 10.2'de otomatik olarak listelenir. H5 GPU çalışma zamanında gerçekten oluşur; CPU çalışma zamanında ise yalnızca taklit edilerek gösterilebilir. H7'de ise ağ isteği gönderilmemesi için bir kayıt istemcisi kullanıldı. Satır numaraları orijinal dosyalara (commit `3d37906`) aittir.

| # | Hata | Belirti | Sebep | Çözüm | Dosya:satır |
|---|---|---|---|---|---|
| H1 | README'deki ana komut yok | `python: can't open file '.../src/main.py'` | README'de anlatılan PubMed hattı repoya konmamış | Builder doğrudan `knowledge_graph_builder/main.py` ile çalıştırıldı | `src/graph-builder/README.md:51,150` |
| H2 | `requirements.txt` kurulamıyor | `pip --dry-run`: `pysb` derlemesi `BadZipFile` → `ERROR: Failed to build 'pysb'` | Donmuş, büyük bir ortam listesi (84 paket); `pysb` eski `ez_setup` ile setuptools indirmeye çalışıyor; liste Colab paketlerini de eski sürümlere sabitliyor | Yalnızca kodun import ettiği paketler kuruldu | `src/graph-builder/requirements.txt:57,73` |
| H3 | Retrieval için `requirements.txt` yok | `ERROR: Could not open requirements file` | README'de var deniyor ama dosya yok | Paket listesi kodun import'larından çıkarıldı | `README.md:39` |
| H4 | Builder import edilemiyor | `ImportError: cannot import name 'MAIN_LUPUS_USER_PROMPT'` | 8 prompt sabiti `prompts.py`'de yok | Var olan iki prompt import edildi, `MAIN_USER_PROMPT` kullanıldı | `knowledge_graph_builder/main.py:35-46,66` |
| H5 | Gömme modeli yüklenemiyor (CPU'da taklit) | `Error loading model ...` → model `None`, chunk'lar atlanıyor | Sabit `cuda:2` | `cuda` (varsa) / `cpu` | `knowledge_graph_builder/embedding.py:21` |
| H6 | Hata yakalama bloğu çöküyor | `TypeError: print() got an unexpected keyword argument 'exc_info'` | `print`'e `exc_info` verilmiş | `print` + `traceback.print_exc()` | `knowledge_graph_builder/main.py:352` |
| H7 | LLM yanlış sunucuya gidiyor | İstemciye `llm.jetstream-cloud.org`, `api_key='empty'` veriliyor | `call_openai` adresi ve anahtarı sabit yazıyor, config'i yok sayıyor | Config `base_url` + ortamdaki anahtar + 6 yeniden deneme | `knowledge_graph_builder/llm.py:114-116` |
| H8 | Boş klasörde çökme / sabit paralellik | `ValueError: max_workers must be greater than 0` | `min(6, len(input_files))` | `EGOT_BUILDER_WORKERS` parametresi, en az 1 | `knowledge_graph_builder/main.py:342` |
| H9 | Sunucu README'deki gibi başlamıyor | `ModuleNotFoundError: No module named 'src'` | Kod `from src....` diye import ediyor, klasör düzeni farklı | `<server>/src` + `<server>/app.py` düzeni (sembolik bağlantı) | `app.py:6`, `QA_integration.py:31,33`, `got.py:11-16` |
| H10 | LangChain 1.x uyumsuzluğu | `ModuleNotFoundError: No module named 'langchain.prompts'` | `langchain.prompts/retrievers` 1.x'te kaldırıldı | `langchain_core.prompts`, `langchain_classic.retrievers(...)` | `QA_integration.py:14,17,19`, `got.py:8`, `metrics/evaluation.py:3` |
| H11 | ragas zorunlu import | `ModuleNotFoundError: No module named 'ragas'` | Modül seviyesinde import; ragas 0.4.3 güncel langchain-community ile de import edilemiyor | `try/except`; yalnızca `evaluation=True` iken gerekli | `metrics/evaluation.py:6-10` |
| H12 | GoT modu import edilemiyor | `ImportError: cannot import name 'FORMAT_ANSWER_PROMPT'` | Sabit `constants.py`'de tanımlı değil | İsteğe bağlı import | `got.py:16` |
| H13 | Final sentez çöküyor | `NameError: name 'client' is not defined` (kullanıcı yalnızca "An error occurred..." görür) | Tanımsız `client`/`deepseek_client`, sabit model adları | Sentez `self.llm` ile; `MHOP_SYNTHESIS_PROMPT` seçeneği; `EGOT_MAX_DEPTH` | `got.py:251,265-266,398,471-480` |
| H14 | UltraDomain değerlendirme betiği | `ModuleNotFoundError: No module named 'constants'` | Dosyanın adı `constant.py`; ayrıca `OPENAI_URL` | `constant` + `OPENAI_BASE_URL` yedeği | `scripts/eval_ultradomain.py:10,16` |
| H15 | MultiHop değerlendirme betiği | `FileNotFoundError: [Errno 2] No such file or directory: ''` | CSV yolları boş string | Fonksiyonlar import edildi | `scripts/eval_multihopqa.py:57-62` |
| H16 | Kanıt listesi ayrıştırılamıyor | `json.loads` → `JSONDecodeError`; `ast.literal_eval` → `SyntaxError` | numpy repr'i; sözlükler arasında virgül yok | `re.sub(r"\}\s*\{", "}, {", s)` + `ast.literal_eval` | `datasets/mhoprag_contexts.csv` (`evidence_list`) |
| H17 | CSV üreticisi sessizce boş çıktı veriyor | `Found 0 JSON files.` + boş CSV'ler | Sabit `/home/exouser/...` yolları | Yollar `EGOT_JSON_DIR` / `EGOT_CSV_DIR` ortam değişkenlerinden | `scripts/neo4j/raw_json_to_csv_generator.py:9-10` |

**Colab'da benim karşılaştığım ek hatalar** (çalıştırma sırasında doldurulacak):

| # | Hata | Belirti | Sebep | Çözüm | Hücre / dosya:satır |
|---|---|---|---|---|---|
| C1 | | | | | |
| C2 | | | | | |
| C3 | | | | | |

### 10.2 Bu çalıştırmada yakalanan hata mesajları ve uygulanan yamalar

In [ ]:
# 10.2 Bu çalıştırmada gerçekten yakalanan hata mesajları (yukarıdaki hücrelerin çıktılarından toplandı)
display(pd.DataFrame(ERRORS))
print("Uygulanan yamalar:")
display(pd.DataFrame(PATCHES))
run(["git", "-C", str(REPO), "diff", "--stat"], tail=20)

## 11. Teknik Değerlendirme: eGOT ile GraphRAG

### 11.0 Bu çalıştırmada ölçülen değerler

Aşağıdaki hücre, değerlendirmede atıf yapılan değerleri tek yerde toplar: indeksleme ve sorgu aşamalarının LLM çağrısı, token, süre ve tahmini maliyeti; GoT'un soru başına süre ve çağrı dağılımı. Bu sayılardan **otomatik bir özet paragrafı** üretilir: cevap kalitesi, hız farkı, bir GoT sorusunun kaç belgenin indekslenmesine denk geldiği ve toplam süre. Metne elle sayı yazılmadığı için paragraf her çalıştırmada o çalıştırmanın sonuçlarını yansıtır. Hücre ayrıca repodaki önceki deney çıktılarının (`*_eval_results`) satır sayılarını gösterir.

In [ ]:
# 11.0 Bu çalıştırmada ölçülen değerler (değerlendirmede bunlara atıf yapılır)
from IPython.display import Markdown

display(COSTS)
per_q = GOT[["seconds", "llm_çağrısı", "token_girdi", "token_çıktı"]].describe().loc[["mean", "min", "max"]].round(1)
print(f"GoT (d={MAX_DEPTH}) soru başına — teorik çağrı sayısı 2 + 9·{MAX_DEPTH} + 1 = {theory}:")
display(per_q)

# Ölçülen sayılardan otomatik özet (metne elle sayı yazılmaz; bu paragraf her çalıştırmada yeniden üretilir)
n_docs = max(1, len(list(JSON_DIR.glob("*.json"))))
idx_tok = BUILDER_USAGE["prompt_tokens"] + BUILDER_USAGE["completion_tokens"]
q_tok = (GOT["token_girdi"] + GOT["token_çıktı"]).mean()
lines = [f"- **Cevap kalitesi (GoT, d={MAX_DEPTH}, {len(GOT)} soru):** EM = {GOT['EM'].mean():.2f}, "
         f"F1 = {GOT['F1'].mean():.2f}, içerme = {GOT['içerme'].mean():.2f}, kanıt bulma = {GOT['kanıt_bulma'].mean():.2f}."]
if not BASE.empty:
    lines.append(f"- **GoT'suz mod (aynı sorular):** EM = {BASE['EM'].mean():.2f}, F1 = {BASE['F1'].mean():.2f}, "
                 f"içerme = {BASE['içerme'].mean():.2f}, kanıt bulma = {BASE['kanıt_bulma'].mean():.2f} "
                 "(uzun cevaplar yüzünden EM bu modda doğrudan karşılaştırılamaz).")
lines.append(f"- **Sorgu maliyeti:** GoT soru başına ortalama {GOT['seconds'].mean():.1f} sn sürdü ve "
             f"{GOT['llm_çağrısı'].mean():.1f} LLM çağrısı yaptı (teorik değer {theory}); ortalama {q_tok:,.0f} token.")
if not BASE.empty and BASE["seconds"].mean() > 0:
    lines.append(f"- **Hız farkı:** GoT'suz mod soru başına ortalama {BASE['seconds'].mean():.1f} sn sürdü; GoT bunun "
                 f"yaklaşık {GOT['seconds'].mean() / BASE['seconds'].mean():.0f} katı.")
lines.append(f"- **İndeksleme maliyeti:** {n_docs} belge için {BUILDER_USAGE['calls']} LLM çağrısı, {idx_tok:,} token, "
             f"{BUILD_SECONDS:.0f} sn (belge başına ~{idx_tok / n_docs:,.0f} token). Bir GoT sorusu, token olarak yaklaşık "
             f"{q_tok / max(1, idx_tok / n_docs):.1f} belgenin indekslenmesine denk geliyor.")
costs = [c for c in COSTS["tahmini maliyet ($)"] if c is not None]
if len(costs) == len(COSTS):
    lines.append(f"- **Tahmini maliyet:** indeksleme {costs[0]:.4f} \\$, GoT sorguları {costs[1]:.4f} \\$ "
                 "(Bölüm 0.1'deki fiyatlarla).")
if LOCAL_LLM:
    lines.append("- **Maliyet:** Model Colab GPU'sunda yerel çalıştı; API ücreti yok. Harcanan kaynak, süre ve Colab'ın "
                 "GPU kotasıdır.")
    ollama_log = OLLAMA_LOG.read_text(errors="replace") if OLLAMA_LOG.exists() else ""
    n_tok = [int(x) for x in re.findall(r"stop processing: n_tokens = (\d+)", ollama_log)]   # Ollama'nın istek kayıtları
    if n_tok:
        lines.append(f"- **Bağlam penceresi:** En uzun istek (prompt + cevap) {max(n_tok):,} token tuttu; pencere "
                     f"{OLLAMA_CONTEXT_LENGTH:,} token. Pencereye sığmadığı için reddedilen istek sayısı: "
                     f"{ollama_log.count('exceeds the available context size')}.")
lines.append(f"- **Toplam süre:** notebook'un başından bu hücreye kadar {(time.time() - NB_T0) / 60:.1f} dakika geçti.")
display(Markdown("**Bu çalıştırmanın ölçülen değerleri (otomatik üretildi):**\n\n" + "\n".join(lines)))

# Repodaki *_eval_results dosyaları: kaç satır olmalı, kaç satır var?
rows = []
for f in sorted((REPO / "datasets").glob("*/*eval_results/*.jsonl")):
    domain = f.parts[-3]
    n_q = sum(1 for l in (REPO / "datasets" / domain / f"{domain}.jsonl").read_text().splitlines() if l.strip())
    n_rows = sum(1 for l in f.read_text().splitlines() if l.strip())
    first = json.loads(f.read_text().splitlines()[0])
    rows.append({"dosya": f.name[:60], "soru": n_q, "beklenen satır (2×soru)": 2 * n_q, "satır": n_rows,
                 "alanlar": ", ".join(first.keys())})
print("Repodaki *_eval_results dosyaları:")
display(pd.DataFrame(rows))

### 11.1 İndeksleme: graf nasıl kuruluyor?

| | Microsoft GraphRAG [3] | eGOT [1, 6] |
|---|---|---|
| Çıkarılan bilgi | LLM ile varlıklar, ilişkiler ve bunların açıklamaları | LLM ile chunk başına S-P-O üçlüleri (yüklem ≤ 3 kelime) |
| Birleştirme | Varlık ve ilişki açıklamaları LLM ile özetlenir | Belge bazında kural + LLM ile varlık standardizasyonu; belgeler arası birleşme yalnızca adlar birebir aynıysa olur |
| Ek yapı | Graf **Leiden** algoritmasıyla hiyerarşik topluluklara bölünür; her topluluk için LLM bir **topluluk özeti** yazar, alt seviye özetler üst seviyeleri besler | Topluluk tespiti ve özeti **yok**. Builder'daki "community" yalnızca bağlantısız bileşenleri LLM ile birleştirmek için kullanılır. Retrieval sorgusundaki `__Community__` etiketi LLM Graph Builder'dan kalmadır; eGOT hattı böyle düğüm üretmez (Bölüm 4.12, not 6) |
| Kaynakla bağ | Özetler üzerinden | Her üçlü kendi chunk'ına bağlı (`HAS_ENTITY`, `PART_OF`): graf bilgisi her zaman kaynak metinle birlikte gelir ve cevap belgeye atfedilebilir |
| İndeksler | Topluluk özetleri (+ varlık/metin birimleri) | Vektör (chunk gömmesi) + tam metin + graf komşuluğu (hibrit) |

### 11.2 Sorgu zamanı: tek geçiş ile yinelemeli GoT

- **GraphRAG** [3], korpusun bütünü hakkındaki "global" sorular için önceden hesaplanmış topluluk özetlerini kullanır. *Map* adımında her özetten bağımsız ve paralel kısmi cevaplar üretilir, *reduce* adımında bunlar tek cevapta birleştirilir. Hangi özetlerin okunacağı sorudan önce bellidir; arama sırasında yön değiştirmez.
- **eGOT** ise sabit bir özet katmanı yerine ham chunk metni ve üçlüler üzerinde **yinelemeli** arama yapar. LLM, toplanan kanıta bakarak yeni alt sorular (thought) üretir ve puanlar; her thought için hibrit + graf genişletmeli retrieval yapılır. Uyarlamalı genişletmede soruya çok benzeyen varlıklar için 2 adım, orta benzerlikteki varlıklar için 1 adım graf komşuluğu bağlama eklenir (Bölüm 1.3). Bu, çok adımlı (multi-hop) sorularda ilk retrieval'ın kaçırdığı ara varlıklara sonraki turlarda ulaşmayı amaçlar. Graph of Thoughts [4] düşünceleri bir grafın düğümleri olarak ele alıp üretme, birleştirme ve puanlama işlemlerini tanımlar. eGOT'taki biçim bunun sadeleştirilmiş bir hâlidir: dal başına en iyi thought'un genişletildiği, sabit genişlikte (3 dal) bir arama ve sonunda bütün kanıtın tek bir sentezde birleştirilmesi.

### 11.3 Maliyet nerede oluşuyor?

- **GraphRAG'de** maliyetin büyük kısmı **indekslemede** oluşur: her parçadan varlık/ilişki çıkarımı, ayrıca hiyerarşinin her seviyesindeki her topluluk için LLM ile özet yazımı. Global sorguda map adımı, kullanılan topluluk özeti sayısı kadar LLM çağrısı yapar.
- **eGOT'ta** indeksleme daha hafiftir. Chunk başına 1 çıkarım çağrısı, belge başına 1 standardizasyon çağrısı ve bağlantısız bileşen sayısına bağlı birkaç çıkarım çağrısı yapılır (en fazla 10 + 3); özetleme adımı yoktur. Buna karşılık **sorgu başına** maliyet yüksektir: 2 + 9·d + 1 çağrı (d = 2 için 21, repo varsayılanı d = 3 için 30). Her çağrıya en fazla 4 belgelik bağlam eklenir ve son derinlikteki 6 çağrının ürettiği thought'lar hiç kullanılmaz. GoT'suz mod (`graph_vector_fulltext`) ise soru başına 1 cevap çağrısı (+ arka planda 1 özetleme) yapar.
- Koddan çıkan beklenti şudur: İndeksleme maliyeti belge sayısıyla, sorgu maliyeti ise soru sayısı × derinlikle doğrusal büyür. Soru sayısı arttıkça eGOT'un toplam maliyetinde sorgu aşamasının payı artar; GraphRAG'de ise ağırlık indekslemededir. Bu çalıştırmadaki gerçek oran Bölüm 11.0'daki tabloda görülür: "İndeksleme" satırını "Sorgu: GoT" satırıyla ve soru başına GoT değerleriyle karşılaştırın.

### 11.4 Bu çalıştırmadan gözlemler

Aşağıdakileri yorumlarken Bölüm 9 ve 11.0'daki tablolara bakın:
- **EM/F1 ve içerme:** GoT'un `mhop` sentezi kısa cevap verdiği için EM ile değerlendirilebilir. GoT'suz mod uzun cevap verdiği için EM'si düşük çıkar; iki mod en adil biçimde *içerme* ve *kanıt bulma* ile karşılaştırılır.
- **Süre:** GoT soru başına ~21 sıralı LLM çağrısı yaptığı için GoT'suz moddan (1 çağrı) çok daha uzun sürmesi beklenir. Gerçek süreler Bölüm 9.3'teki tablodadır.
- **Kanıt bulma:** Korpus yalnızca seçilen soruların kanıtlarından oluştuğu için retrieval'ın doğru belgeleri bulması kolaydır. Bu yüzden yüksek kanıt bulma oranı, eGOT'un büyük bir korpusta da aynı başarıyı göstereceği anlamına gelmez.

### 11.5 Sınırlamalar

1. **Küçük, kapalı korpus ve küçük N.** KG yalnızca seçilen `N_QUESTIONS` sorunun kanıtlarından kuruldu; tam MultiHop-RAG haber korpusundaki dikkat dağıtıcı belgeler yok. 8 soruluk bir örneklemde varyans yüksektir, istatistiksel bir sonuç çıkarılamaz. Sonuçlar makaledeki sayılarla **karşılaştırılamaz**.
2. **Repo eksikleri:** Yazarların MultiHop-RAG için kullandığı `MAIN_MHOP_PROMPT` repoda yok; genel prompt kullanıldı (H4). Repo'nun kanıt biçimi kaynak ve tarih bilgisini dışarıda bırakıyor (Bölüm 5.3); bu, özellikle *temporal* soruları zorlaştırır.
3. **Model ve ayarlar:** Makale ve repodaki dosya adları farklı LLM kombinasyonlarına işaret ediyor (ör. `llama_deepseek`, `llama_gpt`). Biz her aşamada tek model kullandık: Bölüm 0.1'deki `LLM_MODEL` (varsayılan olarak 4-bit nicemlenmiş, 7B'lik Qwen2.5-7B-Instruct), `temperature=0.2` (builder) / `0` (retrieval). Bu, makaledeki modellerden çok daha küçük bir modeldir. Üçlü çıkarımı, thought üretimi ve sentez kalitesi bu yüzden makaledekinden düşük olabilir. LLM çıktıları deterministik olmadığı için tekrar çalıştırmalar küçük farklar gösterebilir.
4. **Repodaki `*_eval_results` dosyaları kullanılamaz.** Bu dosyalar `eval_ultradomain.py`'nin LLM-hakem çıktılarıdır. Betik her soruyu iki kez değerlendirir; ikincisinde cevapların yerleri değiştirilir (`queries += queries; answers1, answers2 = answers1 + answers2, answers2 + answers1`). Her satırda yalnızca "Answer 1"/"Answer 2" kazananı yazar; hangi cevabın hangi yönteme ait olduğu dosyada **yazmaz**. Üstelik JSON'u ayrıştırılamayan değerlendirmeler sessizce atlanır: Bölüm 11.0'daki tabloda bazı `mix` dosyaları 260 yerine 258–259 satırdır, bu da sıra eşleşmesini belirsiz kılar. Bu yüzden bu dosyalardan **kazanma oranı türetmedik**.

## 12. Kaynaklar

1. Sanda, N., Gyori, B. M., Quaranta, V., Ganguly, A., Paul, A. (2026). *eGoT: enhanced graph-of-thoughts for multi-hop knowledge retrieval and hypothesis generation in biomedicine.* **Bioinformatics**, 42(Supplement_1), btag216 (ISMB 2026). doi: [10.1093/bioinformatics/btag216](https://doi.org/10.1093/bioinformatics/btag216) · PMC: [PMC13341121](https://pmc.ncbi.nlm.nih.gov/articles/PMC13341121/)
2. Neural Dynamics Lab, Northeastern University (2026). *eGoT: Enhanced Graph-of-Thoughts for Multi-Hop Knowledge Retrieval and Hypothesis Generation in Biomedicine* (lab blogu). https://neuraldynamics.lab.ai.northeastern.edu/blog-egot-2026
3. Edge, D. vd. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* arXiv:2404.16130. https://arxiv.org/abs/2404.16130
4. Besta, M. vd. (2024). *Graph of Thoughts: Solving Elaborate Problems with Large Language Models.* AAAI 2024, 38(16), 17682–17690. arXiv:2308.09687. https://arxiv.org/abs/2308.09687
5. Tang, Y., Yang, Y. (2024). *MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries.* arXiv:2401.15391 (COLM 2024). https://arxiv.org/abs/2401.15391
6. eGOT kaynak kodu: https://github.com/NNeuralDynamics/eGOT, commit `3d37906c1b22472ee970c85e8dbd69ebcd5f8ffa`.
7. Neo4j Labs, *LLM Graph Builder* (eGOT retrieval kodunun dayandığı proje; `backend/src/shared/constants.py`, `backend/src/QA_integration.py`). https://github.com/neo4j-labs/llm-graph-builder
8. McDermott, R. *ai-knowledge-graph* (eGOT graph-builder'ının dayandığı proje). https://github.com/robert-mcdermott/ai-knowledge-graph
9. OpenAI, *Deprecations* (gpt-4o-mini-2024-07-18 kapanış tarihi: 14 Ocak 2027). https://developers.openai.com/api/docs/deprecations — erişim: 3 Kasım 2026.
10. OpenAI, *API Pricing* (gpt-4o-mini). https://openai.com/api/pricing/ — erişim: Kasım 2026; fiyatlar değişebilir.
11. Neo4j, *Operations Manual — Set an initial password*: https://neo4j.com/docs/operations-manual/current/configuration/set-initial-password/ ve *Upgrade and Migration Guide — Install and configure Neo4j 5* (Java 17/21): https://neo4j.com/docs/upgrade-migration-guide/current/version-5/migration/install-and-configure/
12. Neo4j, *APOC Core — Installation* (labs/ → plugins/). https://neo4j.com/docs/apoc/current/installation/
13. Neo4j, *Cypher Manual 5 — Vector functions* (`vector.similarity.cosine`). https://neo4j.com/docs/cypher-manual/5/functions/vector/
14. LangChain, *LangChain v1 migration guide* (`langchain-classic`). https://docs.langchain.com/oss/python/migrate/langchain-v1
15. *sentence-transformers/all-MiniLM-L6-v2* model kartı (384 boyutlu gömme). https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
16. Google Colab, *backend-info* (çalışma zamanında kurulu paket sürümleri). https://github.com/googlecolab/backend-info
17. MultiHop-RAG veri seti (Hugging Face). https://huggingface.co/datasets/yixuantt/MultiHopRAG
18. Ollama, sürüm 0.40.14. https://github.com/ollama/ollama — OpenAI uyumluluğu: https://github.com/ollama/ollama/blob/v0.40.14/docs/api/openai-compatibility.mdx · bağlam penceresi, `OLLAMA_NUM_PARALLEL` ve `keep_alive`: https://github.com/ollama/ollama/blob/v0.40.14/docs/faq.mdx
19. Qwen Team (2024). *Qwen2.5 Technical Report.* arXiv:2412.15115. https://huggingface.co/papers/2412.15115
20. *Qwen/Qwen2.5-7B-Instruct* model kartı (7,61B parametre, Apache 2.0, bağlam uzunluğu, JSON çıktısı). https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
21. OpenAI Help Center, *Setting up and managing prepaid API billing.* https://help.openai.com/en/articles/8264644 — erişim: Kasım 2026.